# Six local recovery settings after Haiyan

This streamlined notebook fixes the analysis at six municipalities and six matched 5×5 POI kernels: Tacloban, Palo, Guiuan, Alangalang, Ormoc, and Baybay.

The visual sequence is intentionally simple:

1. show the six colour-coded municipalities and the solid-black Haiyan path;
2. pair a fixed square spatial map with the uncomposited daily raw NTL trajectory for each location;
3. show GHSL G3 support, the median 60-day baseline, and common-extent recovery stages with Tacloban insets;
4. present separate municipality and POI recovery grids using the original spatial-completeness-strip aesthetic;
5. compare raw daily magnitude against normalized four-day recovery; and
6. summarize impact and T50/T80/T90 using markers rather than threshold lines.

`DNB_BRDF_Corrected_NTL` with `Mandatory_Quality_Flag == 0` remains the principal signal. Four-day composites use the temporal median. Raw magnitudes use the spatial median. Gap-filled NTL remains diagnostic only.

> **Interpretive boundary.** Regional NGCP agreement supports local analysis as a nested test of electricity-dependent nocturnal activity. It does not validate municipal electricity restoration. Local claims remain conditional on observability and on construct-matched evidence from grid, daytime physical-recovery, housing, and relocation studies.

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import json
import re
import unicodedata
import warnings

import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr

from rasterio.enums import Resampling
from rasterio.features import rasterize
from shapely.geometry import Point

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# ============================================================
# 2. PATHS AND ANALYTICAL SETTINGS
# ============================================================

PROJECT_DIR_OVERRIDE = Path(
    "/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/"
    "02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery"
)

project_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    PROJECT_DIR_OVERRIDE,
]
PROJECT_DIR = next(
    (candidate for candidate in project_candidates if (candidate / "datasets").exists()),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not locate the project datasets directory. "
        "Set PROJECT_DIR_OVERRIDE to the project root."
    )

DATA_DIR = PROJECT_DIR / "datasets"
VNP46_DIR = DATA_DIR / "VNP46"
PROCESSED_DIR = VNP46_DIR / "processed"

A2_ZARR_PATH = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"
NGCP_CSV_PATH = DATA_DIR / "ngcp" / "NGCP_Hourly_Demand.csv"


def find_dataset(patterns, label):
    '''Return the first unique match and show all candidates.'''

    matches = []

    for pattern in patterns:
        matches.extend(DATA_DIR.glob(pattern))

    matches = sorted({path.resolve() for path in matches})

    if not matches:
        raise FileNotFoundError(
            f"No {label} matched under {DATA_DIR}.\n"
            f"Patterns: {patterns}"
        )

    if len(matches) > 1:
        print(f"{label}: multiple matches; using {matches[0]}")
        for candidate in matches:
            print("  ", candidate)

    return matches[0]


GHSL_PATH = find_dataset(
    [
        "VNP46/GHSL_SMOD_E2015.tif",
        "ghsl/GHSL_SMOD_E2015.tif",
        "**/*GHSL*SMOD*.tif",
    ],
    "GHSL SMOD raster",
)

MUNICITIES_PATH = find_dataset(
    [
        "**/*MuniCities*.shp",
        "**/*municities*.shp",
        "**/*Muni*Cit*.shp",
        "**/*Municipal*.shp",
    ],
    "MuniCities shapefile",
)

ROADS_PATH = find_dataset(
    [
        "**/*Roads*.shp",
        "**/*roads*.shp",
        "**/*Road*.shp",
    ],
    "Roads shapefile",
)

HAIYAN_TRACK_PATH = find_dataset(
    [
        "yolanda-path-line-/Yolanda Path Line.shp",
        "**/Yolanda Path Line.shp",
        "**/*Yolanda*Path*.shp",
        "**/*Haiyan*Path*.shp",
    ],
    "Haiyan/Yolanda path shapefile",
)

OUTPUT_DIR = PROJECT_DIR / "output" / "six_location_recovery"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Event and temporal design
EVENT_DATE = pd.Timestamp("2013-11-08")
BASELINE_DAYS = 60
ANALYSIS_START = EVENT_DATE - pd.Timedelta(days=180)
PROFILE_END = EVENT_DATE + pd.Timedelta(days=363)
BASELINE_START = EVENT_DATE - pd.Timedelta(days=BASELINE_DAYS)
PRE_EVENT_END = EVENT_DATE - pd.Timedelta(days=1)

STAGE_WINDOWS = {
    "Baseline": (BASELINE_START, PRE_EVENT_END),
    "Stage 1 (0–59 d)": (
        EVENT_DATE,
        EVENT_DATE + pd.Timedelta(days=59),
    ),
    "Stage 2 (60–119 d)": (
        EVENT_DATE + pd.Timedelta(days=60),
        EVENT_DATE + pd.Timedelta(days=119),
    ),
    "Stage 3 (120–179 d)": (
        EVENT_DATE + pd.Timedelta(days=120),
        EVENT_DATE + pd.Timedelta(days=179),
    ),
    "Extended (180–363 d)": (
        EVENT_DATE + pd.Timedelta(days=180),
        PROFILE_END,
    ),
}

# VNP46A2 layers
DNB_BAND = "DNB_BRDF_Corrected_NTL"
GAP_FILLED_BAND = "Gap_Filled_DNB_BRDF_Corrected_NTL"
MQF_BAND = "Mandatory_Quality_Flag"
SPATIAL_DIMS = ("y", "x")

# RQ1-informed settlement configurations. G3 is fixed for this notebook.
GHSL_MASKS = {
    "G2": (23, 30),
    "G3": (22, 23, 30),
    "G4": (21, 22, 23, 30),
}
SETTLEMENT_MASK = "G3"
SPATIAL_COMPLETENESS_PCT = 10.0
RQ_CLIP_PERCENTILE = 95.0

MIN_BASELINE_OBSERVATIONS = {
    1: 5,
    4: 3,
}

# Metric admissibility and persistence rules. T50/T80/T90 are measured
# against the matched 60-day baseline after the observed post-event nadir.
PERSISTENCE_BLOCKS = 2
MIN_EVENT_RETENTION_PCT = 20.0
MIN_EVENT_COMPOSITES = 8
MAX_INTERPRETABLE_GAP_DAYS = 24

# Fixed six-location analytical design. These are affected comparisons,
# not unaffected controls.
CORE_LOCATIONS = [
    {
        "recovery_setting": "Regional urban impact centre",
        "core_location": "Tacloban",
        "municipality_name": "Tacloban City",
        "poi_name": "Tacloban City Centre",
        "why": (
            "Major urban/service centre within the severely affected "
            "Leyte Gulf corridor"
        ),
    },
    {
        "recovery_setting": "Adjacent impacted settlement",
        "core_location": "Palo",
        "municipality_name": "Palo",
        "poi_name": "Palo Centre",
        "why": (
            "Tests recovery close to Tacloban but under a different "
            "settlement and reconstruction context"
        ),
    },
    {
        "recovery_setting": "First-landfall setting",
        "core_location": "Guiuan",
        "municipality_name": "Guiuan",
        "poi_name": "Guiuan Centre",
        "why": (
            "Represents the initial Haiyan landfall and severe eastern exposure"
        ),
    },
    {
        "recovery_setting": "Inland contrast",
        "core_location": "Alangalang",
        "municipality_name": "Alangalang",
        "poi_name": "Alangalang Centre",
        "why": (
            "Separates inland/peri-urban recovery from coastal and "
            "storm-surge trajectories"
        ),
    },
    {
        "recovery_setting": "Western urban comparison",
        "core_location": "Ormoc",
        "municipality_name": "Ormoc City",
        "poi_name": "Ormoc City Centre",
        "why": (
            "Tests whether recovery differs outside the principal eastern "
            "impact corridor"
        ),
    },
    {
        "recovery_setting": "Western secondary city",
        "core_location": "Baybay",
        "municipality_name": "Baybay City",
        "poi_name": "Baybay City Centre",
        "why": (
            "Adds a second western urban comparison under a different local context"
        ),
    },
]

TARGET_UNITS = [location["municipality_name"] for location in CORE_LOCATIONS]
LOCATION_ORDER = TARGET_UNITS.copy()
POI_ORDER = [location["poi_name"] for location in CORE_LOCATIONS]
DISPLAY_NAMES = {
    location["municipality_name"]: location["core_location"]
    for location in CORE_LOCATIONS
}
LOCATION_COLORS = {
    "Tacloban City": "#0072B2",
    "Palo": "#E69F00",
    "Guiuan": "#009E73",
    "Alangalang": "#CC79A7",
    "Ormoc City": "#D55E00",
    "Baybay City": "#56B4E9",
}

# Every local stage map uses the same square extent around its POI.
LOCAL_MAP_RADIUS_PIXELS = 14

FOCUS_UNIT = "Tacloban City"
POI_KERNEL_SIZE = 5

KNOWN_FILL_VALUES = (-9999.0, -32768.0, 6553.5, 65535.0)

print("Project:", PROJECT_DIR)
print("MuniCities:", MUNICITIES_PATH)
print("Roads:", ROADS_PATH)
print("Haiyan/Yolanda path:", HAIYAN_TRACK_PATH)
print("Primary GHSL mask:", SETTLEMENT_MASK, GHSL_MASKS[SETTLEMENT_MASK])
print("Spatial-completeness gate:", f"{SPATIAL_COMPLETENESS_PCT:.0f}%")

In [ ]:
# ============================================================
# 3. EVIDENCE MATRIX
# ============================================================

evidence_matrix = pd.DataFrame(
    [
        {
            "unit": "Tacloban City",
            "documented_behaviour": (
                "Catastrophic storm-surge and wind damage; severe distribution-system "
                "damage; central rebuilding and northern relocation produced spatially "
                "different recovery processes."
            ),
            "NTL_profile_to_test": (
                "Large observable shock if cloud gaps permit; central functional rebound "
                "may precede housing and relocation outcomes; later spatial redistribution "
                "is possible."
            ),
            "construct_boundary": (
                "Agreement with physical rebuilding does not establish household recovery; "
                "new northern lights may indicate relocation rather than return."
            ),
            "sources": (
                "Sheykhmousa et al. 2019; Ghaffarian et al. 2021; "
                "JICA 2015; Palagi & Javernick-Will 2018"
            ),
        },
        {
            "unit": "Palo / Tanauan / Tolosa / Dulag",
            "documented_behaviour": (
                "Severely affected eastern Leyte corridor; Tanauan field surveys document "
                "extreme storm-surge impacts; DORELCO distribution facilities were heavily damaged."
            ),
            "NTL_profile_to_test": (
                "Strong shock and staged recovery are plausible, but small G2 support may "
                "make G3 and interval timing more defensible."
            ),
            "construct_boundary": (
                "Municipality-wide radiance cannot identify household service or feeder sequence."
            ),
            "sources": "Yi et al. 2015; JICA 2015; Ghaffarian et al. 2020",
        },
        {
            "unit": "Guiuan",
            "documented_behaviour": (
                "Near Haiyan's first landfall; severe damage and Eastern Samar distribution-system loss."
            ),
            "NTL_profile_to_test": (
                "Large shock is plausible, but low-light support and post-landfall cloud may "
                "make magnitude or timing unidentifiable."
            ),
            "construct_boundary": "A missing or weak curve is not evidence of limited impact.",
            "sources": "NDRRMC 2014; JICA 2015",
        },
        {
            "unit": "Basey",
            "documented_behaviour": (
                "Storm-surge-exposed Samar coast within a heavily damaged distribution-service region."
            ),
            "NTL_profile_to_test": (
                "Disruption is plausible; observability may dominate because the illuminated support is small."
            ),
            "construct_boundary": "Prefer not observable over not recovered when support is inadequate.",
            "sources": "NDRRMC 2014; JICA 2015",
        },
        {
            "unit": "Alangalang",
            "documented_behaviour": (
                "Inland municipal centre used as a morphology and exposure contrast to the coastal corridor."
            ),
            "NTL_profile_to_test": (
                "A smaller or shorter shock than exposed coastal units is plausible, but not assumed."
            ),
            "construct_boundary": "This is a contrast, not an unaffected control.",
            "sources": "Principe et al. 2026; NDRRMC 2014",
        },
        {
            "unit": "Ormoc City / Baybay City",
            "documented_behaviour": (
                "Western Leyte urban hubs with persistent lights and different exposure from Leyte Gulf."
            ),
            "NTL_profile_to_test": (
                "Smaller shock or earlier functional return may occur, while regional grid disruption "
                "can still affect both cities."
            ),
            "construct_boundary": "Do not label either city unaffected without hazard and utility evidence.",
            "sources": "Principe et al. 2026; Arroyo & Åstrand 2019",
        },
    ]
)

In [ ]:
# ============================================================
# 4. VNP46A2 HELPERS
# ============================================================


def open_zarr_safely(path):
    try:
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks="auto",
            mask_and_scale=True,
            decode_cf=True,
        )
    except (ImportError, ModuleNotFoundError, ValueError):
        return xr.open_zarr(
            path,
            consolidated=None,
            chunks=None,
            mask_and_scale=True,
            decode_cf=True,
        )


def standardise_date_dimension(ds):
    if "date" not in ds.variables:
        raise KeyError(f"No `date` variable found. Variables: {list(ds.variables)}")

    if "date" not in ds.coords:
        ds = ds.set_coords("date")

    observation_dim = ds["date"].dims[0]
    dates = pd.DatetimeIndex(pd.to_datetime(ds["date"].values)).normalize()
    ds = ds.assign_coords(date=(observation_dim, dates.values))

    if observation_dim != "date":
        ds = ds.swap_dims({observation_dim: "date"})

    return ds.sortby("date")


def prepare_spatial_metadata(ds):
    ds = ds.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)

    if ds.rio.crs is None and "spatial_ref" in ds.variables:
        attrs = ds["spatial_ref"].attrs
        stored_crs = attrs.get("crs_wkt") or attrs.get("spatial_ref")

        if stored_crs is not None:
            ds = ds.rio.write_crs(stored_crs, inplace=False)

    if ds.rio.crs is None:
        x_min = float(ds["x"].min())
        x_max = float(ds["x"].max())
        y_min = float(ds["y"].min())
        y_max = float(ds["y"].max())

        if (
            -180 <= x_min <= 180
            and -180 <= x_max <= 180
            and -90 <= y_min <= 90
            and -90 <= y_max <= 90
        ):
            ds = ds.rio.write_crs("EPSG:4326", inplace=False)
        else:
            raise ValueError("The VNP46A2 CRS could not be recovered.")

    return ds


def clean_radiance(values):
    cleaned = values.astype("float32").where(np.isfinite(values))
    fill_values = list(KNOWN_FILL_VALUES)

    for source in (values.attrs, values.encoding):
        for key in ("_FillValue", "missing_value"):
            if source.get(key) is not None:
                fill_values.append(source[key])

    for fill_value in fill_values:
        try:
            fill_value = float(fill_value)
            if np.isfinite(fill_value):
                cleaned = cleaned.where(~np.isclose(cleaned, fill_value))
        except (TypeError, ValueError):
            continue

    return cleaned.where(cleaned >= 0)


if not A2_ZARR_PATH.exists():
    raise FileNotFoundError(f"VNP46A2 Zarr not found:\n{A2_ZARR_PATH}")

a2 = prepare_spatial_metadata(
    standardise_date_dimension(open_zarr_safely(A2_ZARR_PATH))
).sel(date=slice(ANALYSIS_START, PROFILE_END))

required_bands = [DNB_BAND, MQF_BAND]
missing_bands = [band for band in required_bands if band not in a2.data_vars]

if missing_bands:
    raise KeyError(
        f"Missing A2 bands: {missing_bands}\n"
        f"Available: {list(a2.data_vars)}"
    )

dnb = clean_radiance(a2[DNB_BAND])
mqf = a2[MQF_BAND]
dnb, mqf = xr.align(dnb, mqf, join="inner")

gap_filled = (
    clean_radiance(a2[GAP_FILLED_BAND])
    if GAP_FILLED_BAND in a2.data_vars
    else None
)

print("A2 dimensions:", dict(a2.sizes))
print("A2 CRS:", a2.rio.crs)
print("Available dates:", pd.Timestamp(a2.date.min().item()).date(), "to", pd.Timestamp(a2.date.max().item()).date())

In [ ]:
# ============================================================
# 5. LOAD MUNICIPALITIES AND ROADS; DISCOVER FIELD NAMES
# ============================================================


def normalise_column_name(value):
    return re.sub(r"[^A-Z0-9]", "", str(value).upper())


def resolve_column(gdf, candidates, label):
    lookup = {normalise_column_name(column): column for column in gdf.columns}

    for candidate in candidates:
        key = normalise_column_name(candidate)
        if key in lookup:
            return lookup[key]

    for candidate in candidates:
        key = normalise_column_name(candidate)
        for normalised, column in lookup.items():
            if key in normalised or normalised in key:
                return column

    raise KeyError(
        f"Could not identify the {label} field. "
        f"Available columns: {list(gdf.columns)}"
    )


def canonical_unit_name(value):
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(character for character in text if not unicodedata.combining(character))
    tokens = re.sub(r"[^A-Z0-9]+", " ", text.upper()).split()
    stop_words = {"CITY", "OF", "MUNICIPALITY", "MUNICIPAL", "MUN"}
    return " ".join(token for token in tokens if token not in stop_words)


municipalities = gpd.read_file(MUNICITIES_PATH)

if municipalities.crs is None:
    raise ValueError("MuniCities shapefile does not contain a CRS.")

municipalities = municipalities.loc[
    municipalities.geometry.notna() & ~municipalities.geometry.is_empty
].copy()

try:
    municipalities["geometry"] = municipalities.geometry.make_valid()
except AttributeError:
    municipalities["geometry"] = municipalities.geometry.buffer(0)

NAME_COLUMN = resolve_column(
    municipalities,
    [
        "ADM3_EN",
        "ADM3_NAME",
        "MuniCity",
        "Muni_City",
        "CITY_MUN",
        "NAME_2",
        "NAME_3",
        "LGU_NAME",
        "MUNICIPALITY",
        "NAME",
    ],
    "city/municipality name",
)

province_candidates = [
    "ADM2_EN",
    "ADM2_NAME",
    "PROVINCE",
    "PROV_NAME",
    "NAME_1",
]

try:
    PROVINCE_COLUMN = resolve_column(
        municipalities,
        province_candidates,
        "province name",
    )
except KeyError:
    PROVINCE_COLUMN = None

municipalities["unit_name"] = municipalities[NAME_COLUMN].astype(str).str.strip()
municipalities["unit_key"] = municipalities["unit_name"].map(canonical_unit_name)
municipalities["province_name"] = (
    municipalities[PROVINCE_COLUMN].astype(str).str.strip()
    if PROVINCE_COLUMN is not None
    else "Not supplied"
)

# Exact canonical matching first; conservative substring matching second.
selected_rows = []
unmatched_targets = []

for target in TARGET_UNITS:
    target_key = canonical_unit_name(target)
    exact = municipalities.loc[municipalities["unit_key"] == target_key]

    if len(exact) == 1:
        selected_rows.append(exact.index[0])
        continue

    partial = municipalities.loc[
        municipalities["unit_key"].str.contains(target_key, regex=False)
        | pd.Series(
            [target_key in key for key in municipalities["unit_key"]],
            index=municipalities.index,
        )
    ]

    if len(partial) == 1:
        selected_rows.append(partial.index[0])
    else:
        unmatched_targets.append(target)

selected_municipalities = municipalities.loc[
    list(dict.fromkeys(selected_rows))
].copy()

target_name_lookup = {
    canonical_unit_name(name): name
    for name in TARGET_UNITS
}
selected_municipalities["source_unit_name"] = selected_municipalities["unit_name"]
selected_municipalities["unit_name"] = selected_municipalities["unit_key"].map(
    target_name_lookup
)

if selected_municipalities["unit_name"].isna().any():
    raise ValueError("A selected boundary could not be assigned to the fixed six-location design.")

if selected_municipalities.empty:
    raise ValueError(
        "None of TARGET_UNITS matched the MuniCities shapefile. "
        "Inspect the available names printed below and edit TARGET_UNITS."
    )

selected_municipalities["profile_id"] = np.arange(
    1,
    len(selected_municipalities) + 1,
)

roads = gpd.read_file(ROADS_PATH)

if roads.crs is None:
    raise ValueError("Roads shapefile does not contain a CRS.")

roads = roads.loc[roads.geometry.notna() & ~roads.geometry.is_empty].copy()

print("Name field:", NAME_COLUMN)
print("Province field:", PROVINCE_COLUMN)
print("Matched targets:")
display(selected_municipalities[["unit_name", "province_name", "unit_key"]])

if unmatched_targets:
    print("Unmatched targets; edit TARGET_UNITS if needed:", unmatched_targets)

print("First 30 available municipality names:")
display(
    municipalities[["unit_name", "province_name"]]
    .sort_values(["province_name", "unit_name"])
    .head(30)
)

haiyan_track = gpd.read_file(HAIYAN_TRACK_PATH)
if haiyan_track.crs is None:
    raise ValueError("The Haiyan/Yolanda path shapefile does not contain a CRS.")
haiyan_track = haiyan_track.loc[
    haiyan_track.geometry.notna() & ~haiyan_track.geometry.is_empty
].copy()

In [ ]:
# ============================================================
# 6. ALIGN GHSL AND RASTERIZE MUNICIPAL SUPPORT
# ============================================================

ghsl = rxr.open_rasterio(GHSL_PATH, masked=True)

if "band" in ghsl.dims:
    ghsl = ghsl.isel(band=0, drop=True)

if ghsl.rio.crs is None:
    raise ValueError("The GHSL raster does not contain a CRS.")

viirs_template = dnb.isel(date=0, drop=True)

ghsl_viirs = ghsl.rio.reproject_match(
    viirs_template,
    resampling=Resampling.nearest,
).assign_coords(x=viirs_template["x"], y=viirs_template["y"])

ghsl_mask = ghsl_viirs.isin(GHSL_MASKS[SETTLEMENT_MASK]).fillna(False)

municipalities_raster_crs = municipalities.to_crs(a2.rio.crs)
selected_raster_crs = selected_municipalities.to_crs(a2.rio.crs)
roads_raster_crs = roads.to_crs(a2.rio.crs)

raster_bounds = dnb.rio.bounds()
municipalities_raster_crs = municipalities_raster_crs.cx[
    raster_bounds[0]:raster_bounds[2],
    raster_bounds[1]:raster_bounds[3],
].copy()


def rasterize_units(gdf, value_column):
    shapes = [
        (geometry, int(value))
        for geometry, value in zip(gdf.geometry, gdf[value_column])
        if geometry is not None and not geometry.is_empty
    ]

    values = rasterize(
        shapes,
        out_shape=(dnb.sizes["y"], dnb.sizes["x"]),
        transform=dnb.rio.transform(recalc=True),
        fill=0,
        all_touched=False,
        dtype="int32",
    )

    return xr.DataArray(
        values,
        dims=SPATIAL_DIMS,
        coords={"y": dnb["y"], "x": dnb["x"]},
    )


municipalities_raster_crs = municipalities_raster_crs.copy()
municipalities_raster_crs["all_unit_id"] = np.arange(
    1,
    len(municipalities_raster_crs) + 1,
)

all_zone_id = rasterize_units(municipalities_raster_crs, "all_unit_id")
selected_zone_id = rasterize_units(selected_raster_crs, "profile_id")

study_mask = selected_zone_id > 0
rq_base_mask = (ghsl_mask & study_mask).compute()

print("GHSL classes:", GHSL_MASKS[SETTLEMENT_MASK])
print("RQ settlement pixels within the six selected municipalities:", f"{int(rq_base_mask.sum().item()):,}")
print("Any selected LGU represented on the VIIRS grid:", bool((selected_zone_id > 0).any().item()))

haiyan_track_raster_crs = haiyan_track.to_crs(a2.rio.crs)

In [ ]:
# ============================================================
# 7. BUILD THE RELIABILITY-QUALIFIED CUBE
# ============================================================

rq_unclipped = dnb.where((mqf == 0) & rq_base_mask)
rq_quantile_source = rq_unclipped

if hasattr(rq_unclipped.data, "rechunk"):
    spatial_axes = {
        rq_unclipped.get_axis_num(dimension): -1
        for dimension in SPATIAL_DIMS
    }
    rq_quantile_source = rq_unclipped.copy(
        data=rq_unclipped.data.rechunk(spatial_axes)
    )

rq_daily_p95 = (
    rq_quantile_source
    .quantile(RQ_CLIP_PERCENTILE / 100.0, dim=SPATIAL_DIMS, skipna=True)
    .squeeze(drop=True)
    .compute()
)

rq_cube = xr.where(
    rq_unclipped > rq_daily_p95,
    rq_daily_p95,
    rq_unclipped,
)
rq_cube.name = "reliability_qualified_ntl"

print(
    "Daily P95 range:",
    f"{float(rq_daily_p95.min(skipna=True)):.2f}",
    "to",
    f"{float(rq_daily_p95.max(skipna=True)):.2f}",
    "nW cm⁻² sr⁻¹",
)

In [ ]:
# ============================================================
# 8. PROFILE FUNCTIONS
# ============================================================


def crop_to_support(cube, support_mask):
    support_values = np.asarray(support_mask.fillna(False).values, dtype=bool)
    rows, columns = np.where(support_values)

    if len(rows) == 0:
        return None, None

    y_slice = slice(rows.min(), rows.max() + 1)
    x_slice = slice(columns.min(), columns.max() + 1)

    return (
        cube.isel(y=y_slice, x=x_slice),
        support_mask.isel(y=y_slice, x=x_slice),
    )


def build_pixel_matched_profile(
    cube,
    support_mask,
    aggregation_days,
    unit_name,
    unit_type,
    method="Reliability-qualified DNB-BRDF",
):
    '''Build a daily or non-overlapping multi-day profile.'''

    selected = (
        cube
        .sel(date=slice(ANALYSIS_START, PROFILE_END))
        .where(support_mask)
    )

    dates = pd.DatetimeIndex(selected["date"].values).normalize()
    block_numbers = np.floor_divide(
        (dates - EVENT_DATE).days,
        aggregation_days,
    ).astype(int)

    composites = (
        selected
        .assign_coords(block=("date", block_numbers))
        .groupby("block")
        .median(dim="date", skipna=True)
    )

    blocks = composites["block"].values.astype(int)
    block_start = EVENT_DATE + pd.to_timedelta(blocks * aggregation_days, unit="D")
    block_end = block_start + pd.Timedelta(days=aggregation_days - 1)

    baseline_blocks = blocks[
        (block_start >= BASELINE_START)
        & (block_end <= PRE_EVENT_END)
    ]

    if len(baseline_blocks) == 0:
        raise ValueError(f"{unit_name}: no complete baseline blocks were found.")

    expected_baseline_blocks = BASELINE_DAYS // aggregation_days
    if BASELINE_DAYS % aggregation_days != 0:
        raise ValueError(
            "BASELINE_DAYS must be divisible by aggregation_days so the "
            "baseline contains complete, non-overlapping composites."
        )
    if len(baseline_blocks) != expected_baseline_blocks:
        raise ValueError(
            f"{unit_name}: expected {expected_baseline_blocks} complete baseline "
            f"blocks inside {BASELINE_START.date()}–{PRE_EVENT_END.date()}, "
            f"but found {len(baseline_blocks)}."
        )

    baseline_composites = composites.sel(block=baseline_blocks).compute()
    baseline_observations = baseline_composites.notnull().sum(dim="block")

    ntl0 = baseline_composites.median(dim="block", skipna=True)
    baseline_quantiles = baseline_composites.quantile(
        [0.25, 0.75],
        dim="block",
        skipna=True,
    )
    ntl0_q25 = baseline_quantiles.sel(quantile=0.25, drop=True)
    ntl0_q75 = baseline_quantiles.sel(quantile=0.75, drop=True)

    minimum_baseline = MIN_BASELINE_OBSERVATIONS[aggregation_days]
    fixed_mask = (
        support_mask
        & (baseline_observations >= minimum_baseline)
        & np.isfinite(ntl0)
        & (ntl0 > 0)
    ).compute()

    fixed_pixel_count = int(fixed_mask.sum().item())

    if fixed_pixel_count == 0:
        raise ValueError(
            f"{unit_name}: no baseline-lit {SETTLEMENT_MASK} pixels "
            f"met the baseline requirement."
        )

    paired_valid = composites.notnull() & fixed_mask & ntl0.notnull()
    valid_pixel_count = paired_valid.sum(dim=SPATIAL_DIMS)
    spatial_coverage_pct = 100.0 * valid_pixel_count / fixed_pixel_count

    current_radiance = composites.where(paired_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    matched_baseline = ntl0.where(paired_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    recovery_pct = 100.0 * current_radiance / matched_baseline

    # Raw magnitude before baseline normalization: spatial median across the
    # currently valid, fixed baseline-lit pixels.
    raw_median_source = composites.where(paired_valid)
    matched_median_source = ntl0.where(paired_valid)

    if hasattr(raw_median_source.data, "rechunk"):
        raw_spatial_axes = {
            raw_median_source.get_axis_num(dimension): -1
            for dimension in SPATIAL_DIMS
        }
        raw_median_source = raw_median_source.copy(
            data=raw_median_source.data.rechunk(raw_spatial_axes)
        )
    if hasattr(matched_median_source.data, "rechunk"):
        matched_spatial_axes = {
            matched_median_source.get_axis_num(dimension): -1
            for dimension in SPATIAL_DIMS
        }
        matched_median_source = matched_median_source.copy(
            data=matched_median_source.data.rechunk(matched_spatial_axes)
        )

    raw_median_ntl = raw_median_source.median(
        dim=SPATIAL_DIMS,
        skipna=True,
    )
    matched_baseline_median_ntl = matched_median_source.median(
        dim=SPATIAL_DIMS,
        skipna=True,
    )

    uncertainty_valid = (
        paired_valid
        & ntl0_q25.notnull()
        & ntl0_q75.notnull()
        & (ntl0_q25 > 0)
    )
    uncertainty_current = composites.where(uncertainty_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    baseline_q25_sum = ntl0_q25.where(uncertainty_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )
    baseline_q75_sum = ntl0_q75.where(uncertainty_valid).sum(
        dim=SPATIAL_DIMS,
        skipna=True,
        min_count=1,
    )

    recovery_low_pct = 100.0 * uncertainty_current / baseline_q75_sum
    recovery_high_pct = 100.0 * uncertainty_current / baseline_q25_sum

    reduced = xr.Dataset(
        {
            "current_radiance": current_radiance,
            "matched_baseline_radiance": matched_baseline,
            "raw_median_ntl": raw_median_ntl,
            "matched_baseline_median_ntl": matched_baseline_median_ntl,
            "recovery_pct": recovery_pct,
            "recovery_low_pct": recovery_low_pct,
            "recovery_high_pct": recovery_high_pct,
            "spatial_coverage_pct": spatial_coverage_pct,
            "valid_pixel_count": valid_pixel_count,
        }
    ).compute()

    profile = (
        reduced
        .to_dataframe()
        .reset_index()
        .sort_values("block")
        .reset_index(drop=True)
    )
    profile["date_start"] = EVENT_DATE + pd.to_timedelta(
        profile["block"] * aggregation_days,
        unit="D",
    )
    profile["date_end"] = profile["date_start"] + pd.Timedelta(
        days=aggregation_days - 1
    )
    profile["date_mid"] = profile["date_start"] + pd.to_timedelta(
        (aggregation_days - 1) / 2,
        unit="D",
    )

    below_gate = profile["spatial_coverage_pct"] < SPATIAL_COMPLETENESS_PCT
    profile.loc[
        below_gate,
        [
            "current_radiance",
            "matched_baseline_radiance",
            "raw_median_ntl",
            "matched_baseline_median_ntl",
            "recovery_pct",
            "recovery_low_pct",
            "recovery_high_pct",
        ],
    ] = np.nan

    profile["observation_status"] = np.where(
        profile["recovery_pct"].notna(),
        "observed",
        "not observable",
    )
    profile["unit_name"] = unit_name
    profile["unit_type"] = unit_type
    profile["method"] = method
    profile["aggregation_days"] = aggregation_days
    profile["ghsl_mask"] = SETTLEMENT_MASK
    profile["sc_threshold_pct"] = SPATIAL_COMPLETENESS_PCT
    profile["baseline_start"] = BASELINE_START
    profile["baseline_end"] = PRE_EVENT_END
    profile["baseline_days"] = BASELINE_DAYS
    profile["temporal_composite_statistic"] = "median"
    profile["raw_spatial_statistic"] = "median"
    profile["baseline_definition"] = (
        "per-pixel median of complete median reliability-qualified composites "
        "within the 60 days before Haiyan"
    )

    baseline_rows = profile.loc[
        (profile["date_start"] >= BASELINE_START)
        & (profile["date_end"] <= PRE_EVENT_END)
    ]

    report = {
        "unit_name": unit_name,
        "unit_type": unit_type,
        "aggregation_days": aggregation_days,
        "ghsl_mask": SETTLEMENT_MASK,
        "ghsl_classes": str(GHSL_MASKS[SETTLEMENT_MASK]),
        "sc_threshold_pct": SPATIAL_COMPLETENESS_PCT,
        "baseline_start": BASELINE_START.date(),
        "baseline_end": PRE_EVENT_END.date(),
        "baseline_blocks": len(baseline_blocks),
        "minimum_baseline_observations": minimum_baseline,
        "temporal_composite_statistic": "median",
        "raw_spatial_statistic": "median",
        "fixed_baseline_pixels": fixed_pixel_count,
        "median_baseline_coverage_pct": baseline_rows[
            "spatial_coverage_pct"
        ].median(),
    }

    return profile, composites.where(fixed_mask), ntl0.where(fixed_mask), fixed_mask, report

In [ ]:
# ============================================================
# 9. BUILD REGIONAL AND MUNICIPAL PROFILES
# ============================================================

regional_cube, regional_support = crop_to_support(rq_cube, rq_base_mask)

(
    regional_daily,
    regional_composites_daily,
    regional_ntl0_daily,
    regional_fixed_daily,
    regional_daily_report,
) = build_pixel_matched_profile(
    cube=regional_cube,
    support_mask=regional_support,
    aggregation_days=1,
    unit_name="Six-municipality pooled support",
    unit_type="Regional bridge",
)

(
    regional_four_day,
    regional_composites_four_day,
    regional_ntl0_four_day,
    regional_fixed_four_day,
    regional_four_day_report,
) = build_pixel_matched_profile(
    cube=regional_cube,
    support_mask=regional_support,
    aggregation_days=4,
    unit_name="Six-municipality pooled support",
    unit_type="Regional bridge",
)

municipal_profiles = []
municipal_reports = []
municipal_failures = []

for row in selected_raster_crs.itertuples():
    unit_support = (selected_zone_id == int(row.profile_id)) & ghsl_mask
    local_cube, local_support = crop_to_support(rq_cube, unit_support)

    if local_cube is None:
        municipal_failures.append(
            {
                "unit_name": row.unit_name,
                "reason": f"No {SETTLEMENT_MASK} pixels in the municipality",
            }
        )
        continue

    for aggregation_days in (1, 4):
        try:
            profile, _, _, _, report = build_pixel_matched_profile(
                cube=local_cube,
                support_mask=local_support,
                aggregation_days=aggregation_days,
                unit_name=row.unit_name,
                unit_type="City/municipality",
            )
            municipal_profiles.append(profile)
            municipal_reports.append(report)
        except ValueError as error:
            municipal_failures.append(
                {
                    "unit_name": row.unit_name,
                    "aggregation_days": aggregation_days,
                    "reason": str(error),
                }
            )

if not municipal_profiles:
    raise ValueError(
        f"No municipal profiles could be built with {SETTLEMENT_MASK}. "
        "Inspect municipal_failures or rerun with SETTLEMENT_MASK = 'G3'."
    )

municipal_profiles = pd.concat(municipal_profiles, ignore_index=True)
municipal_reports = pd.DataFrame(municipal_reports)
municipal_failures = pd.DataFrame(municipal_failures)

municipal_daily = municipal_profiles.loc[
    municipal_profiles["aggregation_days"] == 1
].copy()
municipal_four_day = municipal_profiles.loc[
    municipal_profiles["aggregation_days"] == 4
].copy()

if municipal_four_day.empty:
    raise ValueError(
        "Daily profiles were available, but no four-day municipal profile "
        "passed the baseline support requirement."
    )

missing_municipal_profiles = [
    name for name in LOCATION_ORDER
    if name not in set(municipal_four_day["unit_name"])
]
if missing_municipal_profiles:
    raise ValueError(
        "The fixed six-location design is incomplete. Missing municipal profiles: "
        + ", ".join(missing_municipal_profiles)
    )

municipal_four_day["unit_name"] = pd.Categorical(
    municipal_four_day["unit_name"],
    categories=LOCATION_ORDER,
    ordered=True,
)
municipal_daily["unit_name"] = pd.Categorical(
    municipal_daily["unit_name"],
    categories=LOCATION_ORDER,
    ordered=True,
)

print("Municipal profiles built:", municipal_four_day["unit_name"].nunique())
display(
    municipal_reports.loc[
        municipal_reports["aggregation_days"] == 4,
        [
            "unit_name",
            "fixed_baseline_pixels",
            "median_baseline_coverage_pct",
            "ghsl_mask",
            "sc_threshold_pct",
        ],
    ].sort_values("unit_name")
)

if not municipal_failures.empty:
    print("Profiles withheld or unavailable:")
    display(municipal_failures)

In [ ]:
# ============================================================
# 10. SIX MATCHED POI KERNELS
# ============================================================

poi_coordinates = {
    "Tacloban City Centre": (125.0015, 11.2434),
    "Palo Centre": (124.9904, 11.1577),
    "Guiuan Centre": (125.7232, 11.0312),
    "Alangalang Centre": (124.8465, 11.2071),
    "Ormoc City Centre": (124.6068, 11.0088),
    "Baybay City Centre": (124.7989, 10.6766),
}

poi_table = pd.DataFrame(
    [
        {
            **location,
            "longitude": poi_coordinates[location["poi_name"]][0],
            "latitude": poi_coordinates[location["poi_name"]][1],
            "color": LOCATION_COLORS[location["municipality_name"]],
        }
        for location in CORE_LOCATIONS
    ]
)


def poi_window(longitude, latitude, kernel_size, municipality_name):
    point = gpd.GeoSeries(
        [Point(longitude, latitude)],
        crs="EPSG:4326",
    ).to_crs(a2.rio.crs).iloc[0]

    x_index = int(np.abs(dnb["x"].values - point.x).argmin())
    y_index = int(np.abs(dnb["y"].values - point.y).argmin())
    half = kernel_size // 2

    x_slice = slice(max(0, x_index - half), min(dnb.sizes["x"], x_index + half + 1))
    y_slice = slice(max(0, y_index - half), min(dnb.sizes["y"], y_index + half + 1))

    parent = selected_raster_crs.loc[
        selected_raster_crs["unit_key"] == canonical_unit_name(municipality_name)
    ]
    if len(parent) != 1:
        raise ValueError(f"Expected one selected boundary for {municipality_name}.")

    parent_mask = selected_zone_id == int(parent.iloc[0]["profile_id"])
    local_cube = rq_cube.isel(x=x_slice, y=y_slice)
    local_support = (
        ghsl_mask.isel(x=x_slice, y=y_slice)
        & parent_mask.isel(x=x_slice, y=y_slice)
    )
    return local_cube, local_support


poi_profiles = []
poi_reports = []
poi_failures = []

for poi in poi_table.itertuples():
    local_cube, local_support = poi_window(
        poi.longitude,
        poi.latitude,
        POI_KERNEL_SIZE,
        poi.municipality_name,
    )

    for aggregation_days in (1, 4):
        try:
            profile, _, _, _, report = build_pixel_matched_profile(
                cube=local_cube,
                support_mask=local_support,
                aggregation_days=aggregation_days,
                unit_name=poi.poi_name,
                unit_type=f"POI {POI_KERNEL_SIZE}×{POI_KERNEL_SIZE}",
            )
            profile["municipality_name"] = poi.municipality_name
            poi_profiles.append(profile)
            report["municipality_name"] = poi.municipality_name
            poi_reports.append(report)
        except ValueError as error:
            poi_failures.append(
                {
                    "poi_name": poi.poi_name,
                    "municipality_name": poi.municipality_name,
                    "aggregation_days": aggregation_days,
                    "reason": str(error),
                }
            )

poi_profiles = pd.concat(poi_profiles, ignore_index=True) if poi_profiles else pd.DataFrame()
poi_reports = pd.DataFrame(poi_reports)
poi_failures = pd.DataFrame(poi_failures)

poi_daily = poi_profiles.loc[
    poi_profiles["aggregation_days"] == 1
].copy() if not poi_profiles.empty else pd.DataFrame()
poi_four_day = poi_profiles.loc[
    poi_profiles["aggregation_days"] == 4
].copy() if not poi_profiles.empty else pd.DataFrame()

missing_poi_profiles = [
    name for name in POI_ORDER
    if name not in set(poi_four_day["unit_name"])
]
if missing_poi_profiles:
    raise ValueError(
        "The fixed six-location design is incomplete. Missing POI profiles: "
        + ", ".join(missing_poi_profiles)
    )

poi_four_day["unit_name"] = pd.Categorical(
    poi_four_day["unit_name"], categories=POI_ORDER, ordered=True
)
poi_daily["unit_name"] = pd.Categorical(
    poi_daily["unit_name"], categories=POI_ORDER, ordered=True
)

print("Municipality profiles:", municipal_four_day["unit_name"].nunique())
print("POI profiles:", poi_four_day["unit_name"].nunique())

In [ ]:
# ============================================================
# 11. NGCP REGIONAL CONSISTENCY CHECK
# ============================================================


def build_ngcp_profile(path):
    if not path.exists():
        raise FileNotFoundError(f"NGCP CSV not found:\n{path}")

    raw = pd.read_csv(path, skiprows=2, low_memory=False)
    raw.columns = [str(column).strip() for column in raw.columns]

    daily = pd.DataFrame(
        {
            "date": pd.to_datetime(
                raw["DATE"].astype(str).str.strip(),
                dayfirst=True,
                errors="coerce",
            ).dt.normalize(),
            "load_mw": pd.to_numeric(raw["1"], errors="coerce"),
        }
    ).dropna(subset=["date", "load_mw"])

    daily = (
        daily
        .groupby("date", as_index=False)["load_mw"]
        .median()
        .sort_values("date")
    )
    daily = daily.loc[
        (daily["date"] >= ANALYSIS_START)
        & (daily["date"] <= PROFILE_END)
    ].copy()

    daily["block"] = np.floor_divide(
        (daily["date"] - EVENT_DATE).dt.days,
        4,
    ).astype(int)

    profile = daily.groupby("block", as_index=False)["load_mw"].median()
    profile["date_start"] = EVENT_DATE + pd.to_timedelta(profile["block"] * 4, unit="D")
    profile["date_end"] = profile["date_start"] + pd.Timedelta(days=3)

    baseline = profile.loc[
        (profile["date_start"] >= BASELINE_START)
        & (profile["date_end"] <= PRE_EVENT_END),
        "load_mw",
    ].median()

    if not np.isfinite(baseline) or baseline <= 0:
        raise ValueError("NGCP baseline is unavailable or non-positive.")

    profile["recovery_pct"] = 100.0 * profile["load_mw"] / baseline
    return profile, baseline


def safe_correlation(x, y):
    paired = pd.DataFrame(
        {
            "x": pd.to_numeric(x, errors="coerce"),
            "y": pd.to_numeric(y, errors="coerce"),
        }
    ).dropna()

    if len(paired) < 3 or paired["x"].nunique() < 2 or paired["y"].nunique() < 2:
        return np.nan

    return paired["x"].corr(paired["y"])


ngcp_four_day, ngcp_baseline_mw = build_ngcp_profile(NGCP_CSV_PATH)

regional_ngcp = (
    regional_four_day[["date_start", "recovery_pct", "spatial_coverage_pct"]]
    .merge(
        ngcp_four_day[["date_start", "recovery_pct"]].rename(
            columns={"recovery_pct": "ngcp_recovery_pct"}
        ),
        on="date_start",
        how="inner",
    )
    .dropna(subset=["recovery_pct", "ngcp_recovery_pct"])
)

post_bridge = regional_ngcp.loc[regional_ngcp["date_start"] >= EVENT_DATE]
bridge_r = safe_correlation(post_bridge["recovery_pct"], post_bridge["ngcp_recovery_pct"])
bridge_n = len(post_bridge)

if bridge_n >= 20 and np.isfinite(bridge_r) and bridge_r >= 0.70:
    bridge_decision = (
        "Regional construct consistency is supported. Local trajectories may be analysed "
        "as conditional functional NTL profiles, but not labelled as municipal electricity restoration."
    )
elif bridge_n >= 12 and np.isfinite(bridge_r) and bridge_r >= 0.50:
    bridge_decision = (
        "Regional consistency is moderate. Local trajectories remain exploratory and require "
        "stronger external evidence."
    )
else:
    bridge_decision = (
        "The municipality-clipped regional bridge is weak or sparsely observed. Do not bank "
        "on NGCP alignment for local interpretation in this configuration."
    )

bridge_summary = pd.DataFrame(
    [
        {
            "GHSL mask": SETTLEMENT_MASK,
            "SC threshold (%)": SPATIAL_COMPLETENESS_PCT,
            "Paired post-event composites": bridge_n,
            "Pearson r": bridge_r,
            "NGCP baseline (MW)": ngcp_baseline_mw,
            "Decision": bridge_decision,
        }
    ]
)

In [ ]:
# ============================================================
# 12. RECOVERY METRIC FUNCTIONS
# ============================================================


def expected_event_blocks(aggregation_days=4):
    inclusive_days = (PROFILE_END - EVENT_DATE).days + 1
    return int(np.ceil(inclusive_days / aggregation_days))


def longest_missing_run_days(profile, aggregation_days=4):
    expected_blocks = np.arange(
        0,
        int(np.floor((PROFILE_END - EVENT_DATE).days / aggregation_days)) + 1,
    )
    observed_blocks = set(
        profile.loc[
            (profile["date_start"] >= EVENT_DATE)
            & (profile["date_start"] <= PROFILE_END)
            & profile["recovery_pct"].notna(),
            "block",
        ].astype(int)
    )
    missing = np.array(
        [block not in observed_blocks for block in expected_blocks],
        dtype=int,
    )

    longest = current = 0
    for value in missing:
        current = current + 1 if value else 0
        longest = max(longest, current)

    return longest * aggregation_days


def observability_class(profile, aggregation_days=4):
    post = profile.loc[
        (profile["date_start"] >= EVENT_DATE)
        & (profile["date_start"] <= PROFILE_END)
    ]
    retained = int(post["recovery_pct"].notna().sum())
    expected = expected_event_blocks(aggregation_days)
    retained_pct = 100.0 * retained / expected if expected else np.nan
    max_gap = longest_missing_run_days(profile, aggregation_days)

    if (
        retained >= MIN_EVENT_COMPOSITES
        and retained_pct >= MIN_EVENT_RETENTION_PCT
        and max_gap <= MAX_INTERPRETABLE_GAP_DAYS
    ):
        label, flag = "interpretable with interval timing", "OBS_OK"
    elif retained >= MIN_EVENT_COMPOSITES and retained_pct >= MIN_EVENT_RETENTION_PCT:
        label, flag = "observation-limited", "OBS_LIMITED"
    else:
        label, flag = "not observable", "NOT_OBSERVABLE"

    return {
        "retained_composites": retained,
        "expected_composites": expected,
        "retained_pct": retained_pct,
        "max_gap_days": max_gap,
        "observability": label,
        "quality_flag": flag,
    }


def persistent_crossing(
    profile,
    threshold,
    value_column="recovery_pct",
    start_block=0,
):
    """First persistent threshold return at or after the observed impact block."""

    post = (
        profile.loc[
            (profile["date_start"] >= EVENT_DATE)
            & (profile["date_start"] <= PROFILE_END)
            & (profile["block"] >= start_block)
            & profile[value_column].notna()
        ]
        .sort_values("block")
        .set_index("block", drop=False)
    )

    for block, row in post.iterrows():
        required = list(range(int(block), int(block) + PERSISTENCE_BLOCKS))
        if not set(required).issubset(post.index):
            continue
        if not (post.loc[required, value_column] >= threshold).all():
            continue

        previous_below = post.loc[
            (post["block"] < block) & (post[value_column] < threshold)
        ]
        lower_day = int((row["date_start"] - EVENT_DATE).days)
        if not previous_below.empty:
            lower_day = max(
                0,
                int(
                    (
                        previous_below.iloc[-1]["date_end"]
                        + pd.Timedelta(days=1)
                        - EVENT_DATE
                    ).days
                ),
            )

        upper_day = int((row["date_start"] - EVENT_DATE).days)
        return {
            "day": upper_day,
            "lower_day": lower_day,
            "upper_day": upper_day,
            "date": row["date_start"],
            "observed_value": float(row[value_column]),
        }

    return None


def format_interval(crossing):
    if crossing is None:
        return None
    return f"{crossing['lower_day']}–{crossing['upper_day']} d"


def format_baseline_range(optimistic, conservative):
    if optimistic is None and conservative is None:
        return None
    earliest = optimistic["day"] if optimistic is not None else None
    latest = conservative["day"] if conservative is not None else None
    if earliest is not None and latest is not None:
        return f"{min(earliest, latest)}–{max(earliest, latest)} d"
    if earliest is not None:
        return f"≥{earliest} d; conservative crossing absent"
    return f"≤{latest} d; optimistic crossing absent"


def calculate_recovery_metrics(profile):
    """Calculate impact and persistent return-to-baseline milestones.

    T50/T80/T90 mean the first of two consecutive admissible four-day
    composites at or above 50/80/90% of the matched 60-day baseline,
    searched only after the observed Stage-1 nadir. They are not fractions
    of the shock-to-baseline amplitude.
    """

    profile = profile.sort_values("date_start").copy()
    obs = observability_class(profile, aggregation_days=4)
    event = profile.loc[
        (profile["date_start"] >= EVENT_DATE)
        & (profile["date_start"] <= PROFILE_END)
    ]
    impact_window = event.loc[
        event["date_start"] <= EVENT_DATE + pd.Timedelta(days=59)
    ].dropna(subset=["recovery_pct"])

    impact_row = None
    if impact_window.empty:
        impact_recovery = impact_drop = np.nan
        impact_drop_low = impact_drop_high = np.nan
        impact_date, impact_block = pd.NaT, np.nan
    else:
        impact_row = impact_window.loc[impact_window["recovery_pct"].idxmin()]
        impact_recovery = float(impact_row["recovery_pct"])
        impact_date = impact_row["date_start"]
        impact_block = int(impact_row["block"])
        impact_drop = 100.0 - impact_recovery
        impact_drop_low = 100.0 - float(impact_row["recovery_high_pct"])
        impact_drop_high = 100.0 - float(impact_row["recovery_low_pct"])

    crossings = {}
    for threshold in (50, 80, 90):
        threshold_lost = bool(
            impact_row is not None and impact_recovery < threshold
        )

        if threshold_lost:
            central = persistent_crossing(
                profile,
                threshold,
                "recovery_pct",
                start_block=impact_block,
            )
            optimistic_lost = (
                pd.notna(impact_row["recovery_high_pct"])
                and float(impact_row["recovery_high_pct"]) < threshold
            )
            conservative_lost = (
                pd.notna(impact_row["recovery_low_pct"])
                and float(impact_row["recovery_low_pct"]) < threshold
            )
            optimistic = (
                persistent_crossing(
                    profile,
                    threshold,
                    "recovery_high_pct",
                    start_block=impact_block,
                )
                if optimistic_lost
                else None
            )
            conservative = (
                persistent_crossing(
                    profile,
                    threshold,
                    "recovery_low_pct",
                    start_block=impact_block,
                )
                if conservative_lost
                else None
            )
        else:
            central = optimistic = conservative = None

        if impact_row is None:
            status = "not observable in the 0–59 day impact window"
        elif not threshold_lost:
            status = "threshold not lost at observed nadir"
        elif central is not None:
            status = "supported; persistent return after observed nadir"
        elif obs["quality_flag"] == "NOT_OBSERVABLE":
            status = "not observable"
        elif obs["quality_flag"] == "OBS_LIMITED":
            status = "not identifiable: observation-limited"
        else:
            status = f"not recovered by {(PROFILE_END - EVENT_DATE).days} d"

        crossings[threshold] = {
            "central": central,
            "optimistic": optimistic,
            "conservative": conservative,
            "lost": threshold_lost,
            "status": status,
        }

    post_observed = event.dropna(subset=["recovery_pct"]).copy()
    slope = np.nan
    if not post_observed.empty and pd.notna(impact_date):
        slope_end_day = (
            crossings[90]["central"]["day"]
            if crossings[90]["central"] is not None
            else int((PROFILE_END - EVENT_DATE).days)
        )
        slope_data = post_observed.loc[
            (post_observed["date_start"] >= impact_date)
            & (
                post_observed["date_start"]
                <= EVENT_DATE + pd.Timedelta(days=slope_end_day)
            )
        ]
        if len(slope_data) >= 3:
            x = (slope_data["date_start"] - EVENT_DATE).dt.days.to_numpy(dtype=float)
            y = slope_data["recovery_pct"].to_numpy(dtype=float)
            slope = float(np.polyfit(x, y, 1)[0])

    stage3 = event.loc[
        (event["date_start"] >= EVENT_DATE + pd.Timedelta(days=120))
        & (event["date_start"] <= EVENT_DATE + pd.Timedelta(days=179))
    ].dropna(subset=["recovery_pct"])
    stability_mad = (
        float(np.median(np.abs(stage3["recovery_pct"] - stage3["recovery_pct"].median())))
        if not stage3.empty
        else np.nan
    )
    stable_share = (
        float(stage3["recovery_pct"].between(90, 110).sum() * 100.0 / len(stage3))
        if not stage3.empty
        else np.nan
    )

    result = {
        "unit_name": profile["unit_name"].iloc[0],
        "unit_type": profile["unit_type"].iloc[0],
        "metric_definition": (
            "persistent return to 50/80/90% of the matched 60-day baseline "
            "after the observed 0–59 day nadir"
        ),
        "ghsl_mask": SETTLEMENT_MASK,
        "ghsl_classes": str(GHSL_MASKS[SETTLEMENT_MASK]),
        "sc_threshold_pct": SPATIAL_COMPLETENESS_PCT,
        "baseline_days": BASELINE_DAYS,
        "baseline_start": BASELINE_START.date(),
        "baseline_end": PRE_EVENT_END.date(),
        "impact_date": impact_date,
        "impact_block": impact_block,
        "impact_recovery_pct": impact_recovery,
        "impact_drop_pct": impact_drop,
        "impact_drop_range_pct": (
            f"{impact_drop_low:.1f}–{impact_drop_high:.1f}"
            if np.isfinite(impact_drop_low) and np.isfinite(impact_drop_high)
            else None
        ),
        "recovery_slope_pct_per_day": slope,
        "observed_days_below_baseline": int(
            (post_observed["recovery_pct"] < 100).sum() * 4
        ),
        "stage3_stability_mad_pct": stability_mad,
        "stage3_within_90_110_pct": stable_share,
        "median_event_sc_pct": float(event["spatial_coverage_pct"].median()),
        "minimum_event_sc_pct": float(event["spatial_coverage_pct"].min()),
        **obs,
    }

    for threshold, crossing in crossings.items():
        result[f"T{threshold}_threshold_lost"] = crossing["lost"]
        result[f"T{threshold}_day"] = (
            crossing["central"]["day"]
            if crossing["central"] is not None
            else np.nan
        )
        result[f"T{threshold}_observation_interval"] = format_interval(
            crossing["central"]
        )
        result[f"T{threshold}_baseline_range"] = format_baseline_range(
            crossing["optimistic"],
            crossing["conservative"],
        )
        result[f"T{threshold}_status"] = crossing["status"]

    return result

In [ ]:
# ============================================================
# 13. MUNICIPALITY AND POI METRIC TABLES
# ============================================================

municipal_metrics = pd.DataFrame(
    [
        calculate_recovery_metrics(group)
        for _, group in municipal_four_day.groupby("unit_name", sort=True)
    ]
)

poi_metrics = (
    pd.DataFrame(
        [
            calculate_recovery_metrics(group)
            for _, group in poi_four_day.groupby("unit_name", sort=True)
        ]
    )
    if not poi_four_day.empty
    else pd.DataFrame()
)

In [ ]:
# ============================================================
# 14. STAGE SUMMARIES
# ============================================================


def summarise_stages(profile_table):
    rows = []

    for unit_name, profile in profile_table.groupby("unit_name", sort=True):
        for stage_name, (stage_start, stage_end) in STAGE_WINDOWS.items():
            stage = profile.loc[
                (profile["date_start"] >= stage_start)
                & (profile["date_end"] <= stage_end)
            ]
            inclusive_days = (stage_end - stage_start).days + 1
            expected = int(np.ceil(inclusive_days / 4))
            observed = stage.dropna(subset=["recovery_pct"])

            rows.append(
                {
                    "unit_name": unit_name,
                    "stage": stage_name,
                    "expected_composites": expected,
                    "observed_composites": len(observed),
                    "retained_pct": 100.0 * len(observed) / expected if expected else np.nan,
                    "median_recovery_pct": observed["recovery_pct"].median(),
                    "recovery_q25_pct": observed["recovery_pct"].quantile(0.25),
                    "recovery_q75_pct": observed["recovery_pct"].quantile(0.75),
                    "median_sc_pct": stage["spatial_coverage_pct"].median(),
                    "minimum_sc_pct": stage["spatial_coverage_pct"].min(),
                    "not_observable_composites": int(stage["recovery_pct"].isna().sum()),
                }
            )

    return pd.DataFrame(rows)


municipal_stage_summary = summarise_stages(municipal_four_day)
poi_stage_summary = (
    summarise_stages(poi_four_day)
    if not poi_four_day.empty
    else pd.DataFrame()
)

In [ ]:
# ============================================================
# 17. BUILD REGIONAL STAGE MAPS
# ============================================================


def stage_map_from_profile(stage_name, stage_start, stage_end):
    if stage_name == "Baseline":
        return regional_ntl0_four_day.compute()

    eligible = regional_four_day.loc[
        (regional_four_day["date_start"] >= stage_start)
        & (regional_four_day["date_end"] <= stage_end)
        & regional_four_day["recovery_pct"].notna()
    ]
    blocks = eligible["block"].astype(int).to_numpy()

    if len(blocks) == 0:
        return xr.full_like(
            regional_composites_four_day.isel(block=0),
            np.nan,
        ).compute()

    return (
        regional_composites_four_day
        .sel(block=blocks)
        .median(dim="block", skipna=True)
        .compute()
    )


stage_maps = {
    stage_name: stage_map_from_profile(stage_name, stage_start, stage_end)
    for stage_name, (stage_start, stage_end) in STAGE_WINDOWS.items()
}

map_values = []

for stage_map in stage_maps.values():
    values = np.asarray(stage_map.values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size:
        map_values.append(values)

if not map_values:
    raise ValueError("No stage-map radiance values were available.")

map_color_max = max(float(np.nanpercentile(np.concatenate(map_values), 98)), 1.0)


def extract_line_coordinates(geodataframe):
    x_values = []
    y_values = []

    def add_line(line):
        x, y = line.xy
        x_values.extend([*x, None])
        y_values.extend([*y, None])

    for geometry in geodataframe.geometry:
        if geometry is None or geometry.is_empty:
            continue
        if geometry.geom_type == "LineString":
            add_line(geometry)
        elif geometry.geom_type == "MultiLineString":
            for line in geometry.geoms:
                add_line(line)
        elif geometry.geom_type == "Polygon":
            add_line(geometry.exterior)
        elif geometry.geom_type == "MultiPolygon":
            for polygon in geometry.geoms:
                add_line(polygon.exterior)

    return x_values, y_values


selected_boundaries = selected_raster_crs.copy()
selected_boundaries["geometry"] = selected_boundaries.geometry.boundary
boundary_x, boundary_y = extract_line_coordinates(selected_boundaries)

study_bounds = dnb.rio.bounds()
roads_for_map = roads_raster_crs.cx[
    study_bounds[0]:study_bounds[2],
    study_bounds[1]:study_bounds[3],
].copy()

if len(roads_for_map) > 12000:
    roads_for_map = roads_for_map.iloc[:: max(1, len(roads_for_map) // 12000)].copy()

road_x, road_y = extract_line_coordinates(roads_for_map)

In [ ]:
# ============================================================
# SHARED VISUAL LANGUAGE AND EXPORT HELPERS
# ============================================================

RECOVERY_COLORSCALE = [
    [0.00, "#8E1B1B"],
    [0.35, "#E67E22"],
    [0.70, "#F4D03F"],
    [100 / 140, "#F5F5F5"],
    [0.86, "#00F7FF"],
    [1.00, "#4800FF"],
]
SC_GREEN_COLORSCALE = [
    [0.00, "rgba(255,255,255,0.00)"],
    [0.20, "rgba(220,242,215,0.25)"],
    [0.40, "rgba(166,219,160,0.45)"],
    [0.60, "rgba(90,174,110,0.65)"],
    [0.80, "rgba(25,130,70,0.82)"],
    [1.00, "rgba(0,88,43,0.98)"],
]
MILESTONE_COLORS = {50: "#56B4E9", 80: "#009E73", 90: "#D55E00"}
MILESTONE_SYMBOLS = {50: "circle", 80: "square", 90: "diamond"}


def subset_bbox(data_array, bounds):
    x_min, y_min, x_max, y_max = bounds
    x_values = data_array["x"].values
    y_values = data_array["y"].values
    x_slice = slice(x_min, x_max) if x_values[0] < x_values[-1] else slice(x_max, x_min)
    y_slice = slice(y_min, y_max) if y_values[0] < y_values[-1] else slice(y_max, y_min)
    return data_array.sel(x=x_slice, y=y_slice)


def square_bounds(total_bounds, pad_fraction=0.05):
    x_min, y_min, x_max, y_max = map(float, total_bounds)
    side = max(x_max - x_min, y_max - y_min)
    side *= 1.0 + 2.0 * pad_fraction
    x_mid = (x_min + x_max) / 2.0
    y_mid = (y_min + y_max) / 2.0
    return (
        x_mid - side / 2.0,
        y_mid - side / 2.0,
        x_mid + side / 2.0,
        y_mid + side / 2.0,
    )


def polygon_coordinates(geometry):
    x_values, y_values = [], []
    polygons = [geometry] if geometry.geom_type == "Polygon" else list(geometry.geoms)
    for polygon in polygons:
        x, y = polygon.exterior.xy
        x_values.extend([*x, None])
        y_values.extend([*y, None])
    return x_values, y_values


def hex_to_rgba(hex_color, alpha):
    value = hex_color.lstrip("#")
    red, green, blue = (int(value[index:index + 2], 16) for index in (0, 2, 4))
    return f"rgba({red},{green},{blue},{alpha})"


def axis_id(prefix, number):
    return prefix if number == 1 else f"{prefix}{number}"


def rectangle_line(bounds):
    x_min, y_min, x_max, y_max = bounds
    return (
        [x_min, x_max, x_max, x_min, x_min],
        [y_min, y_min, y_max, y_max, y_min],
    )


def safe_stem(value):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").lower()


def save_plotly_figure(figure, stem):
    html_path = FIGURE_DIR / f"{stem}.html"
    figure.write_html(html_path, include_plotlyjs="cdn")
    try:
        figure.write_image(FIGURE_DIR / f"{stem}.png", scale=2)
    except Exception as error:
        print(f"PNG skipped for {stem}: {error}")
    return html_path

In [ ]:
# ============================================================
# SHARED VISUAL LANGUAGE, MAP GEOMETRY, AND EXPORT HELPERS
# ============================================================

PLOT_FONT = "Arial"
PLOT_TEXT_COLOR = "#243B5A"
PLOT_GRID_COLOR = "#E8EDF3"
EVENT_COLOR = "#0057FF"
BASELINE_COLOR = "#6C7882"
NTL_DISPLAY_MAX = 5.0

RECOVERY_COLORSCALE = [
    [0.00, "#8E1B1B"],
    [0.35, "#E67E22"],
    [0.70, "#F4D03F"],
    [100 / 140, "#F5F5F5"],
    [0.86, "#00F7FF"],
    [1.00, "#4800FF"],
]

SC_GREEN_COLORSCALE = [
    [0.00, "rgba(255,255,255,0.00)"],
    [0.20, "rgba(220,242,215,0.25)"],
    [0.40, "rgba(166,219,160,0.45)"],
    [0.60, "rgba(90,174,110,0.65)"],
    [0.80, "rgba(25,130,70,0.82)"],
    [1.00, "rgba(0,88,43,0.98)"],
]

MILESTONE_COLORS = {
    50: "#56B4E9",
    80: "#009E73",
    90: "#D55E00",
}

MILESTONE_SYMBOLS = {
    50: "circle",
    80: "square",
    90: "diamond",
}


def subset_bbox(data_array, bounds):
    x_min, y_min, x_max, y_max = bounds
    x_values = data_array["x"].values
    y_values = data_array["y"].values
    x_slice = slice(x_min, x_max) if x_values[0] < x_values[-1] else slice(x_max, x_min)
    y_slice = slice(y_min, y_max) if y_values[0] < y_values[-1] else slice(y_max, y_min)
    return data_array.sel(x=x_slice, y=y_slice)


def square_bounds(total_bounds, pad_fraction=0.05):
    x_min, y_min, x_max, y_max = map(float, total_bounds)
    side = max(x_max - x_min, y_max - y_min)
    side *= 1.0 + 2.0 * pad_fraction
    x_mid = (x_min + x_max) / 2.0
    y_mid = (y_min + y_max) / 2.0
    return (
        x_mid - side / 2.0,
        y_mid - side / 2.0,
        x_mid + side / 2.0,
        y_mid + side / 2.0,
    )


def polygon_coordinates(geometry):
    x_values, y_values = [], []
    polygons = [geometry] if geometry.geom_type == "Polygon" else list(geometry.geoms)
    for polygon in polygons:
        x, y = polygon.exterior.xy
        x_values.extend([*x, None])
        y_values.extend([*y, None])
    return x_values, y_values


def hex_to_rgba(hex_color, alpha):
    value = hex_color.lstrip("#")
    red, green, blue = (int(value[index:index + 2], 16) for index in (0, 2, 4))
    return f"rgba({red},{green},{blue},{alpha})"


def axis_id(prefix, number):
    return prefix if number == 1 else f"{prefix}{number}"


def rectangle_line(bounds):
    x_min, y_min, x_max, y_max = bounds
    return (
        [x_min, x_max, x_max, x_min, x_min],
        [y_min, y_min, y_max, y_max, y_min],
    )


def degree_minute_label(value, coordinate):
    absolute_value = abs(float(value))
    degrees = int(np.floor(absolute_value))
    minutes = int(round((absolute_value - degrees) * 60))
    if minutes == 60:
        degrees += 1
        minutes = 0
    if coordinate == "longitude":
        direction = "E" if value >= 0 else "W"
    else:
        direction = "N" if value >= 0 else "S"
    return f"{degrees}°{minutes:02d}′{direction}"


def map_ticks(bounds, interval=0.5):
    x_values = np.arange(
        np.ceil(bounds[0] / interval) * interval,
        np.floor(bounds[2] / interval) * interval + 0.001,
        interval,
    )
    y_values = np.arange(
        np.ceil(bounds[1] / interval) * interval,
        np.floor(bounds[3] / interval) * interval + 0.001,
        interval,
    )
    return (
        x_values,
        [degree_minute_label(value, "longitude") for value in x_values],
        y_values,
        [degree_minute_label(value, "latitude") for value in y_values],
    )


def safe_stem(value):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_").lower()


def save_plotly_figure(figure, stem):
    html_path = FIGURE_DIR / f"{stem}.html"
    figure.write_html(html_path, include_plotlyjs="cdn")
    try:
        figure.write_image(FIGURE_DIR / f"{stem}.png", scale=2)
    except Exception as error:
        print(f"PNG skipped for {stem}: {error}")
    return html_path


# Shared display geometries. These are defined once and reused.
selected_display = selected_municipalities.to_crs("EPSG:4326").copy()
roads_display = roads.to_crs("EPSG:4326").copy()
haiyan_track_display = haiyan_track.to_crs("EPSG:4326").copy()

OVERVIEW_BOUNDS = square_bounds(selected_display.total_bounds, pad_fraction=0.06)
OVERVIEW_X_TICKS, OVERVIEW_X_LABELS, OVERVIEW_Y_TICKS, OVERVIEW_Y_LABELS = map_ticks(
    OVERVIEW_BOUNDS,
    interval=0.5,
)

tacloban_display = selected_display.loc[
    selected_display["unit_name"] == "Tacloban City"
].copy()
if tacloban_display.empty:
    raise ValueError("Tacloban City was not found in selected_display.")

TACLOBAN_INSET_BOUNDS = square_bounds(
    tacloban_display.total_bounds,
    pad_fraction=0.18,
)

haiyan_track_x, haiyan_track_y = extract_line_coordinates(haiyan_track_display)

# Display-only background: median of every valid MQF == 0 observation in the
# 60-day baseline. It is intentionally not restricted to G3 or the study mask.
baseline_valid_dnb = dnb.sel(date=slice(BASELINE_START, PRE_EVENT_END)).where(
    mqf.sel(date=slice(BASELINE_START, PRE_EVENT_END)) == 0
)
baseline_valid_count = baseline_valid_dnb.count("date")
median_60day_ntl = (
    baseline_valid_dnb
    .median("date", skipna=True)
    .where(baseline_valid_count >= 1)
    .compute()
)
median_60day_ntl.name = "median_60day_valid_observation_ntl"


## 1. Six recovery settings, fixed before interpretation

The sample is fixed at six municipalities and six matched 5×5 POI kernels. It covers the principal eastern impact corridor, first landfall, an inland comparison, and two western urban comparisons. The western locations remain affected comparisons—not unaffected controls. The Haiyan/Yolanda path shapefile is overlaid on every spatial figure to make the exposure logic visible; the track is contextual and is not a local hazard-intensity measure. The design is consistent with documented landfall and heterogeneous physical recovery evidence ([JICA recovery plan](https://openjicareport.jica.go.jp/pdf/12233946_01.pdf); [Sheykhmousa et al., 2019](https://doi.org/10.3390/rs11101174)).

In [ ]:
# ============================================================
# FIGURE 1. SIX-LOCATION RATIONALE
# ============================================================

selection_summary = pd.DataFrame(CORE_LOCATIONS).rename(
    columns={
        "recovery_setting": "Recovery setting",
        "core_location": "Core location",
        "why": "Why it is here",
    }
)[["Recovery setting", "Core location", "Why it is here"]]

sample_note = (
    f"Municipalities: n={municipal_four_day['unit_name'].nunique()} | "
    f"POI kernels: n={poi_four_day['unit_name'].nunique()} | "
    f"Kernel: {POI_KERNEL_SIZE}×{POI_KERNEL_SIZE} VIIRS pixels"
)

fig_selection_rationale = go.Figure(
    go.Table(
        columnwidth=[0.24, 0.16, 0.60],
        header=dict(
            values=[f"<b>{column}</b>" for column in selection_summary.columns],
            fill_color="#243B5A",
            font=dict(color="white", size=20),
            align="left",
            height=46,
        ),
        cells=dict(
            values=[selection_summary[column] for column in selection_summary.columns],
            fill_color=[["#F5F8FB", "#FFFFFF"] * 3] * 3,
            font=dict(color="#243B5A", size=17),
            align="left",
            height=66,
        ),
    )
)
fig_selection_rationale.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    width=1800,
    height=720,
    title=dict(
        text=f"Why these six locations?<br><sup>{sample_note}</sup>",
        x=0.5,
        xanchor="center",
        font=dict(size=30, color="#243B5A"),
    ),
    margin=dict(l=35, r=35, t=115, b=30),
)
fig_selection_rationale.show()

In [ ]:
# ============================================================
# FIGURE 2. SIX MUNICIPALITIES, POIs, AND HAIYAN PATH
# ============================================================


# ------------------------------------------------------------
# Coordinate-label helper
# ------------------------------------------------------------

def degree_minute_label(value, coordinate):
    absolute_value = abs(float(value))
    degrees = int(np.floor(absolute_value))
    minutes = int(round((absolute_value - degrees) * 60))

    if minutes == 60:
        degrees += 1
        minutes = 0

    if coordinate == "longitude":
        direction = "E" if value >= 0 else "W"
    else:
        direction = "N" if value >= 0 else "S"

    return f"{degrees}°{minutes:02d}′{direction}"


# ------------------------------------------------------------
# Display layers
# ------------------------------------------------------------

selected_display = (
    selected_municipalities
    .to_crs("EPSG:4326")
    .copy()
)

roads_display = (
    roads
    .to_crs("EPSG:4326")
    .copy()
)

haiyan_track_display = (
    haiyan_track
    .to_crs("EPSG:4326")
    .copy()
)


# ------------------------------------------------------------
# Common square map extent
# ------------------------------------------------------------

overview_bounds = square_bounds(
    selected_display.total_bounds,
    pad_fraction=0.06,
)

x_tick_values = np.arange(
    np.ceil(overview_bounds[0] * 2) / 2,
    np.floor(overview_bounds[2] * 2) / 2 + 0.001,
    0.5,
)

y_tick_values = np.arange(
    np.ceil(overview_bounds[1] * 2) / 2,
    np.floor(overview_bounds[3] * 2) / 2 + 0.001,
    0.5,
)

x_tick_labels = [
    degree_minute_label(value, "longitude")
    for value in x_tick_values
]

y_tick_labels = [
    degree_minute_label(value, "latitude")
    for value in y_tick_values
]


# ------------------------------------------------------------
# Roads within the displayed extent
# ------------------------------------------------------------

roads_overview = roads_display.cx[
    overview_bounds[0]:overview_bounds[2],
    overview_bounds[1]:overview_bounds[3],
].copy()

if len(roads_overview) > 9000:
    roads_overview = roads_overview.iloc[
        ::max(1, len(roads_overview) // 9000)
    ].copy()

overview_road_x, overview_road_y = (
    extract_line_coordinates(roads_overview)
)


# ------------------------------------------------------------
# Haiyan path
# ------------------------------------------------------------

haiyan_track_x, haiyan_track_y = (
    extract_line_coordinates(haiyan_track_display)
)


# ------------------------------------------------------------
# Tacloban inset
# ------------------------------------------------------------

tacloban_display = selected_display.loc[
    selected_display["unit_name"] == "Tacloban City"
].copy()

if tacloban_display.empty:
    raise ValueError(
        "Tacloban City was not found in selected_display."
    )

tacloban_inset_bounds = square_bounds(
    tacloban_display.total_bounds,
    pad_fraction=0.22,
)


# ------------------------------------------------------------
# POI label offsets
# Positive ax moves right; positive ay moves downward
# ------------------------------------------------------------

POI_LABEL_OFFSETS = {
    "Tacloban City": (55, -42),
    "Palo": (55, 34),
    "Guiuan": (58, -12),
    "Alangalang": (-70, -30),
    "Ormoc City": (-65, -30),
    "Baybay City": (-65, -2),
}


# ------------------------------------------------------------
# Build figure
# ------------------------------------------------------------

fig_locations = go.Figure()
map_annotations = []


# ------------------------------------------------------------
# Main roads
# ------------------------------------------------------------

fig_locations.add_trace(
    go.Scatter(
        x=overview_road_x,
        y=overview_road_y,
        mode="lines",
        line=dict(
            color="#C7D0DA",
            width=1.35,
        ),
        name="Roads",
        legend="legend2",
        legendgroup="map-context",
        hoverinfo="skip",
    )
)


# ------------------------------------------------------------
# Main Haiyan path
# ------------------------------------------------------------

fig_locations.add_trace(
    go.Scatter(
        x=haiyan_track_x,
        y=haiyan_track_y,
        mode="lines",
        line=dict(
            color="#111111",
            width=4.0,
        ),
        name="Haiyan Path",
        legend="legend2",
        legendgroup="map-context",
        hovertemplate=(
            "Haiyan Path"
            "<extra></extra>"
        ),
    )
)


# ------------------------------------------------------------
# Main city and municipality boundaries
# ------------------------------------------------------------

for municipality_name in LOCATION_ORDER:
    unit = selected_display.loc[
        selected_display["unit_name"]
        == municipality_name
    ]

    if len(unit) != 1:
        raise ValueError(
            f"Expected one boundary for {municipality_name}; "
            f"found {len(unit)}."
        )

    color = LOCATION_COLORS[municipality_name]

    polygon_x, polygon_y = polygon_coordinates(
        unit.geometry.iloc[0]
    )

    fig_locations.add_trace(
        go.Scatter(
            x=polygon_x,
            y=polygon_y,
            mode="lines",
            fill="toself",
            fillcolor=hex_to_rgba(color, 0.14),
            line=dict(
                color=color,
                width=2.4,
            ),
            name=DISPLAY_NAMES[municipality_name],
            legendgroup=(
                f"municipality-{municipality_name}"
            ),
            hovertemplate=(
                f"{DISPLAY_NAMES[municipality_name]}"
                "<extra></extra>"
            ),
        )
    )


# ------------------------------------------------------------
# Main characteristic POIs
# Transparent squares with POI-anchored labels
# ------------------------------------------------------------

for poi in poi_table.itertuples():
    municipality_name = poi.municipality_name
    color = poi.color

    label_ax, label_ay = POI_LABEL_OFFSETS.get(
        municipality_name,
        (50, -20),
    )

    poi_label = (
        f"<b>{DISPLAY_NAMES[municipality_name]} "
        f"({POI_KERNEL_SIZE}×{POI_KERNEL_SIZE})</b>"
    )

    # Transparent POI square.
    fig_locations.add_trace(
        go.Scatter(
            x=[poi.longitude],
            y=[poi.latitude],
            mode="markers",
            marker=dict(
                size=15,
                color="rgba(255,255,255,0)",
                symbol="square",
                line=dict(
                    color=color,
                    width=3.0,
                ),
            ),
            showlegend=False,
            hovertemplate=(
                f"{poi.poi_name}<br>"
                f"{POI_KERNEL_SIZE}×{POI_KERNEL_SIZE} kernel"
                "<extra></extra>"
            ),
        )
    )

    # Offset background creates a restrained drop shadow.
    map_annotations.append(
        dict(
            x=poi.longitude,
            y=poi.latitude,
            xref="x",
            yref="y",
            text=poi_label,
            showarrow=True,
            ax=label_ax + 4,
            ay=label_ay + 4,
            axref="pixel",
            ayref="pixel",
            arrowhead=0,
            arrowwidth=0.1,
            arrowcolor="rgba(0,0,0,0)",
            standoff=9,
            xanchor=(
                "left"
                if label_ax > 0
                else "right"
            ),
            yanchor="middle",
            align=(
                "left"
                if label_ax > 0
                else "right"
            ),
            font=dict(
                size=16,
                color="rgba(0,0,0,0)",
            ),
            bgcolor="rgba(36,59,90,0.18)",
            bordercolor="rgba(0,0,0,0)",
            borderwidth=0,
            borderpad=3,
        )
    )

    # Visible POI label and leader line.
    map_annotations.append(
        dict(
            x=poi.longitude,
            y=poi.latitude,
            xref="x",
            yref="y",
            text=poi_label,
            showarrow=True,
            ax=label_ax,
            ay=label_ay,
            axref="pixel",
            ayref="pixel",
            arrowhead=0,
            arrowsize=1,
            arrowwidth=1.4,
            arrowcolor=color,

            # Stop the line at the POI square.
            standoff=9,

            xanchor=(
                "left"
                if label_ax > 0
                else "right"
            ),
            yanchor="middle",
            align=(
                "left"
                if label_ax > 0
                else "right"
            ),
            font=dict(
                size=16,
                color=color,
            ),
            bgcolor="rgba(255,255,255,0.94)",
            bordercolor=hex_to_rgba(color, 0.55),
            borderwidth=1,
            borderpad=3,
        )
    )


# ------------------------------------------------------------
# Tacloban inset roads
# ------------------------------------------------------------

fig_locations.add_trace(
    go.Scatter(
        x=overview_road_x,
        y=overview_road_y,
        mode="lines",
        line=dict(
            color="#C7D0DA",
            width=1.1,
        ),
        showlegend=False,
        hoverinfo="skip",
        xaxis="x2",
        yaxis="y2",
    )
)


# ------------------------------------------------------------
# Tacloban inset Haiyan path
# ------------------------------------------------------------

fig_locations.add_trace(
    go.Scatter(
        x=haiyan_track_x,
        y=haiyan_track_y,
        mode="lines",
        line=dict(
            color="#111111",
            width=2.4,
        ),
        showlegend=False,
        hoverinfo="skip",
        xaxis="x2",
        yaxis="y2",
    )
)


# ------------------------------------------------------------
# Tacloban inset municipality boundaries
# ------------------------------------------------------------

for municipality_name in LOCATION_ORDER:
    unit = selected_display.loc[
        selected_display["unit_name"]
        == municipality_name
    ]

    if unit.empty:
        continue

    color = LOCATION_COLORS[municipality_name]

    polygon_x, polygon_y = polygon_coordinates(
        unit.geometry.iloc[0]
    )

    fig_locations.add_trace(
        go.Scatter(
            x=polygon_x,
            y=polygon_y,
            mode="lines",
            fill="toself",
            fillcolor=hex_to_rgba(color, 0.10),
            line=dict(
                color=color,
                width=2.2,
            ),
            showlegend=False,
            hoverinfo="skip",
            xaxis="x2",
            yaxis="y2",
        )
    )


# ------------------------------------------------------------
# Tacloban inset POIs
# Non-Tacloban points are clipped by the inset axes
# ------------------------------------------------------------

for poi in poi_table.itertuples():
    fig_locations.add_trace(
        go.Scatter(
            x=[poi.longitude],
            y=[poi.latitude],
            mode="markers",
            marker=dict(
                size=13,
                color="rgba(255,255,255,0)",
                symbol="square",
                line=dict(
                    color=poi.color,
                    width=2.6,
                ),
            ),
            showlegend=False,
            hovertemplate=(
                f"{poi.poi_name}<br>"
                f"{POI_KERNEL_SIZE}×{POI_KERNEL_SIZE} kernel"
                "<extra></extra>"
            ),
            xaxis="x2",
            yaxis="y2",
        )
    )


# ------------------------------------------------------------
# Tacloban inset title
# ------------------------------------------------------------

map_annotations.append(
    dict(
        x=0.5,
        y=1.03,
        xref="x2 domain",
        yref="y2 domain",
        text="<b>Tacloban</b>",
        showarrow=False,
        xanchor="center",
        yanchor="bottom",
        font=dict(
            size=24,
            color="#243B5A",
        ),
        bgcolor="rgba(255,255,255,0.85)",
    )
)


# ------------------------------------------------------------
# Figure layout
# ------------------------------------------------------------

fig_locations.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",

    # Square figure canvas.
    width=1000,
    height=1000,

    # Main longitude axis.
    xaxis=dict(
        domain=[0.00, 0.70],
        title=None,
        range=[
            overview_bounds[0],
            overview_bounds[2],
        ],
        tickmode="array",
        tickvals=x_tick_values,
        ticktext=x_tick_labels,
        ticks="outside",
        ticklen=5,
        showgrid=True,
        gridcolor="rgba(36,59,90,0.11)",
        gridwidth=1,
        zeroline=False,
        constrain="domain",
    ),

    # Main latitude axis.
    yaxis=dict(
        domain=[0.00, 1.00],
        title=None,
        range=[
            overview_bounds[1],
            overview_bounds[3],
        ],
        tickmode="array",
        tickvals=y_tick_values,
        ticktext=y_tick_labels,
        ticks="outside",
        ticklen=5,
        showgrid=True,
        gridcolor="rgba(36,59,90,0.11)",
        gridwidth=1,
        zeroline=False,
        scaleanchor="x",
        scaleratio=1,
        constrain="domain",
    ),

    # Tacloban inset longitude axis.
    xaxis2=dict(
        domain=[0.47, 0.68],
        anchor="y2",
        range=[
            tacloban_inset_bounds[0],
            tacloban_inset_bounds[2],
        ],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        mirror=True,
        linecolor="#243B5A",
        linewidth=2,
        constrain="domain",
    ),

    # Tacloban inset latitude axis.
    yaxis2=dict(
        domain=[0.68, 0.94],
        anchor="x2",
        range=[
            tacloban_inset_bounds[1],
            tacloban_inset_bounds[3],
        ],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        mirror=True,
        linecolor="#243B5A",
        linewidth=2,
        scaleanchor="x2",
        scaleratio=1,
        constrain="domain",
    ),

    # City and municipality legend.
    legend=dict(
        title=dict(
            text="<b>City / Municipality</b>",
            side="top",
            font=dict(size=18),
        ),
        orientation="h",
        x=0.00,
        y=0.10,
        xanchor="left",
        yanchor="top",
        entrywidth=80,
        entrywidthmode="pixels",
        itemwidth=30,
        traceorder="normal",
        itemsizing="constant",
        tracegroupgap=2,
        font=dict(size=16),
        bgcolor="rgba(0,0,0,0.05)",
        bordercolor="rgba(36,59,90,0.20)",
        borderwidth=1,
    ),

    # Roads and Haiyan-path legend.
    legend2=dict(
        orientation="h",
        x=0.53,
        y=0.62,
        xanchor="left",
        yanchor="bottom",
        font=dict(size=18),
        bgcolor="rgba(0,0,0,0.05)",
        bordercolor="rgba(36,59,90,0.20)",
        borderwidth=1,
    ),

    font=dict(
        family="Arial",
        size=14,
        color="#243B5A",
    ),

    margin=dict(
        l=90,
        r=0,
        t=0,
        b=10,
    ),

    annotations=map_annotations,
)

fig_locations.show()

## 3. Settlement support and the median pre-Haiyan baseline

In [ ]:
# ============================================================
# FIGURE 4. GHSL G3 SUPPORT WITH TACLOBAN INSET
# ============================================================

GHSL_CLASS_COLORS = {
    11: "#cdf57a",
    12: "#abcd66",
    13: "#375623",
    21: "#ffff00",
    22: "#a87000",
    23: "#732600",
    30: "#ff0000",
}

G3_CLASSES = (22, 23, 30)

G3_CLASS_LABELS = {
    22: "Semi-dense urban",
    23: "Dense urban",
    30: "Urban centre",
}

G3_COLORSCALE = [
    [0.000000, GHSL_CLASS_COLORS[22]],
    [0.062499, GHSL_CLASS_COLORS[22]],
    [0.062500, GHSL_CLASS_COLORS[23]],
    [0.562499, GHSL_CLASS_COLORS[23]],
    [0.562500, GHSL_CLASS_COLORS[30]],
    [1.000000, GHSL_CLASS_COLORS[30]],
]


# ------------------------------------------------------------
# Coordinate-label helper
# ------------------------------------------------------------

def degree_minute_label(value, coordinate):
    absolute_value = abs(float(value))
    degrees = int(np.floor(absolute_value))
    minutes = int(round((absolute_value - degrees) * 60))

    if minutes == 60:
        degrees += 1
        minutes = 0

    if coordinate == "longitude":
        direction = "E" if value >= 0 else "W"
    else:
        direction = "N" if value >= 0 else "S"

    return f"{degrees}°{minutes:02d}′{direction}"


# ------------------------------------------------------------
# Common square map extent
# ------------------------------------------------------------

map_bounds = square_bounds(
    selected_display.total_bounds,
    pad_fraction=0.06,
)

x_tick_values = np.arange(
    np.ceil(map_bounds[0] * 2) / 2,
    np.floor(map_bounds[2] * 2) / 2 + 0.001,
    0.5,
)

y_tick_values = np.arange(
    np.ceil(map_bounds[1] * 2) / 2,
    np.floor(map_bounds[3] * 2) / 2 + 0.001,
    0.5,
)

x_tick_labels = [
    degree_minute_label(value, "longitude")
    for value in x_tick_values
]

y_tick_labels = [
    degree_minute_label(value, "latitude")
    for value in y_tick_values
]


# ------------------------------------------------------------
# GHSL G3 pixels
# ------------------------------------------------------------

ghsl_g3_codes = (
    ghsl_viirs
    .where(ghsl_viirs.isin(G3_CLASSES))
    .rio.write_crs(ghsl_viirs.rio.crs)
)

ghsl_g3_display = (
    ghsl_g3_codes
    .rio.reproject(
        "EPSG:4326",
        resampling=Resampling.nearest,
    )
    .rio.clip_box(
        minx=map_bounds[0],
        miny=map_bounds[1],
        maxx=map_bounds[2],
        maxy=map_bounds[3],
    )
)


# ------------------------------------------------------------
# G3 class shares within the displayed extent
# ------------------------------------------------------------

ghsl_g3_values = np.asarray(
    ghsl_g3_display.values,
    dtype=float,
)

ghsl_g3_values = ghsl_g3_values[
    np.isfinite(ghsl_g3_values)
]

if not ghsl_g3_values.size:
    raise ValueError(
        "No GHSL G3 pixels were found within the displayed extent."
    )

g3_class_share_pct = {
    class_value: (
        100.0
        * np.count_nonzero(
            ghsl_g3_values == class_value
        )
        / ghsl_g3_values.size
    )
    for class_value in G3_CLASSES
}


# ------------------------------------------------------------
# Tacloban inset
# ------------------------------------------------------------

tacloban_display = selected_display.loc[
    selected_display["unit_name"] == "Tacloban City"
].copy()

if tacloban_display.empty:
    raise ValueError(
        "Tacloban City was not found in selected_display."
    )

tacloban_inset_bounds = square_bounds(
    tacloban_display.total_bounds,
    pad_fraction=0.18,
)

ghsl_tacloban_inset = subset_bbox(
    ghsl_g3_display,
    tacloban_inset_bounds,
)


# ------------------------------------------------------------
# Build figure
# ------------------------------------------------------------

fig_locations_g3 = go.Figure()


# ------------------------------------------------------------
# Main GHSL map
# ------------------------------------------------------------

fig_locations_g3.add_trace(
    go.Heatmap(
        x=ghsl_g3_display["x"].values,
        y=ghsl_g3_display["y"].values,
        z=ghsl_g3_display.values,
        zmin=22,
        zmax=30,
        colorscale=G3_COLORSCALE,
        showscale=False,
        zsmooth=False,
        hoverongaps=False,
        hovertemplate=(
            "GHSL class: %{z:.0f}"
            "<extra></extra>"
        ),
    )
)


# ------------------------------------------------------------
# Roads and Haiyan path
# Separate legend from the city/municipality legend
# ------------------------------------------------------------

fig_locations_g3.add_trace(
    go.Scatter(
        x=overview_road_x,
        y=overview_road_y,
        mode="lines",
        line=dict(
            color="#D5DCE4",
            width=0.5,
        ),
        name="Roads",
        legend="legend2",
        legendgroup="map-context",
        hoverinfo="skip",
    )
)

fig_locations_g3.add_trace(
    go.Scatter(
        x=haiyan_track_x,
        y=haiyan_track_y,
        mode="lines",
        line=dict(
            color="#111111",
            width=4.0,
        ),
        name="Haiyan Path",
        legend="legend2",
        legendgroup="map-context",
        hovertemplate=(
            "Haiyan Path"
            "<extra></extra>"
        ),
    )
)


# ------------------------------------------------------------
# Main city and municipality boundaries
# ------------------------------------------------------------

for municipality_name in LOCATION_ORDER:
    unit = selected_display.loc[
        selected_display["unit_name"]
        == municipality_name
    ]

    if unit.empty:
        continue

    polygon_x, polygon_y = polygon_coordinates(
        unit.geometry.iloc[0]
    )

    fig_locations_g3.add_trace(
        go.Scatter(
            x=polygon_x,
            y=polygon_y,
            mode="lines",
            line=dict(
                color=LOCATION_COLORS[municipality_name],
                width=2.4,
            ),
            name=DISPLAY_NAMES[municipality_name],
            legendgroup=(
                f"municipality-{municipality_name}"
            ),
            hovertemplate=(
                f"{DISPLAY_NAMES[municipality_name]}"
                "<extra></extra>"
            ),
        )
    )


# ------------------------------------------------------------
# Tacloban inset GHSL raster
# ------------------------------------------------------------

fig_locations_g3.add_trace(
    go.Heatmap(
        x=ghsl_tacloban_inset["x"].values,
        y=ghsl_tacloban_inset["y"].values,
        z=ghsl_tacloban_inset.values,
        zmin=22,
        zmax=30,
        colorscale=G3_COLORSCALE,
        showscale=False,
        zsmooth=False,
        hoverongaps=False,
        xaxis="x2",
        yaxis="y2",
        hovertemplate=(
            "GHSL class: %{z:.0f}"
            "<extra></extra>"
        ),
    )
)


# ------------------------------------------------------------
# Inset roads
# ------------------------------------------------------------

fig_locations_g3.add_trace(
    go.Scatter(
        x=overview_road_x,
        y=overview_road_y,
        mode="lines",
        line=dict(
            color="#D5DCE4",
            width=0.5,
        ),
        showlegend=False,
        hoverinfo="skip",
        xaxis="x2",
        yaxis="y2",
    )
)


# ------------------------------------------------------------
# Inset Haiyan path
# ------------------------------------------------------------

fig_locations_g3.add_trace(
    go.Scatter(
        x=haiyan_track_x,
        y=haiyan_track_y,
        mode="lines",
        line=dict(
            color="#111111",
            width=2.4,
        ),
        showlegend=False,
        hoverinfo="skip",
        xaxis="x2",
        yaxis="y2",
    )
)


# ------------------------------------------------------------
# Inset city and municipality boundaries
# Drawn after the raster so they remain visible
# ------------------------------------------------------------

for municipality_name in LOCATION_ORDER:
    unit = selected_display.loc[
        selected_display["unit_name"]
        == municipality_name
    ]

    if unit.empty:
        continue

    polygon_x, polygon_y = polygon_coordinates(
        unit.geometry.iloc[0]
    )

    fig_locations_g3.add_trace(
        go.Scatter(
            x=polygon_x,
            y=polygon_y,
            mode="lines",
            line=dict(
                color=LOCATION_COLORS[municipality_name],
                width=2.4,
            ),
            showlegend=False,
            hoverinfo="skip",
            xaxis="x2",
            yaxis="y2",
        )
    )


# ------------------------------------------------------------
# Subtle proportional G3 pixel-share bar
# ------------------------------------------------------------

# Place the share bar near its lower-right corner.
share_bar_x0 = 0.475
share_bar_x1 = 0.505   # Wider difference makes the bar thicker
share_bar_y0 = 0.15
share_bar_y1 = 0.31
share_bar_height = share_bar_y1 - share_bar_y0

share_shapes = []

share_annotations = [
    dict(
        x=share_bar_x0,
        y=share_bar_y1 + 0.025,
        xref="paper",
        yref="paper",
        text="GHSL: G3 Pixel Share",
        showarrow=False,
        xanchor="left",
        yanchor="bottom",
        font=dict(
            size=18,
        ),
    )
]

cumulative_share = 0.0

for class_value in G3_CLASSES:
    class_fraction = (
        g3_class_share_pct[class_value] / 100.0
    )

    segment_y0 = (
        share_bar_y0
        + cumulative_share * share_bar_height
    )

    segment_y1 = (
        segment_y0
        + class_fraction * share_bar_height
    )

    segment_midpoint = (
        segment_y0 + segment_y1
    ) / 2.0

    share_shapes.append(
        dict(
            type="rect",
            xref="paper",
            yref="paper",
            x0=share_bar_x0,
            x1=share_bar_x1,
            y0=segment_y0,
            y1=segment_y1,
            line=dict(width=0),
            fillcolor=GHSL_CLASS_COLORS[class_value],
            layer="above",
        )
    )

    share_annotations.append(
        dict(
            x=share_bar_x1 + 0.010,
            y=segment_midpoint,
            xref="paper",
            yref="paper",
            text=(
                f"{G3_CLASS_LABELS[class_value]}<br>"
                f"{g3_class_share_pct[class_value]:.1f}%"
            ),
            showarrow=False,
            xanchor="left",
            yanchor="middle",
            align="left",
            font=dict(
                size=16,
            ),
        )
    )

    cumulative_share += class_fraction


# Border around the proportional pixel-share bar.
share_shapes.append(
    dict(
        type="rect",
        xref="paper",
        yref="paper",
        x0=share_bar_x0,
        x1=share_bar_x1,
        y0=share_bar_y0,
        y1=share_bar_y1,
        line=dict(
            color="rgba(36,59,90,0.35)",
            width=0.8,
        ),
        fillcolor="rgba(0,0,0,0)",
        layer="above",
    )
)


# ------------------------------------------------------------
# Figure layout
# ------------------------------------------------------------

fig_locations_g3.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",

    # Square figure canvas.
    width=1000,
    height=1000,

    shapes=share_shapes,

    # Main map longitude axis.
    xaxis=dict(
        domain=[0.00, 0.70],
        title=None,
        range=[
            map_bounds[0],
            map_bounds[2],
        ],
        tickmode="array",
        tickvals=x_tick_values,
        ticktext=x_tick_labels,
        ticks="outside",
        ticklen=5,
        showgrid=True,
        gridcolor="rgba(36,59,90,0.11)",
        gridwidth=1,
        zeroline=False,
        constrain="domain",
    ),

    # Main map latitude axis.
    yaxis=dict(
        domain=[0.00, 1.00],
        title=None,
        range=[
            map_bounds[1],
            map_bounds[3],
        ],
        tickmode="array",
        tickvals=y_tick_values,
        ticktext=y_tick_labels,
        ticks="outside",
        ticklen=5,
        showgrid=True,
        gridcolor="rgba(36,59,90,0.11)",
        gridwidth=1,
        zeroline=False,

        # Preserve a square geographic aspect.
        scaleanchor="x",
        scaleratio=1,
        constrain="domain",
    ),

    # Tacloban inset longitude axis.
    xaxis2=dict(
        domain=[0.47, 0.68],
        anchor="y2",
        range=[
            tacloban_inset_bounds[0],
            tacloban_inset_bounds[2],
        ],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        mirror=True,
        linecolor="#243B5A",
        linewidth=2,
        constrain="domain",
    ),

    # Tacloban inset latitude axis.
    yaxis2=dict(
        domain=[0.68, 0.94],
        anchor="x2",
        range=[
            tacloban_inset_bounds[1],
            tacloban_inset_bounds[3],
        ],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        mirror=True,
        linecolor="#243B5A",
        linewidth=2,
        scaleanchor="x2",
        scaleratio=1,
        constrain="domain",
    ),

    # Compact two-column × three-row legend
    # positioned to the right of the Tacloban inset.
    legend=dict(
        title=dict(
            text="<b>City / Municipality</b>",
            side="top",
            font=dict(size=18),
        ),
        orientation="h",
        x=0,
        y=0.1,
        xanchor="left",
        yanchor="top",
        entrywidth=80,
        entrywidthmode="pixels",
        itemwidth=30,
        traceorder="normal",
        itemsizing="constant",
        tracegroupgap=2,
        font=dict(size=16),
        bgcolor="rgba(0,0,0,0.05)",
        bordercolor="rgba(36,59,90,0.20)",
        borderwidth=1,
    ),

    # Separate road and Haiyan-path legend.
    legend2=dict(
        orientation="h",
        x=0.53,
        y=0.62,
        xanchor="left",
        yanchor="bottom",
        font=dict(size=18),
        bordercolor="rgba(36,59,90,0.20)",
        borderwidth=1,
        bgcolor="rgba(0,0,0,0.05)",
    ),

    font=dict(
        family="Arial",
        size=14,
        color="#243B5A",
    ),

    margin=dict(
        l=90,
        r=0,
        t=0,
        b=10,
    ),

    annotations=[
        # Tacloban title centred above the inset.
        dict(
            x=0.5,
            y=1.03,
            xref="x2 domain",
            yref="y2 domain",
            text="<b>Tacloban</b>",
            showarrow=False,
            xanchor="center",
            yanchor="bottom",
            font=dict(
                size=24,
                color="#243B5A",
            ),
            bgcolor="rgba(255,255,255,0.85)",
        ),
        *share_annotations,
    ],
)

fig_locations_g3.show()

In [ ]:
# ============================================================
# FIGURE 5. G3-MASKED MEDIAN 60-DAY BASELINE NTL
# ============================================================

G3_CLASSES = (22, 23, 30)

NTL_DISPLAY_MIN = 0
NTL_DISPLAY_MAX = 5
NTL_COLORBAR_TICKS = [0, 1, 2, 3, 4, 5]


# ------------------------------------------------------------
# Coordinate labels
# ------------------------------------------------------------

def degree_minute_label(value, coordinate):
    absolute_value = abs(float(value))
    degrees = int(np.floor(absolute_value))
    minutes = int(round((absolute_value - degrees) * 60))

    if minutes == 60:
        degrees += 1
        minutes = 0

    if coordinate == "longitude":
        direction = "E" if value >= 0 else "W"
    else:
        direction = "N" if value >= 0 else "S"

    return f"{degrees}°{minutes:02d}′{direction}"


# ------------------------------------------------------------
# Display layers
# ------------------------------------------------------------

selected_display = (
    selected_municipalities
    .to_crs("EPSG:4326")
    .copy()
)

municipal_land_display = (
    municipalities
    .to_crs("EPSG:4326")
    .copy()
)

haiyan_track_display = (
    haiyan_track
    .to_crs("EPSG:4326")
    .copy()
)


# ------------------------------------------------------------
# Common square map extent
# ------------------------------------------------------------

map_bounds = square_bounds(
    selected_display.total_bounds,
    pad_fraction=0.06,
)

x_tick_values = np.arange(
    np.ceil(map_bounds[0] * 2) / 2,
    np.floor(map_bounds[2] * 2) / 2 + 0.001,
    0.5,
)

y_tick_values = np.arange(
    np.ceil(map_bounds[1] * 2) / 2,
    np.floor(map_bounds[3] * 2) / 2 + 0.001,
    0.5,
)

x_tick_labels = [
    degree_minute_label(value, "longitude")
    for value in x_tick_values
]

y_tick_labels = [
    degree_minute_label(value, "latitude")
    for value in y_tick_values
]


# ------------------------------------------------------------
# Clip land polygons to the display extent
# ------------------------------------------------------------

municipal_land_display = municipal_land_display.cx[
    map_bounds[0]:map_bounds[2],
    map_bounds[1]:map_bounds[3],
].copy()

municipal_land_display = municipal_land_display.loc[
    municipal_land_display.geometry.notna()
    & ~municipal_land_display.geometry.is_empty
].copy()


# ------------------------------------------------------------
# Haiyan path
# ------------------------------------------------------------

haiyan_track_x, haiyan_track_y = (
    extract_line_coordinates(
        haiyan_track_display
    )
)


# ------------------------------------------------------------
# G3-masked reliability-qualified baseline NTL
#
# Use the existing regional baseline and apply only the
# G3 settlement mask.
# ------------------------------------------------------------

g3_baseline_map = (
    regional_ntl0_four_day
    .where(
        ghsl_viirs
        .isin(G3_CLASSES)
        .fillna(False)
    )
)

g3_baseline_display = (
    g3_baseline_map
    .rio.write_crs(a2.rio.crs)
    .rio.reproject(
        "EPSG:4326",
        resampling=Resampling.bilinear,
    )
    .rio.clip_box(
        minx=map_bounds[0],
        miny=map_bounds[1],
        maxx=map_bounds[2],
        maxy=map_bounds[3],
    )
)


# ------------------------------------------------------------
# Full map-grid template for the gray land background
#
# This uses the full GHSL grid rather than the smaller NTL
# raster grid, preventing the gray background from being
# trimmed to the NTL extent.
# ------------------------------------------------------------

land_template = (
    ghsl_viirs
    .rio.write_crs(ghsl_viirs.rio.crs)
    .rio.reproject(
        "EPSG:4326",
        resampling=Resampling.nearest,
    )
    .rio.clip_box(
        minx=map_bounds[0],
        miny=map_bounds[1],
        maxx=map_bounds[2],
        maxy=map_bounds[3],
    )
)

land_shape = (
    land_template.sizes["y"],
    land_template.sizes["x"],
)

land_transform = (
    land_template
    .rio
    .transform(recalc=True)
)

land_shapes = [
    (geometry, 1)
    for geometry in municipal_land_display.geometry
    if geometry is not None
    and not geometry.is_empty
]

if not land_shapes:
    raise ValueError(
        "No municipality land polygons intersect the map extent."
    )

land_mask_values = rasterize(
    land_shapes,
    out_shape=land_shape,
    transform=land_transform,
    fill=0,
    all_touched=True,
    dtype="uint8",
)

land_context_values = np.where(
    land_mask_values == 1,
    1.0,
    np.nan,
)

land_context_display = xr.DataArray(
    land_context_values,
    coords={
        "y": land_template["y"],
        "x": land_template["x"],
    },
    dims=("y", "x"),
    name="continuous_land_context",
)


# ------------------------------------------------------------
# Tacloban inset
# ------------------------------------------------------------

tacloban_display = selected_display.loc[
    selected_display["unit_name"] == "Tacloban City"
].copy()

if tacloban_display.empty:
    raise ValueError(
        "Tacloban City was not found in selected_display."
    )

tacloban_inset_bounds = square_bounds(
    tacloban_display.total_bounds,
    pad_fraction=0.22,
)

land_tacloban_inset = subset_bbox(
    land_context_display,
    tacloban_inset_bounds,
)

baseline_tacloban_inset = subset_bbox(
    g3_baseline_display,
    tacloban_inset_bounds,
)


# ------------------------------------------------------------
# POI label offsets
# ------------------------------------------------------------

POI_LABEL_OFFSETS = {
    "Tacloban City": (55, -42),
    "Palo": (55, 34),
    "Guiuan": (58, -12),
    "Alangalang": (-70, -30),
    "Ormoc City": (-65, -30),
    "Baybay City": (-65, -2),
}


# ------------------------------------------------------------
# Build figure
# ------------------------------------------------------------

fig_locations_g3_ntl = go.Figure()
ntl_annotations = []


# ============================================================
# MAIN MAP
# ============================================================


# ------------------------------------------------------------
# Continuous gray land background
#
# NaN cells remain white and represent water.
# ------------------------------------------------------------

fig_locations_g3_ntl.add_trace(
    go.Heatmap(
        x=land_context_display["x"].values,
        y=land_context_display["y"].values,
        z=land_context_display.values,
        zmin=0,
        zmax=1,
        colorscale=[
            [0.0, "#D9DEE5"],
            [1.0, "#D9DEE5"],
        ],
        showscale=False,
        zsmooth=False,
        hoverongaps=False,
        hoverinfo="skip",
    )
)


# ------------------------------------------------------------
# G3-masked baseline NTL
# ------------------------------------------------------------

fig_locations_g3_ntl.add_trace(
    go.Heatmap(
        x=g3_baseline_display["x"].values,
        y=g3_baseline_display["y"].values,
        z=g3_baseline_display.values,
        colorscale="Inferno",
        zmin=NTL_DISPLAY_MIN,
        zmax=NTL_DISPLAY_MAX,
        zauto=False,
        zsmooth=False,
        hoverongaps=False,
        colorbar=dict(
            orientation="v",
            title=dict(
                text=(
                    "Median baseline NTL<br>"
                    "(nW cm⁻² sr⁻¹)"
                ),
                side="top",
                font=dict(size=16),
            ),
            tickmode="array",
            tickvals=NTL_COLORBAR_TICKS,
            ticktext=[
                str(value)
                for value in NTL_COLORBAR_TICKS
            ],
            tickfont=dict(size=16),

            # Same lower-right position as the earlier GHSL scale.
            x=0.49,
            y=0.23,
            xanchor="left",
            yanchor="middle",
            len=0.18,
            thickness=30,

            outlinecolor="rgba(36,59,90,0.40)",
            outlinewidth=1,
        ),
        hovertemplate=(
            "Median G3 DNB-BRDF: "
            "%{z:.2f} nW cm⁻² sr⁻¹"
            "<extra></extra>"
        ),
    )
)


# ------------------------------------------------------------
# Haiyan path
# ------------------------------------------------------------

fig_locations_g3_ntl.add_trace(
    go.Scatter(
        x=haiyan_track_x,
        y=haiyan_track_y,
        mode="lines",
        line=dict(
            color="#111111",
            width=4.0,
        ),
        name="Haiyan Path",
        legend="legend2",
        legendgroup="map-context",
        hovertemplate=(
            "Haiyan Path"
            "<extra></extra>"
        ),
    )
)


# ------------------------------------------------------------
# City and municipality boundaries
# ------------------------------------------------------------

for municipality_name in LOCATION_ORDER:
    unit = selected_display.loc[
        selected_display["unit_name"]
        == municipality_name
    ]

    if len(unit) != 1:
        raise ValueError(
            f"Expected one boundary for {municipality_name}; "
            f"found {len(unit)}."
        )

    color = LOCATION_COLORS[municipality_name]

    polygon_x, polygon_y = polygon_coordinates(
        unit.geometry.iloc[0]
    )

    fig_locations_g3_ntl.add_trace(
        go.Scatter(
            x=polygon_x,
            y=polygon_y,
            mode="lines",
            line=dict(
                color=color,
                width=2.4,
            ),
            name=DISPLAY_NAMES[municipality_name],
            legendgroup=(
                f"municipality-{municipality_name}"
            ),
            hovertemplate=(
                f"{DISPLAY_NAMES[municipality_name]}"
                "<extra></extra>"
            ),
        )
    )


# ------------------------------------------------------------
# Characteristic POIs
# ------------------------------------------------------------

for poi in poi_table.itertuples():
    municipality_name = poi.municipality_name
    color = poi.color

    label_ax, label_ay = POI_LABEL_OFFSETS.get(
        municipality_name,
        (50, -20),
    )

    poi_label = (
        f"<b>{DISPLAY_NAMES[municipality_name]} "
        f"({POI_KERNEL_SIZE}×{POI_KERNEL_SIZE})</b>"
    )

    # Transparent POI square.
    fig_locations_g3_ntl.add_trace(
        go.Scatter(
            x=[poi.longitude],
            y=[poi.latitude],
            mode="markers",
            marker=dict(
                size=15,
                color="rgba(255,255,255,0)",
                symbol="square",
                line=dict(
                    color=color,
                    width=3.0,
                ),
            ),
            showlegend=False,
            hovertemplate=(
                f"{poi.poi_name}<br>"
                f"{POI_KERNEL_SIZE}×{POI_KERNEL_SIZE} kernel"
                "<extra></extra>"
            ),
        )
    )

    # Subtle shadow.
    ntl_annotations.append(
        dict(
            x=poi.longitude,
            y=poi.latitude,
            xref="x",
            yref="y",
            text=poi_label,
            showarrow=True,
            ax=label_ax + 4,
            ay=label_ay + 4,
            axref="pixel",
            ayref="pixel",
            arrowhead=0,
            arrowwidth=0.1,
            arrowcolor="rgba(0,0,0,0)",
            standoff=9,
            xanchor=(
                "left"
                if label_ax > 0
                else "right"
            ),
            yanchor="middle",
            align=(
                "left"
                if label_ax > 0
                else "right"
            ),
            font=dict(
                size=16,
                color="rgba(0,0,0,0)",
            ),
            bgcolor="rgba(36,59,90,0.18)",
            bordercolor="rgba(0,0,0,0)",
            borderwidth=0,
            borderpad=3,
        )
    )

    # Visible POI label.
    ntl_annotations.append(
        dict(
            x=poi.longitude,
            y=poi.latitude,
            xref="x",
            yref="y",
            text=poi_label,
            showarrow=True,
            ax=label_ax,
            ay=label_ay,
            axref="pixel",
            ayref="pixel",
            arrowhead=0,
            arrowsize=1,
            arrowwidth=1.4,
            arrowcolor=color,
            standoff=9,
            xanchor=(
                "left"
                if label_ax > 0
                else "right"
            ),
            yanchor="middle",
            align=(
                "left"
                if label_ax > 0
                else "right"
            ),
            font=dict(
                size=16,
                color=color,
            ),
            bgcolor="rgba(255,255,255,0.94)",
            bordercolor=hex_to_rgba(
                color,
                0.55,
            ),
            borderwidth=1,
            borderpad=3,
        )
    )


# ============================================================
# TACLOBAN INSET
# ============================================================


# ------------------------------------------------------------
# Inset continuous gray land
# ------------------------------------------------------------

fig_locations_g3_ntl.add_trace(
    go.Heatmap(
        x=land_tacloban_inset["x"].values,
        y=land_tacloban_inset["y"].values,
        z=land_tacloban_inset.values,
        zmin=0,
        zmax=1,
        colorscale=[
            [0.0, "#D9DEE5"],
            [1.0, "#D9DEE5"],
        ],
        showscale=False,
        zsmooth=False,
        hoverongaps=False,
        hoverinfo="skip",
        xaxis="x2",
        yaxis="y2",
    )
)


# ------------------------------------------------------------
# Inset G3-masked NTL
# ------------------------------------------------------------

fig_locations_g3_ntl.add_trace(
    go.Heatmap(
        x=baseline_tacloban_inset["x"].values,
        y=baseline_tacloban_inset["y"].values,
        z=baseline_tacloban_inset.values,
        colorscale="Inferno",
        zmin=NTL_DISPLAY_MIN,
        zmax=NTL_DISPLAY_MAX,
        zauto=False,
        showscale=False,
        zsmooth=False,
        hoverongaps=False,
        xaxis="x2",
        yaxis="y2",
        hovertemplate=(
            "Median G3 DNB-BRDF: "
            "%{z:.2f} nW cm⁻² sr⁻¹"
            "<extra></extra>"
        ),
    )
)


# ------------------------------------------------------------
# Inset Haiyan path
# ------------------------------------------------------------

fig_locations_g3_ntl.add_trace(
    go.Scatter(
        x=haiyan_track_x,
        y=haiyan_track_y,
        mode="lines",
        line=dict(
            color="#111111",
            width=2.4,
        ),
        showlegend=False,
        hoverinfo="skip",
        xaxis="x2",
        yaxis="y2",
    )
)


# ------------------------------------------------------------
# Inset municipality boundaries
# ------------------------------------------------------------

for municipality_name in LOCATION_ORDER:
    unit = selected_display.loc[
        selected_display["unit_name"]
        == municipality_name
    ]

    if unit.empty:
        continue

    color = LOCATION_COLORS[municipality_name]

    polygon_x, polygon_y = polygon_coordinates(
        unit.geometry.iloc[0]
    )

    fig_locations_g3_ntl.add_trace(
        go.Scatter(
            x=polygon_x,
            y=polygon_y,
            mode="lines",
            line=dict(
                color=color,
                width=2.2,
            ),
            showlegend=False,
            hoverinfo="skip",
            xaxis="x2",
            yaxis="y2",
        )
    )


# ------------------------------------------------------------
# Inset POIs
# ------------------------------------------------------------

for poi in poi_table.itertuples():
    fig_locations_g3_ntl.add_trace(
        go.Scatter(
            x=[poi.longitude],
            y=[poi.latitude],
            mode="markers",
            marker=dict(
                size=13,
                color="rgba(255,255,255,0)",
                symbol="square",
                line=dict(
                    color=poi.color,
                    width=2.6,
                ),
            ),
            showlegend=False,
            hovertemplate=(
                f"{poi.poi_name}<br>"
                f"{POI_KERNEL_SIZE}×{POI_KERNEL_SIZE} kernel"
                "<extra></extra>"
            ),
            xaxis="x2",
            yaxis="y2",
        )
    )


# ------------------------------------------------------------
# Tacloban inset title
# ------------------------------------------------------------

ntl_annotations.append(
    dict(
        x=0.5,
        y=1.03,
        xref="x2 domain",
        yref="y2 domain",
        text="<b>Tacloban</b>",
        showarrow=False,
        xanchor="center",
        yanchor="bottom",
        font=dict(
            size=24,
            color="#243B5A",
        ),
        bgcolor="rgba(255,255,255,0.85)",
    )
)


# ============================================================
# FIGURE LAYOUT
# ============================================================

fig_locations_g3_ntl.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",

    # White background represents water.
    plot_bgcolor="white",

    # Same square canvas as the previous spatial maps.
    width=1000,
    height=1000,

    # Main longitude axis.
    xaxis=dict(
        domain=[0.00, 0.70],
        title=None,
        range=[
            map_bounds[0],
            map_bounds[2],
        ],
        tickmode="array",
        tickvals=x_tick_values,
        ticktext=x_tick_labels,
        ticks="outside",
        ticklen=5,
        showgrid=True,
        gridcolor="rgba(36,59,90,0.11)",
        gridwidth=1,
        zeroline=False,
        constrain="domain",
    ),

    # Main latitude axis.
    yaxis=dict(
        domain=[0.00, 1.00],
        title=None,
        range=[
            map_bounds[1],
            map_bounds[3],
        ],
        tickmode="array",
        tickvals=y_tick_values,
        ticktext=y_tick_labels,
        ticks="outside",
        ticklen=5,
        showgrid=True,
        gridcolor="rgba(36,59,90,0.11)",
        gridwidth=1,
        zeroline=False,
        scaleanchor="x",
        scaleratio=1,
        constrain="domain",
    ),

    # Same Tacloban inset position as the previous maps.
    xaxis2=dict(
        domain=[0.47, 0.68],
        anchor="y2",
        range=[
            tacloban_inset_bounds[0],
            tacloban_inset_bounds[2],
        ],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        mirror=True,
        linecolor="#243B5A",
        linewidth=2,
        constrain="domain",
    ),

    yaxis2=dict(
        domain=[0.68, 0.94],
        anchor="x2",
        range=[
            tacloban_inset_bounds[1],
            tacloban_inset_bounds[3],
        ],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        mirror=True,
        linecolor="#243B5A",
        linewidth=2,
        scaleanchor="x2",
        scaleratio=1,
        constrain="domain",
    ),

    # City and municipality legend.
    legend=dict(
        title=dict(
            text="<b>City / Municipality</b>",
            side="top",
            font=dict(size=18),
        ),
        orientation="h",
        x=0.00,
        y=0.10,
        xanchor="left",
        yanchor="top",
        entrywidth=80,
        entrywidthmode="pixels",
        itemwidth=30,
        traceorder="normal",
        itemsizing="constant",
        tracegroupgap=2,
        font=dict(size=16),
        bgcolor="rgba(0,0,0,0.05)",
        bordercolor="rgba(36,59,90,0.20)",
        borderwidth=1,
    ),

    # Haiyan-path legend.
    legend2=dict(
        orientation="h",
        x=0.53,
        y=0.62,
        xanchor="left",
        yanchor="bottom",
        font=dict(size=18),
        bgcolor="rgba(0,0,0,0.05)",
        bordercolor="rgba(36,59,90,0.20)",
        borderwidth=1,
    ),

    font=dict(
        family="Arial",
        size=14,
        color="#243B5A",
    ),

    margin=dict(
        l=90,
        r=0,
        t=0,
        b=10,
    ),

    annotations=ntl_annotations,
)

fig_locations_g3_ntl.show()

In [ ]:
# ============================================================
# FIGURE 5. MEDIAN 60-DAY VALID-OBSERVATION NTL
# Layout locked to Figure 4
# ============================================================

# ------------------------------------------------------------
# Coordinate labels
# ------------------------------------------------------------

def degree_minute_label(value, coordinate):
    absolute_value = abs(float(value))
    degrees = int(np.floor(absolute_value))
    minutes = int(round((absolute_value - degrees) * 60))

    if minutes == 60:
        degrees += 1
        minutes = 0

    if coordinate == "longitude":
        direction = "E" if value >= 0 else "W"
    else:
        direction = "N" if value >= 0 else "S"

    return f"{degrees}°{minutes:02d}′{direction}"


# ------------------------------------------------------------
# Exact common extent used by the GHSL map
# ------------------------------------------------------------

map_bounds = square_bounds(
    selected_display.total_bounds,
    pad_fraction=0.06,
)

x_tick_values = np.arange(
    np.ceil(map_bounds[0] * 2) / 2,
    np.floor(map_bounds[2] * 2) / 2 + 0.001,
    0.5,
)

y_tick_values = np.arange(
    np.ceil(map_bounds[1] * 2) / 2,
    np.floor(map_bounds[3] * 2) / 2 + 0.001,
    0.5,
)

x_tick_labels = [
    degree_minute_label(value, "longitude")
    for value in x_tick_values
]

y_tick_labels = [
    degree_minute_label(value, "latitude")
    for value in y_tick_values
]


# ------------------------------------------------------------
# Median of all valid observations during the 60-day baseline
#
# No GHSL mask
# No study-area mask
# No spatial-completeness threshold
# No minimum-observation threshold beyond ≥1 observation
# No percentile clipping
# ------------------------------------------------------------

baseline_dnb = dnb.sel(
    date=slice(BASELINE_START, PRE_EVENT_END)
)

baseline_mqf = mqf.sel(
    date=slice(BASELINE_START, PRE_EVENT_END)
)

baseline_valid_dnb = baseline_dnb.where(
    (baseline_mqf == 0)
    & np.isfinite(baseline_dnb)
)

baseline_valid_count = baseline_valid_dnb.count("date")

median_60day_ntl = (
    baseline_valid_dnb
    .median("date", skipna=True)
    .where(baseline_valid_count >= 1)
    .compute()
)

median_60day_ntl.name = "median_60day_valid_observation_ntl"

if median_60day_ntl.rio.crs is None:
    median_60day_ntl = median_60day_ntl.rio.write_crs(
        a2.rio.crs
    )

median_60day_ntl_display = (
    median_60day_ntl
    .rio.reproject(
        "EPSG:4326",
        resampling=Resampling.nearest,
    )
    .rio.clip_box(
        minx=map_bounds[0],
        miny=map_bounds[1],
        maxx=map_bounds[2],
        maxy=map_bounds[3],
    )
)


# ------------------------------------------------------------
# Tacloban inset — exact extent used by Figure 4
# ------------------------------------------------------------

tacloban_display = selected_display.loc[
    selected_display["unit_name"] == "Tacloban City"
].copy()

if tacloban_display.empty:
    raise ValueError(
        "Tacloban City was not found in selected_display."
    )

tacloban_inset_bounds = square_bounds(
    tacloban_display.total_bounds,
    pad_fraction=0.18,
)

median_60day_tacloban = subset_bbox(
    median_60day_ntl_display,
    tacloban_inset_bounds,
)


# ------------------------------------------------------------
# POI label positions
# ------------------------------------------------------------

poi_label_offsets = {
    "Tacloban City": (55, -42),
    "Palo": (55, 34),
    "Guiuan": (58, -12),
    "Alangalang": (-70, -30),
    "Ormoc City": (-65, -30),
    "Baybay City": (-65, -2),
}


# ------------------------------------------------------------
# Build figure
# ------------------------------------------------------------

fig_median_60day_ntl = go.Figure()


# ------------------------------------------------------------
# Main median NTL raster
# ------------------------------------------------------------

fig_median_60day_ntl.add_trace(
    go.Heatmap(
        x=median_60day_ntl_display["x"].values,
        y=median_60day_ntl_display["y"].values,
        z=median_60day_ntl_display.values,
        colorscale="Inferno",
        zmin=0,
        zmax=5,
        zsmooth=False,
        hoverongaps=False,
        showscale=True,
        colorbar=dict(
            title=dict(
                text=(
                    "Median 60-day NTL"
                    "<br>(nW cm⁻² sr⁻¹)"
                ),
                side="top",
                font=dict(size=16),
            ),

            # Locked to the lower-right colorbar area
            # used by the GHSL figure.
            x=0.475,
            y=0.23,
            xanchor="left",
            yanchor="middle",
            len=0.16,
            thickness=30,

            tickvals=[0, 1, 2, 3, 4, 5],
            tickfont=dict(size=16),
            ticks="outside",
            outlinecolor="rgba(36,59,90,0.35)",
            outlinewidth=0.8,
        ),
        hovertemplate=(
            "Median DNB-BRDF: %{z:.2f} "
            "nW cm⁻² sr⁻¹"
            "<extra></extra>"
        ),
    )
)


# ------------------------------------------------------------
# Haiyan path
# ------------------------------------------------------------

fig_median_60day_ntl.add_trace(
    go.Scatter(
        x=haiyan_track_x,
        y=haiyan_track_y,
        mode="lines",
        line=dict(
            color="#111111",
            width=4.0,
        ),
        name="Haiyan Path",
        legend="legend2",
        legendgroup="map-context",
        hovertemplate=(
            "Haiyan Path"
            "<extra></extra>"
        ),
    )
)


# ------------------------------------------------------------
# Main municipality boundaries
# ------------------------------------------------------------

for municipality_name in LOCATION_ORDER:
    unit = selected_display.loc[
        selected_display["unit_name"] == municipality_name
    ]

    if unit.empty:
        continue

    polygon_x, polygon_y = polygon_coordinates(
        unit.geometry.iloc[0]
    )

    fig_median_60day_ntl.add_trace(
        go.Scatter(
            x=polygon_x,
            y=polygon_y,
            mode="lines",
            line=dict(
                color=LOCATION_COLORS[municipality_name],
                width=2.4,
            ),
            name=DISPLAY_NAMES[municipality_name],
            legendgroup=(
                f"municipality-{municipality_name}"
            ),
            hovertemplate=(
                f"{DISPLAY_NAMES[municipality_name]}"
                "<extra></extra>"
            ),
        )
    )


# ------------------------------------------------------------
# Main POI kernel markers
# ------------------------------------------------------------

for poi in poi_table.itertuples():
    municipality_name = next(
        (
            name
            for name in LOCATION_ORDER
            if DISPLAY_NAMES[name].lower()
            in str(poi.poi_name).lower()
        ),
        None,
    )

    if municipality_name is None:
        continue

    color = LOCATION_COLORS[municipality_name]
    display_name = DISPLAY_NAMES[municipality_name]

    fig_median_60day_ntl.add_trace(
        go.Scatter(
            x=[poi.longitude],
            y=[poi.latitude],
            mode="markers",
            marker=dict(
                size=16,
                symbol="square",
                color="rgba(255,255,255,0.18)",
                line=dict(
                    color=color,
                    width=3,
                ),
            ),
            showlegend=False,
            hovertemplate=(
                f"{display_name} 5×5 POI kernel"
                "<extra></extra>"
            ),
        )
    )


# ------------------------------------------------------------
# Tacloban inset raster
# ------------------------------------------------------------

fig_median_60day_ntl.add_trace(
    go.Heatmap(
        x=median_60day_tacloban["x"].values,
        y=median_60day_tacloban["y"].values,
        z=median_60day_tacloban.values,
        colorscale="Inferno",
        zmin=0,
        zmax=5,
        showscale=False,
        zsmooth=False,
        hoverongaps=False,
        xaxis="x2",
        yaxis="y2",
        hovertemplate=(
            "Median DNB-BRDF: %{z:.2f} "
            "nW cm⁻² sr⁻¹"
            "<extra></extra>"
        ),
    )
)


# ------------------------------------------------------------
# Inset Haiyan path
# ------------------------------------------------------------

fig_median_60day_ntl.add_trace(
    go.Scatter(
        x=haiyan_track_x,
        y=haiyan_track_y,
        mode="lines",
        line=dict(
            color="#111111",
            width=2.4,
        ),
        showlegend=False,
        hoverinfo="skip",
        xaxis="x2",
        yaxis="y2",
    )
)


# ------------------------------------------------------------
# Inset municipality boundaries
# ------------------------------------------------------------

for municipality_name in LOCATION_ORDER:
    unit = selected_display.loc[
        selected_display["unit_name"] == municipality_name
    ]

    if unit.empty:
        continue

    polygon_x, polygon_y = polygon_coordinates(
        unit.geometry.iloc[0]
    )

    fig_median_60day_ntl.add_trace(
        go.Scatter(
            x=polygon_x,
            y=polygon_y,
            mode="lines",
            line=dict(
                color=LOCATION_COLORS[municipality_name],
                width=2.4,
            ),
            showlegend=False,
            hoverinfo="skip",
            xaxis="x2",
            yaxis="y2",
        )
    )


# ------------------------------------------------------------
# Inset POI markers
# ------------------------------------------------------------

for poi in poi_table.itertuples():
    municipality_name = next(
        (
            name
            for name in LOCATION_ORDER
            if DISPLAY_NAMES[name].lower()
            in str(poi.poi_name).lower()
        ),
        None,
    )

    if municipality_name is None:
        continue

    fig_median_60day_ntl.add_trace(
        go.Scatter(
            x=[poi.longitude],
            y=[poi.latitude],
            mode="markers",
            marker=dict(
                size=13,
                symbol="square",
                color="rgba(255,255,255,0.18)",
                line=dict(
                    color=LOCATION_COLORS[municipality_name],
                    width=2.5,
                ),
            ),
            showlegend=False,
            hoverinfo="skip",
            xaxis="x2",
            yaxis="y2",
        )
    )


# ------------------------------------------------------------
# POI callout labels
# ------------------------------------------------------------

poi_annotations = []

for poi in poi_table.itertuples():
    municipality_name = next(
        (
            name
            for name in LOCATION_ORDER
            if DISPLAY_NAMES[name].lower()
            in str(poi.poi_name).lower()
        ),
        None,
    )

    if municipality_name is None:
        continue

    color = LOCATION_COLORS[municipality_name]
    display_name = DISPLAY_NAMES[municipality_name]
    offset_x, offset_y = poi_label_offsets[municipality_name]
    label = f"<b>{display_name} (5×5)</b>"

    # Subtle shadow.
    poi_annotations.append(
        dict(
            x=poi.longitude,
            y=poi.latitude,
            xref="x",
            yref="y",
            text=label,
            showarrow=True,
            ax=offset_x + 3,
            ay=offset_y + 3,
            arrowcolor="rgba(0,0,0,0)",
            # arrowwidth=0,
            bgcolor="rgba(0,0,0,0.14)",
            bordercolor="rgba(0,0,0,0)",
            borderpad=3,
            font=dict(
                family="Arial",
                size=16,
                color="rgba(0,0,0,0)",
            ),
        )
    )

    # Visible label.
    poi_annotations.append(
        dict(
            x=poi.longitude,
            y=poi.latitude,
            xref="x",
            yref="y",
            text=label,
            showarrow=True,
            ax=offset_x,
            ay=offset_y,
            arrowhead=0,
            arrowsize=1,
            # arrowwidth=1.4,
            arrowcolor=color,
            bgcolor="rgba(255,255,255,0.92)",
            bordercolor=hex_to_rgba(color, 0.42),
            borderwidth=1,
            borderpad=3,
            font=dict(
                family="Arial",
                size=16,
                color=color,
            ),
        )
    )


# ------------------------------------------------------------
# Layout — copied from Figure 4
# ------------------------------------------------------------

fig_median_60day_ntl.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",

    width=1000,
    height=1000,

    # Exact main-map placement.
    xaxis=dict(
        domain=[0.00, 0.70],
        title=None,
        range=[
            map_bounds[0],
            map_bounds[2],
        ],
        tickmode="array",
        tickvals=x_tick_values,
        ticktext=x_tick_labels,
        ticks="outside",
        ticklen=5,
        showgrid=True,
        gridcolor="rgba(36,59,90,0.11)",
        gridwidth=1,
        zeroline=False,
        constrain="domain",
    ),

    yaxis=dict(
        domain=[0.00, 1.00],
        title=None,
        range=[
            map_bounds[1],
            map_bounds[3],
        ],
        tickmode="array",
        tickvals=y_tick_values,
        ticktext=y_tick_labels,
        ticks="outside",
        ticklen=5,
        showgrid=True,
        gridcolor="rgba(36,59,90,0.11)",
        gridwidth=1,
        zeroline=False,
        scaleanchor="x",
        scaleratio=1,
        constrain="domain",
    ),

    # Exact Tacloban-inset placement.
    xaxis2=dict(
        domain=[0.47, 0.68],
        anchor="y2",
        range=[
            tacloban_inset_bounds[0],
            tacloban_inset_bounds[2],
        ],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        mirror=True,
        linecolor="#243B5A",
        linewidth=2,
        constrain="domain",
    ),

    yaxis2=dict(
        domain=[0.68, 0.94],
        anchor="x2",
        range=[
            tacloban_inset_bounds[1],
            tacloban_inset_bounds[3],
        ],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        mirror=True,
        linecolor="#243B5A",
        linewidth=2,
        scaleanchor="x2",
        scaleratio=1,
        constrain="domain",
    ),

    # Exact municipality-legend placement.
    legend=dict(
        title=dict(
            text="<b>City / Municipality</b>",
            side="top",
            font=dict(size=18),
        ),
        orientation="h",
        x=0,
        y=0.1,
        xanchor="left",
        yanchor="top",
        entrywidth=80,
        entrywidthmode="pixels",
        itemwidth=30,
        traceorder="normal",
        itemsizing="constant",
        tracegroupgap=2,
        font=dict(size=16),
        bgcolor="rgba(0,0,0,0.05)",
        bordercolor="rgba(36,59,90,0.20)",
        borderwidth=1,
    ),

    # Exact Haiyan-path legend placement.
    legend2=dict(
        orientation="h",
        x=0.53,
        y=0.62,
        xanchor="left",
        yanchor="bottom",
        font=dict(size=18),
        bordercolor="rgba(36,59,90,0.20)",
        borderwidth=1,
        bgcolor="rgba(0,0,0,0.05)",
    ),

    font=dict(
        family="Arial",
        size=14,
        color="#243B5A",
    ),

    # Exact Figure 4 margins.
    margin=dict(
        l=90,
        r=0,
        t=0,
        b=10,
    ),

    annotations=[
        dict(
            x=0.5,
            y=1.03,
            xref="x2 domain",
            yref="y2 domain",
            text="<b>Tacloban</b>",
            showarrow=False,
            xanchor="center",
            yanchor="bottom",
            font=dict(
                size=24,
                color="#243B5A",
            ),
            bgcolor="rgba(255,255,255,0.85)",
        ),
        *poi_annotations,
    ],
)

fig_median_60day_ntl.show()

## 4. Common-extent spatial recovery

Roads are removed from these panels. Municipality boundaries remain thin, the Haiyan path is solid black, and every stage includes a compact Tacloban inset in the upper-right corner.

In [ ]:
# ============================================================
# FIGURE 6. REGIONAL CONTEXT AND TACLOBAN ZOOM — 2 × 5
# ============================================================

# The analytical profiles remain restricted to the six-location study mask.
# This separate display cube removes only that municipal restriction so the
# regional map retains all directly observed G3 nighttime-light pixels.
regional_display_unclipped = dnb.where(
    (mqf == 0) & ghsl_mask
)

# Retain the same daily high-value clipping rule used by the principal RQ cube.
regional_display_cube = xr.where(
    regional_display_unclipped > rq_daily_p95,
    rq_daily_p95,
    regional_display_unclipped,
)

regional_display_selected = regional_display_cube.sel(
    date=slice(ANALYSIS_START, PROFILE_END)
)

regional_display_dates = pd.DatetimeIndex(
    regional_display_selected["date"].values
).normalize()

regional_display_blocks = np.floor_divide(
    (regional_display_dates - EVENT_DATE).days,
    4,
).astype(int)

regional_display_composites = (
    regional_display_selected
    .assign_coords(block=("date", regional_display_blocks))
    .groupby("block")
    .median(dim="date", skipna=True)
)

display_block_values = regional_display_composites["block"].values.astype(int)
display_block_start = EVENT_DATE + pd.to_timedelta(
    display_block_values * 4,
    unit="D",
)
display_block_end = display_block_start + pd.Timedelta(days=3)

display_baseline_blocks = display_block_values[
    (display_block_start >= BASELINE_START)
    & (display_block_end <= PRE_EVENT_END)
]

display_baseline_composites = regional_display_composites.sel(
    block=display_baseline_blocks
)

display_baseline_count = display_baseline_composites.notnull().sum("block")
display_ntl0 = display_baseline_composites.median(
    "block",
    skipna=True,
)

# Require the established four-day baseline support, but do not impose the
# six-municipality study mask on the regional display.
display_fixed_mask = (
    ghsl_mask
    & (display_baseline_count >= MIN_BASELINE_OBSERVATIONS[4])
    & np.isfinite(display_ntl0)
    & (display_ntl0 > 0)
).compute()

display_ntl0 = display_ntl0.where(display_fixed_mask).compute()
regional_display_composites = regional_display_composites.where(
    display_fixed_mask
)


def regional_display_stage_map(stage_name, stage_start, stage_end):
    if stage_name == "Baseline":
        return display_ntl0

    # Use the same temporally admissible blocks as the analytical regional
    # profile, but display their full regional G3 spatial support.
    eligible = regional_four_day.loc[
        (regional_four_day["date_start"] >= stage_start)
        & (regional_four_day["date_end"] <= stage_end)
        & regional_four_day["recovery_pct"].notna()
    ]

    available_blocks = set(
        regional_display_composites["block"].values.astype(int)
    )
    eligible_blocks = [
        int(block)
        for block in eligible["block"].astype(int)
        if int(block) in available_blocks
    ]

    if not eligible_blocks:
        return xr.full_like(display_ntl0, np.nan).compute()

    return (
        regional_display_composites
        .sel(block=eligible_blocks)
        .median("block", skipna=True)
        .compute()
    )


regional_display_stage_maps = {
    stage_name: regional_display_stage_map(
        stage_name,
        stage_start,
        stage_end,
    )
    for stage_name, (stage_start, stage_end) in STAGE_WINDOWS.items()
}

regional_display_recovery_maps = {
    stage_name: xr.where(
        display_ntl0 > 0,
        100.0 * stage_map / display_ntl0,
        np.nan,
    )
    for stage_name, stage_map in regional_display_stage_maps.items()
    if stage_name != "Baseline"
}

stage_names = list(STAGE_WINDOWS.keys())

# Regional row uses the full VIIRS raster extent, matching the older map.
regional_display_bounds = dnb.rio.bounds()

# Second row is a true Tacloban zoom, not an inset.
tacloban_raster = selected_raster_crs.loc[
    selected_raster_crs["unit_name"] == "Tacloban City"
].copy()
tacloban_zoom_bounds = square_bounds(
    tacloban_raster.total_bounds,
    pad_fraction=0.18,
)

full_track_x, full_track_y = extract_line_coordinates(
    haiyan_track_raster_crs
)

tacloban_track = haiyan_track_raster_crs.cx[
    tacloban_zoom_bounds[0]:tacloban_zoom_bounds[2],
    tacloban_zoom_bounds[1]:tacloban_zoom_bounds[3],
].copy()
tacloban_track_x, tacloban_track_y = extract_line_coordinates(
    tacloban_track
)

tacloban_roads = roads_raster_crs.cx[
    tacloban_zoom_bounds[0]:tacloban_zoom_bounds[2],
    tacloban_zoom_bounds[1]:tacloban_zoom_bounds[3],
].copy()
tacloban_road_x, tacloban_road_y = extract_line_coordinates(
    tacloban_roads
)

municipality_stage_boundaries = {}
for municipality_name in LOCATION_ORDER:
    unit = selected_raster_crs.loc[
        selected_raster_crs["unit_name"] == municipality_name
    ].copy()
    unit["geometry"] = unit.geometry.boundary
    municipality_stage_boundaries[municipality_name] = (
        extract_line_coordinates(unit)
    )


fig_regional_recovery = make_subplots(
    rows=2,
    cols=5,
    shared_xaxes=False,
    shared_yaxes=False,
    column_titles=stage_names,
    row_titles=["Regional context", "Tacloban zoom"],
    row_heights=[0.58, 0.42],
    horizontal_spacing=0.012,
    vertical_spacing=0.060,
)

top_y_domain = fig_regional_recovery.layout.yaxis.domain
first_x_domain = fig_regional_recovery.layout.xaxis.domain
last_x_domain = fig_regional_recovery.layout.xaxis5.domain


def add_spatial_overlays(figure, row, column, zoomed=False):
    # Roads remain deliberately subtle, reproducing the visual texture of the
    # earlier regional map without competing with the NTL pixels.
    if zoomed:
        road_values_x, road_values_y = tacloban_road_x, tacloban_road_y
        track_values_x, track_values_y = tacloban_track_x, tacloban_track_y
        road_width = 0.65
        track_width = 3.0
    else:
        road_values_x, road_values_y = road_x, road_y
        track_values_x, track_values_y = full_track_x, full_track_y
        road_width = 0.35
        track_width = 3.2

    figure.add_trace(
        go.Scatter(
            x=road_values_x,
            y=road_values_y,
            mode="lines",
            line=dict(
                color="rgba(235,241,247,0.48)",
                width=road_width,
            ),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=row,
        col=column,
    )

    # A light halo keeps the solid-black Haiyan path visible on dark pixels.
    figure.add_trace(
        go.Scatter(
            x=track_values_x,
            y=track_values_y,
            mode="lines",
            line=dict(
                color="rgba(255,255,255,0.82)",
                width=track_width + 2.0,
            ),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=row,
        col=column,
    )
    figure.add_trace(
        go.Scatter(
            x=track_values_x,
            y=track_values_y,
            mode="lines",
            line=dict(color="#111111", width=track_width),
            showlegend=False,
            hovertemplate="Haiyan Path<extra></extra>",
        ),
        row=row,
        col=column,
    )

    # Only the six selected city/municipality boundaries are emphasized.
    for municipality_name in LOCATION_ORDER:
        boundary_values_x, boundary_values_y = (
            municipality_stage_boundaries[municipality_name]
        )
        figure.add_trace(
            go.Scatter(
                x=boundary_values_x,
                y=boundary_values_y,
                mode="lines",
                line=dict(
                    color=LOCATION_COLORS[municipality_name],
                    width=2.3 if zoomed else 1.8,
                ),
                showlegend=False,
                hovertemplate=(
                    f"{DISPLAY_NAMES[municipality_name]}"
                    "<extra></extra>"
                ),
            ),
            row=row,
            col=column,
        )


for column, stage_name in enumerate(stage_names, start=1):
    if stage_name == "Baseline":
        regional_map = regional_display_stage_maps[stage_name]
        regional_heatmap = go.Heatmap(
            x=regional_map["x"].values,
            y=regional_map["y"].values,
            z=regional_map.values,
            colorscale="Inferno",
            zmin=0,
            zmax=NTL_DISPLAY_MAX,
            showscale=True,
            zsmooth=False,
            hoverongaps=False,
            colorbar=dict(
                title=dict(
                    text="Median baseline NTL<br>(nW cm⁻² sr⁻¹)",
                    side="top",
                    font=dict(size=17),
                ),
                x=first_x_domain[0] + 0.015,
                y=top_y_domain[0] + 0.17,
                xanchor="left",
                yanchor="middle",
                len=0.22,
                thickness=25,
                tickvals=[0, 1, 2, 3, 4, 5],
                tickfont=dict(size=15),
                outlinecolor="rgba(36,59,90,0.35)",
                outlinewidth=0.8,
            ),
            hovertemplate=(
                "Median baseline: %{z:.2f} nW cm⁻² sr⁻¹"
                "<extra></extra>"
            ),
        )
        tacloban_map = subset_bbox(
            regional_display_stage_maps[stage_name],
            tacloban_zoom_bounds,
        )
        tacloban_heatmap = go.Heatmap(
            x=tacloban_map["x"].values,
            y=tacloban_map["y"].values,
            z=tacloban_map.values,
            colorscale="Inferno",
            zmin=0,
            zmax=NTL_DISPLAY_MAX,
            showscale=False,
            zsmooth=False,
            hoverongaps=False,
            hovertemplate=(
                "Median baseline: %{z:.2f} nW cm⁻² sr⁻¹"
                "<extra></extra>"
            ),
        )
    else:
        regional_map = regional_display_recovery_maps[stage_name]
        regional_heatmap = go.Heatmap(
            x=regional_map["x"].values,
            y=regional_map["y"].values,
            z=regional_map.values,
            coloraxis="coloraxis",
            zsmooth=False,
            hoverongaps=False,
            hovertemplate="Recovery: %{z:.1f}%<extra></extra>",
        )
        tacloban_map = subset_bbox(
            regional_display_recovery_maps[stage_name],
            tacloban_zoom_bounds,
        )
        tacloban_heatmap = go.Heatmap(
            x=tacloban_map["x"].values,
            y=tacloban_map["y"].values,
            z=tacloban_map.values,
            coloraxis="coloraxis",
            showscale=False,
            zsmooth=False,
            hoverongaps=False,
            hovertemplate="Recovery: %{z:.1f}%<extra></extra>",
        )

    fig_regional_recovery.add_trace(
        regional_heatmap,
        row=1,
        col=column,
    )
    add_spatial_overlays(
        fig_regional_recovery,
        row=1,
        column=column,
        zoomed=False,
    )

    fig_regional_recovery.add_trace(
        tacloban_heatmap,
        row=2,
        col=column,
    )
    add_spatial_overlays(
        fig_regional_recovery,
        row=2,
        column=column,
        zoomed=True,
    )

    # Regional row: full raster extent, matching the earlier figure.
    fig_regional_recovery.update_xaxes(
        range=[regional_display_bounds[0], regional_display_bounds[2]],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        row=1,
        col=column,
    )
    fig_regional_recovery.update_yaxes(
        range=[regional_display_bounds[1], regional_display_bounds[3]],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        scaleanchor=axis_id("x", column),
        scaleratio=1,
        constrain="domain",
        row=1,
        col=column,
    )

    # Second row: consistent square Tacloban zoom.
    zoom_axis_number = 5 + column
    fig_regional_recovery.update_xaxes(
        range=[tacloban_zoom_bounds[0], tacloban_zoom_bounds[2]],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        mirror=True,
        linecolor=PLOT_TEXT_COLOR,
        linewidth=1.4,
        row=2,
        col=column,
    )
    fig_regional_recovery.update_yaxes(
        range=[tacloban_zoom_bounds[1], tacloban_zoom_bounds[3]],
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        showline=True,
        mirror=True,
        linecolor=PLOT_TEXT_COLOR,
        linewidth=1.4,
        scaleanchor=axis_id("x", zoom_axis_number),
        scaleratio=1,
        constrain="domain",
        row=2,
        col=column,
    )


fig_regional_recovery.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="#2E2E2E",
    width=3000,
    height=1450,
    title=dict(
        text=(
            "Regional baseline and recovery with Tacloban detail | "
            f"{SETTLEMENT_MASK} | SC ≥ {SPATIAL_COMPLETENESS_PCT:.0f}%"
        ),
        x=0.5,
        font=dict(size=33),
    ),
    coloraxis=dict(
        colorscale=RECOVERY_COLORSCALE,
        cmin=0,
        cmax=140,
        colorbar=dict(
            title=dict(
                text="Recovery<br>(% baseline)",
                side="top",
                font=dict(size=17),
            ),
            tickvals=[0, 50, 80, 100, 120, 140],
            x=last_x_domain[1] + 0.008,
            y=sum(top_y_domain) / 2,
            xanchor="left",
            yanchor="middle",
            len=0.34,
            thickness=25,
            tickfont=dict(size=15),
            outlinecolor="rgba(36,59,90,0.35)",
            outlinewidth=0.8,
        ),
    ),
    font=dict(
        family=PLOT_FONT,
        size=19,
        color=PLOT_TEXT_COLOR,
    ),
    margin=dict(l=80, r=120, t=145, b=55),
)

fig_regional_recovery.update_annotations(
    font=dict(size=23, color=PLOT_TEXT_COLOR)
)

fig_regional_recovery.show()


## 4. Zoom in: maps, observability, and trajectory at each POI

Each location is read as one unit. The top row shows baseline radiance followed by stage recovery, with roads, administrative boundaries, the POI centre, and the 5×5 analytical kernel. The middle strip is spatial completeness. The lower panel shows raw daily recovery and the four-day profile. Error bars represent baseline sensitivity from the pixel-level baseline interquartile range.

In [ ]:
# ============================================================
# 10. CHARACTERISTIC POI PROFILES
# ============================================================

poi_table = pd.DataFrame(
    [
        {
            "poi_name": "Tacloban City Centre",
            "group": "City centre",
            "longitude": 125.0015,
            "latitude": 11.2434,
            "rationale": "Regional capital; severe Haiyan impact; dense urban lighting",
        },
        {
            "poi_name": "Ormoc City Centre",
            "group": "City centre",
            "longitude": 124.6068,
            "latitude": 11.0088,
            "rationale": "Western Leyte urban hub and port",
        },
        {
            "poi_name": "Baybay City Centre",
            "group": "City centre",
            "longitude": 124.7989,
            "latitude": 10.6766,
            "rationale": "Smaller western coastal city with persistent urban lights",
        },
        {
            "poi_name": "Palo Centre",
            "group": "Municipal centre",
            "longitude": 124.9904,
            "latitude": 11.1577,
            "rationale": "Eastern Leyte urban corridor and major resettlement context",
        },
        {
            "poi_name": "Alangalang Centre",
            "group": "Municipal centre",
            "longitude": 124.8465,
            "latitude": 11.2071,
            "rationale": "Inland peri-urban exposure contrast",
        },
        {
            "poi_name": "Guiuan Centre",
            "group": "Municipal centre",
            "longitude": 125.7232,
            "latitude": 11.0312,
            "rationale": "Near Haiyan's first landfall",
        },
    ]
)

In [ ]:
# ============================================================
# FIGURES 3A–3F. POI STORY PANELS
# ============================================================

POI_MAP_RADIUS_PIXELS = 9
POI_COLOR_CODE = "#FF3B30"
MUNICIPALITY_COLOR_CODE = "#56CCF2"


def poi_in_raster_crs(longitude, latitude):
    return (
        gpd.GeoSeries(
            [Point(longitude, latitude)],
            crs="EPSG:4326",
        )
        .to_crs(a2.rio.crs)
        .iloc[0]
    )


def rectangle_line(bounds):
    x_min, y_min, x_max, y_max = bounds

    return (
        [x_min, x_max, x_max, x_min, x_min],
        [y_min, y_min, y_max, y_max, y_min],
    )


x_resolution = float(
    np.median(
        np.abs(
            np.diff(dnb["x"].values)
        )
    )
)

y_resolution = float(
    np.median(
        np.abs(
            np.diff(dnb["y"].values)
        )
    )
)

poi_story_figures = {}

for poi in poi_table.itertuples():

    if (
        poi_four_day.empty
        or poi.poi_name not in set(poi_four_day["unit_name"])
    ):
        print(
            f"{poi.poi_name}: no admissible POI profile; "
            "map/time-series panel withheld."
        )
        continue

    point = poi_in_raster_crs(
        poi.longitude,
        poi.latitude,
    )

    display_bounds = (
        point.x - POI_MAP_RADIUS_PIXELS * x_resolution,
        point.y - POI_MAP_RADIUS_PIXELS * y_resolution,
        point.x + POI_MAP_RADIUS_PIXELS * x_resolution,
        point.y + POI_MAP_RADIUS_PIXELS * y_resolution,
    )

    kernel_half = POI_KERNEL_SIZE / 2

    kernel_bounds = (
        point.x - kernel_half * x_resolution,
        point.y - kernel_half * y_resolution,
        point.x + kernel_half * x_resolution,
        point.y + kernel_half * y_resolution,
    )

    kernel_x, kernel_y = rectangle_line(
        kernel_bounds
    )

    local_roads = roads_raster_crs.cx[
        display_bounds[0]:display_bounds[2],
        display_bounds[1]:display_bounds[3],
    ].copy()

    local_road_x, local_road_y = extract_line_coordinates(
        local_roads
    )

    local_units = municipalities_raster_crs.cx[
        display_bounds[0]:display_bounds[2],
        display_bounds[1]:display_bounds[3],
    ].copy()

    local_units["geometry"] = local_units.geometry.boundary

    local_boundary_x, local_boundary_y = extract_line_coordinates(
        local_units
    )

    daily = (
        poi_daily.loc[
            poi_daily["unit_name"] == poi.poi_name
        ]
        .sort_values("date_start")
        .copy()
    )

    four_day = (
        poi_four_day.loc[
            poi_four_day["unit_name"] == poi.poi_name
        ]
        .sort_values("date_start")
        .copy()
    )

    # --------------------------------------------------------
    # T50, T80, AND T90
    # Recovery progress is measured from the post-Haiyan nadir
    # back to 100%, where 100% is the 60-day baseline.
    # --------------------------------------------------------

    post_event_four_day = (
        four_day.loc[
            four_day["date_start"].ge(EVENT_DATE)
            & four_day["recovery_pct"].notna()
            & four_day["spatial_coverage_pct"].ge(
                SPATIAL_COMPLETENESS_PCT
            )
        ]
        .sort_values("date_start")
        .copy()
    )

    recovery_milestones = []
    baseline_pct = 100.0

    if not post_event_four_day.empty:
        nadir_index = post_event_four_day[
            "recovery_pct"
        ].idxmin()

        nadir_date = pd.Timestamp(
            post_event_four_day.loc[nadir_index, "date_start"]
        )

        nadir_pct = float(
            post_event_four_day.loc[nadir_index, "recovery_pct"]
        )

        recovery_search = post_event_four_day.loc[
            post_event_four_day["date_start"].ge(nadir_date)
        ]

        for milestone_pct in (50, 80, 90):
            # Example: if the nadir is 40% of baseline,
            # T50 is reached at 70%: 40 + 0.50 × (100 - 40).
            target_level_pct = (
                nadir_pct
                + (milestone_pct / 100)
                * (baseline_pct - nadir_pct)
            )

            reached = recovery_search.loc[
                recovery_search["recovery_pct"].ge(
                    target_level_pct
                )
            ]

            if reached.empty or nadir_pct >= baseline_pct:
                recovery_milestones.append(
                    {
                        "milestone_pct": milestone_pct,
                        "target_level_pct": target_level_pct,
                        "date": pd.NaT,
                        "recovery_pct": np.nan,
                        "days": np.nan,
                    }
                )
                continue

            milestone_row = reached.iloc[0]
            milestone_date = pd.Timestamp(
                milestone_row["date_start"]
            )

            recovery_milestones.append(
                {
                    "milestone_pct": milestone_pct,
                    "target_level_pct": target_level_pct,
                    "date": milestone_date,
                    "recovery_pct": milestone_row["recovery_pct"],
                    "days": (
                        milestone_date.normalize()
                        - nadir_date.normalize()
                    ).days,
                }
            )

    else:
        recovery_milestones = [
            {
                "milestone_pct": milestone_pct,
                "target_level_pct": np.nan,
                "date": pd.NaT,
                "recovery_pct": np.nan,
                "days": np.nan,
            }
            for milestone_pct in (50, 80, 90)
        ]

    milestone_summary = " | ".join(
        (
            f"T{item['milestone_pct']}: {item['days']:.0f}d"
            if pd.notna(item["date"])
            else f"T{item['milestone_pct']}: not reached"
        )
        for item in recovery_milestones
    )

    figure = make_subplots(
        rows=3,
        cols=len(stage_names),
        specs=[
            [{} for _ in stage_names],
            [
                {"colspan": len(stage_names)}
            ]
            + [None] * (len(stage_names) - 1),
            [
                {"colspan": len(stage_names)}
            ]
            + [None] * (len(stage_names) - 1),
        ],
        row_heights=[
            0.5,
            0.1,
            0.4,
        ],
        horizontal_spacing=0.012,
        vertical_spacing=0.075,
        subplot_titles=stage_names,
    )

    # --------------------------------------------------------
    # ROW 1. BASELINE AND RECOVERY MAPS
    # --------------------------------------------------------

    for column, stage_name in enumerate(
        stage_names,
        start=1,
    ):

        if stage_name == "Baseline":

            local_map = subset_bbox(
                regional_ntl0_four_day,
                display_bounds,
            )

            figure.add_trace(
                go.Heatmap(
                    x=local_map["x"].values,
                    y=local_map["y"].values,
                    z=local_map.values,
                    coloraxis="coloraxis3",
                    zsmooth=False,
                    hoverongaps=False,
                    hovertemplate=(
                        "Baseline: %{z:.2f} "
                        "nW cm⁻² sr⁻¹"
                        "<extra></extra>"
                    ),
                ),
                row=1,
                col=column,
            )

        else:

            local_map = subset_bbox(
                regional_recovery_maps[stage_name],
                display_bounds,
            )

            figure.add_trace(
                go.Heatmap(
                    x=local_map["x"].values,
                    y=local_map["y"].values,
                    z=local_map.values,
                    coloraxis="coloraxis",
                    zsmooth=False,
                    hoverongaps=False,
                    hovertemplate=(
                        "Recovery: %{z:.1f}%"
                        "<extra></extra>"
                    ),
                ),
                row=1,
                col=column,
            )

        figure.add_trace(
            go.Scatter(
                x=local_road_x,
                y=local_road_y,
                mode="lines",
                line=dict(
                    color="rgba(255,255,255,0.72)",
                    width=0.9,
                ),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column,
        )

        figure.add_trace(
            go.Scatter(
                x=local_boundary_x,
                y=local_boundary_y,
                mode="lines",
                line=dict(
                    color=MUNICIPALITY_COLOR_CODE,
                    width=1.1,
                ),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column,
        )

        figure.add_trace(
            go.Scatter(
                x=kernel_x,
                y=kernel_y,
                mode="lines",
                line=dict(
                    color="#00E5FF",
                    width=1.8,
                    dash="dash",
                ),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column,
        )

        figure.add_trace(
            go.Scatter(
                x=[point.x],
                y=[point.y],
                mode="markers",
                marker=dict(
                    size=8,
                    color=POI_COLOR_CODE,
                    symbol="x",
                    line=dict(width=1.2),
                ),
                hovertemplate=(
                    f"{poi.poi_name}"
                    "<extra></extra>"
                ),
                showlegend=False,
            ),
            row=1,
            col=column,
        )

        figure.update_xaxes(
            range=[
                display_bounds[0],
                display_bounds[2],
            ],
            showticklabels=False,
            showgrid=False,
            zeroline=False,
            row=1,
            col=column,
        )

        figure.update_yaxes(
            range=[
                display_bounds[1],
                display_bounds[3],
            ],
            showticklabels=False,
            showgrid=False,
            zeroline=False,
            row=1,
            col=column,
        )

    # --------------------------------------------------------
    # ROW 2. SPATIAL COMPLETENESS AS GREEN HEATMAP
    # --------------------------------------------------------

    figure.add_trace(
        go.Heatmap(
            x=four_day["date_start"],
            y=["Spatial completeness"],
            z=[
                four_day[
                    "spatial_coverage_pct"
                ].to_numpy()
            ],
            coloraxis="coloraxis2",
            zsmooth=False,
            hoverongaps=False,
            xgap=0,
            ygap=0,
            hovertemplate=(
                "%{x|%d %b %Y}"
                "<br>SC: %{z:.1f}%"
                "<extra></extra>"
            ),
            name="Spatial completeness",
            showlegend=False,
        ),
        row=2,
        col=1,
    )

    # --------------------------------------------------------
    # ROW 3. DAILY AND FOUR-DAY RECOVERY
    # --------------------------------------------------------

    figure.add_trace(
        go.Scatter(
            x=daily["date_start"],
            y=daily["recovery_pct"],
            mode="lines+markers",
            name="Daily RQ NTL",
            connectgaps=False,
            line=dict(
                color="#63BE7B",
                width=1.0,
            ),
            marker=dict(
                size=2.7,
            ),
            opacity=0.55,
            hovertemplate=(
                "%{x|%d %b %Y}"
                "<br>Daily: %{y:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

    error_plus = (
        four_day["recovery_high_pct"]
        - four_day["recovery_pct"]
    ).clip(lower=0)

    error_minus = (
        four_day["recovery_pct"]
        - four_day["recovery_low_pct"]
    ).clip(lower=0)

    figure.add_trace(
        go.Scatter(
            x=four_day["date_start"],
            y=four_day["recovery_pct"],
            mode="lines+markers",
            name="RQ NTL (4D) ± baseline IQR",
            connectgaps=False,
            line=dict(
                color="#008F3D",
                width=2.7,
                shape="hv",
            ),
            marker=dict(
                size=5,
            ),
            error_y=dict(
                type="data",
                symmetric=False,
                array=error_plus,
                arrayminus=error_minus,
                color="#006C2E",
                thickness=1.1,
                width=2,
            ),
            hovertemplate=(
                "%{x|%d %b %Y}"
                "<br>Recovery: %{y:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

    # Recovery milestones toward the 60-day baseline (100%).
    reached_milestones = [
        item
        for item in recovery_milestones
        if pd.notna(item["date"])
    ]

    if reached_milestones:
        milestone_colors = {
            50: "#F2C94C",
            80: "#F2994A",
            90: "#9B51E0",
        }

        for item in reached_milestones:
            milestone_pct = item["milestone_pct"]

            figure.add_trace(
                go.Scatter(
                    x=[item["date"]],
                    y=[item["target_level_pct"]],
                    mode="markers",
                    name=f"T{milestone_pct}",
                    marker=dict(
                        size=11,
                        symbol="diamond",
                        color=milestone_colors[milestone_pct],
                        line=dict(
                            color="white",
                            width=1.2,
                        ),
                    ),
                    customdata=[[
                        item["days"],
                        item["target_level_pct"],
                        item["recovery_pct"],
                    ]],
                    hovertemplate=(
                        f"T{milestone_pct}"
                        "<br>%{x|%d %b %Y}"
                        "<br>%{customdata[0]:.0f} days after nadir"
                        "<br>Threshold: %{customdata[1]:.1f}%"
                        "<br>Observed composite: %{customdata[2]:.1f}%"
                        "<extra></extra>"
                    ),
                    showlegend=True,
                ),
                row=3,
                col=1,
            )

    # --------------------------------------------------------
    # REFERENCE LINES
    # --------------------------------------------------------

    figure.add_hline(
        y=100,
        line=dict(
            color="#7F8C8D",
            width=1.2,
            dash="dot",
        ),
        row=3,
        col=1,
    )

    figure.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line=dict(
            color="#0057FF",
            width=2.0,
            dash="dash",
        ),
        row=2,
        col=1,
    )

    figure.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line=dict(
            color="#0057FF",
            width=2.0,
            dash="dash",
        ),
        row=3,
        col=1,
    )

    # --------------------------------------------------------
    # FORCE IDENTICAL SC AND TIME-SERIES X AXES
    # --------------------------------------------------------

    time_start = min(
        daily["date_start"].min(),
        four_day["date_start"].min(),
    )

    time_end = max(
        daily["date_start"].max(),
        four_day["date_start"].max(),
    )

    time_range = [
        time_start,
        time_end,
    ]

    # Row 1 creates len(stage_names) x-axes.
    # The SC panel is therefore the next axis.
    sc_xaxis_reference = (
        f"x{len(stage_names) + 1}"
    )

    figure.update_xaxes(
        range=time_range,
        autorange=False,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        row=2,
        col=1,
    )

    figure.update_xaxes(
        range=time_range,
        autorange=False,
        matches=sc_xaxis_reference,
        title_text="Date",
        showgrid=True,
        gridcolor="#E8EDF3",
        row=3,
        col=1,
    )

    figure.update_yaxes(
        title_text="SC",
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        row=2,
        col=1,
    )

    figure.update_yaxes(
        title_text="Recovery<br>(% baseline)",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#E8EDF3",
        row=3,
        col=1,
    )

    # --------------------------------------------------------
    # LAYOUT
    # --------------------------------------------------------

    figure.update_layout(
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        width=1600,
        height=900,

        title=dict(
            text=(
                f"{poi.poi_name} | {poi.rationale}"
                "<br>"
                f"<sup>{POI_KERNEL_SIZE}×"
                f"{POI_KERNEL_SIZE} VIIRS support; "
                f"{SETTLEMENT_MASK}; "
                f"SC ≥ "
                f"{SPATIAL_COMPLETENESS_PCT:.0f}%"
                f"; {milestone_summary}"
                "</sup>"
            ),
            x=0.5,
            xanchor="center",
        ),

        coloraxis=dict(
            colorscale=RECOVERY_COLORSCALE,
            cmin=0,
            cmax=140,
            colorbar=dict(
                title=(
                    "Recovery<br>"
                    "(% baseline)"
                ),
                tickvals=[
                    0,
                    50,
                    80,
                    100,
                    120,
                    140,
                ],
                x=1.005,
                y=0.85,
                len=0.3,
                thickness=16,
            ),
        ),

        coloraxis3=dict(
            colorscale="Inferno",
            cmin=0,
            cmax=map_color_max,
            colorbar=dict(
                title=(
                    "NTL<br>"
                    "(nW cm⁻² sr⁻¹)"
                ),
                tickformat=".1f",
                x=-0.05,
                y=0.95,
                len=0.16,
                thickness=16,
            ),
        ),

        coloraxis2=dict(
            colorscale=SC_GREEN_COLORSCALE,
            cmin=0,
            cmax=100,
            colorbar=dict(
                title="SC (%)",
                tickvals=[
                    0,
                    100,
                ],
                ticktext=[
                    "0",
                    "100",
                ],
                x=1.005,
                y=0.48,
                len=0.14,
                thickness=16,
            ),
        ),

        legend=dict(
            orientation="h",
            y=0.3,
            x=0.5,
        ),

        font=dict(
            family="Arial",
            size=16,
            color="#243B5A",
        ),

        margin=dict(
            l=90,
            r=150,
            t=125,
            b=65,
        ),

        hovermode="x unified",
    )

    poi_story_figures[poi.poi_name] = figure
    figure.show()


In [ ]:
# ============================================================
# FIGURES 3A–3F. MUNICIPALITY-WIDE G3 RQ NTL STORY PANELS
# ============================================================

MUNICIPALITY_MAP_PADDING = 0.08
municipality_story_figures = {}

if SETTLEMENT_MASK != "G3":
    raise ValueError(
        "Rerun the upstream profile cells with SETTLEMENT_MASK = 'G3' "
        "before generating the municipality-wide figures."
    )

location_context = {
    item["municipality_name"]: item
    for item in CORE_LOCATIONS
}

for municipality_name in LOCATION_ORDER:

    if (
        municipal_four_day.empty
        or municipality_name
        not in set(municipal_four_day["unit_name"].astype(str))
    ):
        print(
            f"{municipality_name}: no admissible G3 municipal profile; "
            "map/time-series panel withheld."
        )
        continue

    municipality = selected_raster_crs.loc[
        selected_raster_crs["unit_name"] == municipality_name
    ].copy()

    if len(municipality) != 1:
        raise ValueError(
            f"Expected one boundary for {municipality_name}; "
            f"found {len(municipality)}."
        )

    municipality_row = municipality.iloc[0]
    profile_id = int(municipality_row["profile_id"])
    display_bounds = square_bounds(
        municipality.total_bounds,
        pad_fraction=MUNICIPALITY_MAP_PADDING,
    )
    municipality_color = LOCATION_COLORS[municipality_name]
    municipality_g3_mask = (
        (selected_zone_id == profile_id)
        & ghsl_mask
    )
    local_g3_mask = subset_bbox(
        municipality_g3_mask,
        display_bounds,
    )
    municipality_context = location_context[municipality_name]

    local_roads = roads_raster_crs.cx[
        display_bounds[0]:display_bounds[2],
        display_bounds[1]:display_bounds[3],
    ].copy()
    local_roads = gpd.clip(local_roads, municipality)

    local_road_x, local_road_y = extract_line_coordinates(
        local_roads
    )

    local_units = municipality.copy()
    local_units["geometry"] = local_units.geometry.boundary

    local_boundary_x, local_boundary_y = extract_line_coordinates(
        local_units
    )

    daily = (
        municipal_daily.loc[
            municipal_daily["unit_name"].astype(str)
            == municipality_name
        ]
        .sort_values("date_start")
        .copy()
    )

    four_day = (
        municipal_four_day.loc[
            municipal_four_day["unit_name"].astype(str)
            == municipality_name
        ]
        .sort_values("date_start")
        .copy()
    )

    # --------------------------------------------------------
    # T50, T80, AND T90
    # Recovery progress is measured from the post-Haiyan nadir
    # back to 100%, where 100% is the 60-day baseline.
    # --------------------------------------------------------

    post_event_four_day = (
        four_day.loc[
            four_day["date_start"].ge(EVENT_DATE)
            & four_day["recovery_pct"].notna()
            & four_day["spatial_coverage_pct"].ge(
                SPATIAL_COMPLETENESS_PCT
            )
        ]
        .sort_values("date_start")
        .copy()
    )

    recovery_milestones = []
    baseline_pct = 100.0

    if not post_event_four_day.empty:
        nadir_index = post_event_four_day[
            "recovery_pct"
        ].idxmin()

        nadir_date = pd.Timestamp(
            post_event_four_day.loc[nadir_index, "date_start"]
        )

        nadir_pct = float(
            post_event_four_day.loc[nadir_index, "recovery_pct"]
        )

        recovery_search = post_event_four_day.loc[
            post_event_four_day["date_start"].ge(nadir_date)
        ]

        for milestone_pct in (50, 80, 90):
            # Example: if the nadir is 40% of baseline,
            # T50 is reached at 70%: 40 + 0.50 × (100 - 40).
            target_level_pct = (
                nadir_pct
                + (milestone_pct / 100)
                * (baseline_pct - nadir_pct)
            )

            reached = recovery_search.loc[
                recovery_search["recovery_pct"].ge(
                    target_level_pct
                )
            ]

            if reached.empty or nadir_pct >= baseline_pct:
                recovery_milestones.append(
                    {
                        "milestone_pct": milestone_pct,
                        "target_level_pct": target_level_pct,
                        "date": pd.NaT,
                        "recovery_pct": np.nan,
                        "days": np.nan,
                    }
                )
                continue

            milestone_row = reached.iloc[0]
            milestone_date = pd.Timestamp(
                milestone_row["date_start"]
            )

            recovery_milestones.append(
                {
                    "milestone_pct": milestone_pct,
                    "target_level_pct": target_level_pct,
                    "date": milestone_date,
                    "recovery_pct": milestone_row["recovery_pct"],
                    "days": (
                        milestone_date.normalize()
                        - nadir_date.normalize()
                    ).days,
                }
            )

    else:
        recovery_milestones = [
            {
                "milestone_pct": milestone_pct,
                "target_level_pct": np.nan,
                "date": pd.NaT,
                "recovery_pct": np.nan,
                "days": np.nan,
            }
            for milestone_pct in (50, 80, 90)
        ]

    milestone_summary = " | ".join(
        (
            f"T{item['milestone_pct']}: {item['days']:.0f}d"
            if pd.notna(item["date"])
            else f"T{item['milestone_pct']}: not reached"
        )
        for item in recovery_milestones
    )

    figure = make_subplots(
        rows=3,
        cols=len(stage_names),
        specs=[
            [{} for _ in stage_names],
            [
                {"colspan": len(stage_names)}
            ]
            + [None] * (len(stage_names) - 1),
            [
                {"colspan": len(stage_names)}
            ]
            + [None] * (len(stage_names) - 1),
        ],
        row_heights=[
            0.5,
            0.1,
            0.4,
        ],
        horizontal_spacing=0.012,
        vertical_spacing=0.075,
        subplot_titles=stage_names,
    )

    # --------------------------------------------------------
    # ROW 1. BASELINE AND RECOVERY MAPS
    # --------------------------------------------------------

    for column, stage_name in enumerate(
        stage_names,
        start=1,
    ):

        if stage_name == "Baseline":

            local_map = subset_bbox(
                regional_ntl0_four_day,
                display_bounds,
            ).where(local_g3_mask)

            figure.add_trace(
                go.Heatmap(
                    x=local_map["x"].values,
                    y=local_map["y"].values,
                    z=local_map.values,
                    coloraxis="coloraxis3",
                    zsmooth=False,
                    hoverongaps=False,
                    hovertemplate=(
                        "G3 baseline: %{z:.2f} "
                        "nW cm⁻² sr⁻¹"
                        "<extra></extra>"
                    ),
                ),
                row=1,
                col=column,
            )

        else:

            local_map = subset_bbox(
                regional_recovery_maps[stage_name],
                display_bounds,
            ).where(local_g3_mask)

            figure.add_trace(
                go.Heatmap(
                    x=local_map["x"].values,
                    y=local_map["y"].values,
                    z=local_map.values,
                    coloraxis="coloraxis",
                    zsmooth=False,
                    hoverongaps=False,
                    hovertemplate=(
                        "G3 recovery: %{z:.1f}%"
                        "<extra></extra>"
                    ),
                ),
                row=1,
                col=column,
            )

        figure.add_trace(
            go.Scatter(
                x=local_road_x,
                y=local_road_y,
                mode="lines",
                line=dict(
                    color="rgba(255,255,255,0.72)",
                    width=0.9,
                ),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column,
        )

        figure.add_trace(
            go.Scatter(
                x=local_boundary_x,
                y=local_boundary_y,
                mode="lines",
                line=dict(
                    color=municipality_color,
                    width=2.0,
                ),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=column,
        )

        figure.update_xaxes(
            range=[
                display_bounds[0],
                display_bounds[2],
            ],
            showticklabels=False,
            showgrid=False,
            zeroline=False,
            row=1,
            col=column,
        )

        figure.update_yaxes(
            range=[
                display_bounds[1],
                display_bounds[3],
            ],
            showticklabels=False,
            showgrid=False,
            zeroline=False,
            row=1,
            col=column,
        )

    # --------------------------------------------------------
    # ROW 2. SPATIAL COMPLETENESS AS GREEN HEATMAP
    # --------------------------------------------------------

    figure.add_trace(
        go.Heatmap(
            x=four_day["date_start"],
            y=["G3 spatial completeness"],
            z=[
                four_day[
                    "spatial_coverage_pct"
                ].to_numpy()
            ],
            coloraxis="coloraxis2",
            zsmooth=False,
            hoverongaps=False,
            xgap=0,
            ygap=0,
            hovertemplate=(
                "%{x|%d %b %Y}"
                "<br>SC: %{z:.1f}%"
                "<extra></extra>"
            ),
            name="G3 spatial completeness",
            showlegend=False,
        ),
        row=2,
        col=1,
    )

    # --------------------------------------------------------
    # ROW 3. DAILY AND FOUR-DAY RECOVERY
    # --------------------------------------------------------

    figure.add_trace(
        go.Scatter(
            x=daily["date_start"],
            y=daily["recovery_pct"],
            mode="lines+markers",
            name="Daily G3 RQ NTL",
            connectgaps=False,
            line=dict(
                color=municipality_color,
                width=1.0,
            ),
            marker=dict(
                size=2.7,
            ),
            opacity=0.55,
            hovertemplate=(
                "%{x|%d %b %Y}"
                "<br>Daily: %{y:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

    error_plus = (
        four_day["recovery_high_pct"]
        - four_day["recovery_pct"]
    ).clip(lower=0)

    error_minus = (
        four_day["recovery_pct"]
        - four_day["recovery_low_pct"]
    ).clip(lower=0)

    figure.add_trace(
        go.Scatter(
            x=four_day["date_start"],
            y=four_day["recovery_pct"],
            mode="lines+markers",
            name="G3 RQ NTL (4D) ± baseline IQR",
            connectgaps=False,
            line=dict(
                color=municipality_color,
                width=2.7,
                shape="hv",
            ),
            marker=dict(
                size=5,
            ),
            error_y=dict(
                type="data",
                symmetric=False,
                array=error_plus,
                arrayminus=error_minus,
                color=municipality_color,
                thickness=1.1,
                width=2,
            ),
            hovertemplate=(
                "%{x|%d %b %Y}"
                "<br>Recovery: %{y:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

    # Recovery milestones toward the 60-day baseline (100%).
    reached_milestones = [
        item
        for item in recovery_milestones
        if pd.notna(item["date"])
    ]

    if reached_milestones:
        for item in reached_milestones:
            milestone_pct = item["milestone_pct"]

            figure.add_trace(
                go.Scatter(
                    x=[item["date"]],
                    y=[item["target_level_pct"]],
                    mode="markers",
                    name=f"T{milestone_pct}",
                    marker=dict(
                        size=11,
                        symbol=MILESTONE_SYMBOLS[milestone_pct],
                        color=MILESTONE_COLORS[milestone_pct],
                        line=dict(
                            color="white",
                            width=1.2,
                        ),
                    ),
                    customdata=[[
                        item["days"],
                        item["target_level_pct"],
                        item["recovery_pct"],
                    ]],
                    hovertemplate=(
                        f"T{milestone_pct}"
                        "<br>%{x|%d %b %Y}"
                        "<br>%{customdata[0]:.0f} days after nadir"
                        "<br>Threshold: %{customdata[1]:.1f}%"
                        "<br>Observed composite: %{customdata[2]:.1f}%"
                        "<extra></extra>"
                    ),
                    showlegend=True,
                ),
                row=3,
                col=1,
            )

    # --------------------------------------------------------
    # REFERENCE LINES
    # --------------------------------------------------------

    figure.add_hline(
        y=100,
        line=dict(
            color="#7F8C8D",
            width=1.2,
            dash="dot",
        ),
        row=3,
        col=1,
    )

    figure.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line=dict(
            color="#0057FF",
            width=2.0,
            dash="dash",
        ),
        row=2,
        col=1,
    )

    figure.add_vline(
        x=EVENT_DATE.to_pydatetime(),
        line=dict(
            color="#0057FF",
            width=2.0,
            dash="dash",
        ),
        row=3,
        col=1,
    )

    # --------------------------------------------------------
    # FORCE IDENTICAL SC AND TIME-SERIES X AXES
    # --------------------------------------------------------

    time_start = min(
        daily["date_start"].min(),
        four_day["date_start"].min(),
    )

    time_end = max(
        daily["date_start"].max(),
        four_day["date_start"].max(),
    )

    time_range = [
        time_start,
        time_end,
    ]

    # Row 1 creates len(stage_names) x-axes.
    # The SC panel is therefore the next axis.
    sc_xaxis_reference = (
        f"x{len(stage_names) + 1}"
    )

    figure.update_xaxes(
        range=time_range,
        autorange=False,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        row=2,
        col=1,
    )

    figure.update_xaxes(
        range=time_range,
        autorange=False,
        matches=sc_xaxis_reference,
        title_text="Date",
        showgrid=True,
        gridcolor="#E8EDF3",
        row=3,
        col=1,
    )

    figure.update_yaxes(
        title_text="SC",
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        row=2,
        col=1,
    )

    figure.update_yaxes(
        title_text="Recovery<br>(% baseline)",
        rangemode="tozero",
        showgrid=True,
        gridcolor="#E8EDF3",
        row=3,
        col=1,
    )

    # --------------------------------------------------------
    # LAYOUT
    # --------------------------------------------------------

    figure.update_layout(
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        width=1600,
        height=900,

        title=dict(
            text=(
                f"{municipality_name} | "
                f"{municipality_context['why']}"
                "<br>"
                f"<sup>Whole municipality; "
                f"{SETTLEMENT_MASK} reliability-qualified NTL; "
                f"SC ≥ "
                f"{SPATIAL_COMPLETENESS_PCT:.0f}%"
                f"; {milestone_summary}"
                "</sup>"
            ),
            x=0.5,
            xanchor="center",
        ),

        coloraxis=dict(
            colorscale=RECOVERY_COLORSCALE,
            cmin=0,
            cmax=140,
            colorbar=dict(
                title=(
                    "Recovery<br>"
                    "(% baseline)"
                ),
                tickvals=[
                    0,
                    50,
                    80,
                    100,
                    120,
                    140,
                ],
                x=1.005,
                y=0.85,
                len=0.3,
                thickness=16,
            ),
        ),

        coloraxis3=dict(
            colorscale="Inferno",
            cmin=0,
            cmax=map_color_max,
            colorbar=dict(
                title=(
                    "NTL<br>"
                    "(nW cm⁻² sr⁻¹)"
                ),
                tickformat=".1f",
                x=-0.05,
                y=0.95,
                len=0.16,
                thickness=16,
            ),
        ),

        coloraxis2=dict(
            colorscale=SC_GREEN_COLORSCALE,
            cmin=0,
            cmax=100,
            colorbar=dict(
                title="SC (%)",
                tickvals=[
                    0,
                    100,
                ],
                ticktext=[
                    "0",
                    "100",
                ],
                x=1.005,
                y=0.48,
                len=0.14,
                thickness=16,
            ),
        ),

        legend=dict(
            orientation="h",
            y=0.3,
            x=0.5,
        ),

        font=dict(
            family="Arial",
            size=16,
            color="#243B5A",
        ),

        margin=dict(
            l=90,
            r=150,
            t=125,
            b=65,
        ),

        hovermode="x unified",
    )

    municipality_story_figures[municipality_name] = figure
    figure.show()

## 5. Separate municipality and POI recovery profiles

The municipality grid is shown first, followed by the POI grid. Both retain the short green spatial-completeness strip and baseline-IQR error bars. T50, T80, and T90 are markers only.

In [ ]:
# ============================================================
# FIGURES 7–8. SEPARATE MUNICIPALITY AND POI PROFILE GRIDS
# ============================================================


PROFILE_VALUE_MODE = "recovery"  # Use "recovery" or "raw".


def build_support_profile_grid(
    profiles,
    metrics,
    order,
    parent_lookup,
    title,
    trace_name,
    raw_trace_name,
    value_mode="recovery",
):
    if value_mode not in {"recovery", "raw"}:
        raise ValueError(
            "value_mode must be either 'recovery' or 'raw'."
        )

    is_recovery = value_mode == "recovery"
    value_column = (
        "recovery_pct"
        if is_recovery
        else "raw_median_ntl"
    )

    columns = 3
    location_rows = 2

    figure = make_subplots(
        rows=location_rows * 2,
        cols=columns,
        row_heights=[0.055, 0.445] * location_rows,
        horizontal_spacing=0.055,
        vertical_spacing=0.070,
    )

    metric_lookup = metrics.set_index("unit_name")
    time_range = [ANALYSIS_START, PROFILE_END]

    top_sc_domain = figure.layout.yaxis.domain
    sc_colorbar_y = float(np.mean(top_sc_domain))
    sc_colorbar_len = 0.10

    for index, unit_name in enumerate(order):
        group = index // columns
        column = index % columns + 1
        sc_row = group * 2 + 1
        recovery_row = sc_row + 1
        parent = parent_lookup[unit_name]
        color = LOCATION_COLORS[parent]
        profile = profiles.loc[
            profiles["unit_name"].astype(str) == unit_name
        ].sort_values("date_start")

        figure.add_trace(
            go.Heatmap(
                x=profile["date_start"],
                y=["SC"],
                z=[profile["spatial_coverage_pct"].to_numpy(dtype=float)],
                colorscale=SC_GREEN_COLORSCALE,
                zmin=0, zmax=100,
                showscale=index == 0,
                colorbar=dict(
                    title="SC (%)",
                    x=1.012,
                    y=sc_colorbar_y,
                    len=sc_colorbar_len,
                    thickness=14,
                    tickvals=[0, 50, 100],
                    tickfont=dict(size=12),
                ) if index == 0 else None,
                zsmooth=False,
                hovertemplate="%{x|%d %b %Y}<br>SC: %{z:.1f}%<extra></extra>",
            ),
            row=sc_row, col=column,
        )

        if is_recovery:
            error_plus = (
                profile["recovery_high_pct"]
                - profile["recovery_pct"]
            ).clip(lower=0)
            error_minus = (
                profile["recovery_pct"]
                - profile["recovery_low_pct"]
            ).clip(lower=0)
            error_y = dict(
                type="data",
                symmetric=False,
                array=error_plus,
                arrayminus=error_minus,
                color=hex_to_rgba(color, 0.50),
                thickness=0.9,
                width=1.5,
            )
            hovertemplate = (
                "%{x|%d %b %Y}"
                "<br>Recovery: %{y:.1f}%"
                "<extra></extra>"
            )
        else:
            error_y = None
            hovertemplate = (
                "%{x|%d %b %Y}"
                "<br>Raw median NTL: %{y:.2f} nW cm⁻² sr⁻¹"
                "<extra></extra>"
            )

        figure.add_trace(
            go.Scatter(
                x=profile["date_start"],
                y=profile[value_column],
                mode="lines+markers",
                name=(trace_name if is_recovery else raw_trace_name),
                showlegend=index == 0,
                connectgaps=False,
                line=dict(color=color, width=3.2, shape="hv"),
                marker=dict(size=5),
                error_y=error_y,
                hovertemplate=hovertemplate,
            ),
            row=recovery_row, col=column,
        )

        if is_recovery:
            metric_row = metric_lookup.loc[unit_name]

            for threshold in (50, 80, 90):
                day = metric_row[f"T{threshold}_day"]

                if pd.isna(day):
                    continue

                date = EVENT_DATE + pd.Timedelta(
                    days=float(day)
                )

                figure.add_trace(
                    go.Scatter(
                        x=[date],
                        y=[threshold],
                        mode="markers",
                        name=f"T{threshold}",
                        legendgroup=f"T{threshold}",
                        showlegend=False,
                        marker=dict(
                            size=15,
                            color=MILESTONE_COLORS[threshold],
                            symbol=MILESTONE_SYMBOLS[threshold],
                            line=dict(color="white", width=1.4),
                        ),
                        hovertemplate=(
                            f"{unit_name}<br>"
                            f"T{threshold}: {day:.0f} days"
                            "<extra></extra>"
                        ),
                    ),
                    row=recovery_row,
                    col=column,
                )

            figure.add_hline(
                y=100,
                line=dict(
                    color="#6C7882",
                    width=1.5,
                    dash="dot",
                ),
                row=recovery_row,
                col=column,
            )

        for technical_row in (sc_row, recovery_row):
            figure.add_vline(
                x=EVENT_DATE.to_pydatetime(),
                line=dict(color="#0057FF", width=2.2, dash="dash"),
                row=technical_row, col=column,
            )
            figure.update_xaxes(
                range=time_range,
                showticklabels=technical_row == recovery_row,
                tickformat="%b<br>%Y",
                nticks=6,
                tickfont=dict(size=14),
                row=technical_row, col=column,
            )
        figure.update_yaxes(
            showticklabels=False, showgrid=False,
            row=sc_row, col=column,
        )
        figure.update_yaxes(
            title_text=(
                (
                    "Recovery (%)"
                    if is_recovery
                    else "Raw median NTL<br>(nW cm⁻² sr⁻¹)"
                )
                if column == 1
                else None
            ),
            title_font=dict(size=17), tickfont=dict(size=14),
            rangemode="tozero", showgrid=True, gridcolor="#E8EDF3",
            row=recovery_row, col=column,
        )

        axis_number = (sc_row - 1) * columns + column
        display_label = DISPLAY_NAMES[parent] if unit_name == parent else unit_name
        sc_yaxis_name = axis_id("yaxis", axis_number)
        label_y = (
            figure.layout[sc_yaxis_name].domain[1]
            - 0.10
        )

        figure.add_annotation(
            x=0.60,
            y=label_y,
            xref=f"{axis_id('x', axis_number)} domain",
            yref="paper",
            text=f"<b>{display_label}</b>",
            showarrow=False,
            xanchor="left",
            yanchor="bottom",
            font=dict(size=20, color=color),
        )

    # Legend-only milestone traces guarantee that T50, T80, and T90
    # are all represented even when the first location lacks a crossing.
    if is_recovery:
        for threshold in (50, 80, 90):
            figure.add_trace(
                go.Scatter(
                    x=[None],
                    y=[None],
                    mode="markers",
                    name=f"T{threshold}",
                    legendgroup=f"T{threshold}",
                    showlegend=True,
                    marker=dict(
                        size=15,
                        color=MILESTONE_COLORS[threshold],
                        symbol=MILESTONE_SYMBOLS[threshold],
                        line=dict(color="white", width=1.4),
                    ),
                    hoverinfo="skip",
                )
            )

    figure.update_layout(
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        width=1400,
        height=1000,
        title=dict(text=title, x=0.5, font=dict(size=31)),
        legend=dict(
            orientation="h", x=0.5, y=1.10,
            xanchor="center", font=dict(size=17),
            bgcolor="rgba(255,255,255,0.86)",
        ),
        font=dict(family="Arial", size=17, color="#243B5A"),
        margin=dict(l=120, r=115, t=175, b=80),
        hovermode="closest",
    )
    return figure


municipality_parent_lookup = {name: name for name in LOCATION_ORDER}
poi_parent_lookup = dict(zip(poi_table["poi_name"], poi_table["municipality_name"]))

fig_municipality_profiles = build_support_profile_grid(
    profiles=municipal_four_day,
    metrics=municipal_metrics,
    order=LOCATION_ORDER,
    parent_lookup=municipality_parent_lookup,
    title=(
        "Reliability-qualified NTL recovery: Six City/Municipalities"
        if PROFILE_VALUE_MODE == "recovery"
        else "Raw reliability-qualified NTL: six municipalities"
    ),
    trace_name="City/Municipality RQ NTL (4D median)",
    raw_trace_name="City/Municipality raw RQ NTL (4D median)",
    value_mode=PROFILE_VALUE_MODE,
)
fig_municipality_profiles.show()

fig_poi_profiles = build_support_profile_grid(
    profiles=poi_four_day,
    metrics=poi_metrics,
    order=POI_ORDER,
    parent_lookup=poi_parent_lookup,
    title=(
        f"Reliability-qualified NTL recovery: Six "
        f"{POI_KERNEL_SIZE}×{POI_KERNEL_SIZE} POI kernels"
        if PROFILE_VALUE_MODE == "recovery"
        else (
            f"Raw reliability-qualified NTL: Six "
            f"{POI_KERNEL_SIZE}×{POI_KERNEL_SIZE} POI kernels"
        )
    ),
    trace_name="POI RQ NTL (4D median)",
    raw_trace_name="POI raw RQ NTL (4D median)",
    value_mode=PROFILE_VALUE_MODE,
)
fig_poi_profiles.show()


### Raw versus normalized observed summary

The left column retains raw daily magnitudes. The right column shows normalized four-day recovery with milestone markers only.

In [ ]:
# ============================================================
# FIGURE 9A–9B. MUNICIPALITY VERSUS POI: RAW AND RECOVERY
# ============================================================

# Support type, rather than location, controls the series colour.
# Location colours remain on the row labels.
MUNICIPALITY_SERIES_COLOR = "#174A7E"
POI_SERIES_COLOR = "#D81B60"

FIGURE_9_GROUPS = [
    {
        "suffix": "9A",
        "municipalities": [
            "Tacloban City",
            "Ormoc City",
            "Guiuan",
        ],
        "subtitle": "Tacloban, Ormoc, and Guiuan",
    },
    {
        "suffix": "9B",
        "municipalities": [
            "Palo",
            "Baybay City",
            "Alangalang",
        ],
        "subtitle": "Palo, Baybay, and Alangalang",
    },
]

location_lookup = {
    location["municipality_name"]: location
    for location in CORE_LOCATIONS
}

municipal_metric_lookup = municipal_metrics.set_index(
    "unit_name"
)
poi_metric_lookup = poi_metrics.set_index("unit_name")

summary_recovery_values = pd.concat(
    [
        municipal_four_day["recovery_pct"],
        poi_four_day["recovery_pct"],
    ],
    ignore_index=True,
).dropna()

summary_recovery_max = max(
    150.0,
    float(summary_recovery_values.max()) * 1.05,
)


def build_raw_vs_recovery_group(
    municipality_names,
    subtitle,
    figure_suffix,
):
    figure = make_subplots(
        rows=3,
        cols=2,
        column_widths=[0.49, 0.51],
        horizontal_spacing=0.085,
        vertical_spacing=0.075,
    )

    support_styles = [
        {
            "support": "Municipality",
            "color": MUNICIPALITY_SERIES_COLOR,
            "dash": "solid",
            "symbol": "circle",
            "legend_name": "City/Municipality",
        },
        {
            "support": "POI kernel",
            "color": POI_SERIES_COLOR,
            "dash": "dash",
            "symbol": "diamond-open",
            "legend_name": "POI kernel",
        },
    ]

    for location_index, municipality_name in enumerate(
        municipality_names
    ):
        row = location_index + 1
        location = location_lookup[municipality_name]
        poi_name = location["poi_name"]
        location_color = LOCATION_COLORS[municipality_name]

        profiles = {
            "Municipality": {
                "raw": municipal_daily.loc[
                    municipal_daily["unit_name"].astype(str)
                    == municipality_name
                ].sort_values("date_start"),
                "recovery": municipal_four_day.loc[
                    municipal_four_day["unit_name"].astype(str)
                    == municipality_name
                ].sort_values("date_start"),
                "metric_name": municipality_name,
                "metric_lookup": municipal_metric_lookup,
            },
            "POI kernel": {
                "raw": poi_daily.loc[
                    poi_daily["unit_name"].astype(str)
                    == poi_name
                ].sort_values("date_start"),
                "recovery": poi_four_day.loc[
                    poi_four_day["unit_name"].astype(str)
                    == poi_name
                ].sort_values("date_start"),
                "metric_name": poi_name,
                "metric_lookup": poi_metric_lookup,
            },
        }

        figure.add_vrect(
            x0=BASELINE_START,
            x1=PRE_EVENT_END,
            fillcolor="#7F8C8D",
            opacity=0.09,
            line_width=0,
            row=row,
            col=1,
        )

        for style in support_styles:
            support = style["support"]
            raw_profile = profiles[support]["raw"]
            recovery_profile = profiles[support]["recovery"]

            figure.add_trace(
                go.Scatter(
                    x=raw_profile["date_start"],
                    y=raw_profile["raw_median_ntl"],
                    mode="lines+markers",
                    name=style["legend_name"],
                    legendgroup=support,
                    showlegend=location_index == 0,
                    connectgaps=False,
                    line=dict(
                        color=style["color"],
                        width=3.2,
                        dash=style["dash"],
                    ),
                    marker=dict(
                        size=5.5,
                        symbol=style["symbol"],
                        color=style["color"],
                        line=dict(
                            color=style["color"],
                            width=1.3,
                        ),
                    ),
                    hovertemplate=(
                        f"{support}<br>"
                        "%{x|%d %b %Y}"
                        "<br>Raw median NTL: "
                        "%{y:.2f} nW cm⁻² sr⁻¹"
                        "<extra></extra>"
                    ),
                ),
                row=row,
                col=1,
            )

            figure.add_trace(
                go.Scatter(
                    x=recovery_profile["date_start"],
                    y=recovery_profile["recovery_pct"],
                    mode="lines+markers",
                    name=style["legend_name"],
                    legendgroup=support,
                    showlegend=False,
                    connectgaps=False,
                    line=dict(
                        color=style["color"],
                        width=3.5,
                        dash=style["dash"],
                        shape="hv",
                    ),
                    marker=dict(
                        size=6.0,
                        symbol=style["symbol"],
                        color=style["color"],
                        line=dict(
                            color=style["color"],
                            width=1.4,
                        ),
                    ),
                    hovertemplate=(
                        f"{support}<br>"
                        "%{x|%d %b %Y}"
                        "<br>Recovery: %{y:.1f}%"
                        "<extra></extra>"
                    ),
                ),
                row=row,
                col=2,
            )

        # Filled markers are municipality milestones; open markers
        # are the matched POI-kernel milestones.
        for support, marker_suffix in [
            ("Municipality", ""),
            ("POI kernel", "-open"),
        ]:
            metric_name = profiles[support]["metric_name"]
            metric_lookup = profiles[support]["metric_lookup"]
            metric_row = metric_lookup.loc[metric_name]

            for threshold in (50, 80, 90):
                day = metric_row[f"T{threshold}_day"]

                if pd.isna(day):
                    continue

                milestone_date = EVENT_DATE + pd.Timedelta(
                    days=float(day)
                )

                figure.add_trace(
                    go.Scatter(
                        x=[milestone_date],
                        y=[threshold],
                        mode="markers",
                        name=f"T{threshold}",
                        legendgroup=f"T{threshold}",
                        showlegend=False,
                        marker=dict(
                            size=(14 if support == "Municipality" else 16),
                            color=MILESTONE_COLORS[threshold],
                            symbol=(
                                MILESTONE_SYMBOLS[threshold]
                                + marker_suffix
                            ),
                            line=dict(
                                color=MILESTONE_COLORS[threshold],
                                width=1.7,
                            ),
                        ),
                        hovertemplate=(
                            f"{metric_name}<br>"
                            f"{support}<br>"
                            f"T{threshold}: {day:.0f} days"
                            "<extra></extra>"
                        ),
                    ),
                    row=row,
                    col=2,
                )

        for column in (1, 2):
            figure.add_vline(
                x=EVENT_DATE.to_pydatetime(),
                line=dict(
                    color="#0057FF",
                    width=2.2,
                    dash="dash",
                ),
                row=row,
                col=column,
            )

            figure.update_xaxes(
                range=[ANALYSIS_START, PROFILE_END],
                showticklabels=row == 3,
                title_text="Date" if row == 3 else None,
                tickformat="%b<br>%Y",
                nticks=7,
                tickfont=dict(size=14),
                showgrid=True,
                gridcolor="#E8EDF3",
                row=row,
                col=column,
            )

        figure.add_hline(
            y=100,
            line=dict(
                color="#6C7882",
                width=1.5,
                dash="dot",
            ),
            row=row,
            col=2,
        )

        figure.update_yaxes(
            title_text=(
                "Raw daily median NTL"
                if row == 2
                else None
            ),
            rangemode="tozero",
            tickfont=dict(size=14),
            showgrid=True,
            gridcolor="#E8EDF3",
            row=row,
            col=1,
        )

        figure.update_yaxes(
            title_text=(
                "Recovery (% baseline)"
                if row == 2
                else None
            ),
            range=[0, summary_recovery_max],
            tickfont=dict(size=14),
            showgrid=True,
            gridcolor="#E8EDF3",
            row=row,
            col=2,
        )

        raw_axis_number = (row - 1) * 2 + 1

        figure.add_annotation(
            x=0.015,
            y=0.96,
            xref=f"{axis_id('x', raw_axis_number)} domain",
            yref=f"{axis_id('y', raw_axis_number)} domain",
            text=f"<b>{location['core_location']}</b>",
            showarrow=False,
            xanchor="left",
            yanchor="top",
            font=dict(size=20, color=location_color),
            bgcolor="rgba(255,255,255,0.88)",
        )

    # Legend-only traces ensure that all recovery milestones appear,
    # even if the first municipality lacks T80 or T90.
    for threshold in (50, 80, 90):
        figure.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                name=f"T{threshold}",
                legendgroup=f"T{threshold}",
                showlegend=True,
                marker=dict(
                    size=14,
                    color=MILESTONE_COLORS[threshold],
                    symbol=MILESTONE_SYMBOLS[threshold],
                    line=dict(color="white", width=1.3),
                ),
                hoverinfo="skip",
            )
        )


    figure.update_layout(
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="white",
        autosize=False,
        width=1600,
        height=1000,
        title=dict(
            text=(
                f"Raw vs Baseline Normalized "
                "Observed Nighttime Lights"
                f"<br><sup>{subtitle}</sup>"
            ),
            x=0.5,
            font=dict(size=30),
        ),
        legend=dict(
            orientation="h",
            x=0.5,
            y=1.05,
            xanchor="center",
            font=dict(size=24),
            bgcolor="rgba(255,255,255,0.90)",
            traceorder="normal",
        ),
        font=dict(
            family="Arial",
            size=16,
            color="#243B5A",
        ),
        margin=dict(l=120, r=55, t=180, b=80),
        hovermode="closest",
    )

    return figure


fig_raw_vs_observed_9a = build_raw_vs_recovery_group(
    municipality_names=FIGURE_9_GROUPS[0]["municipalities"],
    subtitle=FIGURE_9_GROUPS[0]["subtitle"],
    figure_suffix=FIGURE_9_GROUPS[0]["suffix"],
)
fig_raw_vs_observed_9a.show()

fig_raw_vs_observed_9b = build_raw_vs_recovery_group(
    municipality_names=FIGURE_9_GROUPS[1]["municipalities"],
    subtitle=FIGURE_9_GROUPS[1]["subtitle"],
    figure_suffix=FIGURE_9_GROUPS[1]["suffix"],
)
fig_raw_vs_observed_9b.show()

fig_raw_vs_observed_summaries = [
    fig_raw_vs_observed_9a,
    fig_raw_vs_observed_9b,
]


In [ ]:
# ============================================================
# FIGURE 10. POOLED SIX-MUNICIPALITY REGIONAL BRIDGE
# ============================================================

fig_regional_bridge = go.Figure()
fig_regional_bridge.add_trace(
    go.Scatter(
        x=regional_four_day["date_start"],
        y=regional_four_day["recovery_pct"],
        mode="lines+markers",
        name="Six-municipality pooled RQ NTL",
        connectgaps=False,
        line=dict(color="#002FFF", width=5.0, shape="hv"),
        marker=dict(size=5),
    )
)
fig_regional_bridge.add_trace(
    go.Scatter(
        x=ngcp_four_day["date_start"],
        y=ngcp_four_day["recovery_pct"],
        mode="lines",
        name="NGCP 01:00 load",
        connectgaps=False,
        line=dict(color="#111111", width=4.5, dash="dash"),
    )
)
fig_regional_bridge.add_hline(y=100, line=dict(color="#5B6770", width=2, dash="dot"))
fig_regional_bridge.add_vline(
    x=EVENT_DATE.to_pydatetime(),
    line=dict(color="#0057FF", width=2.8, dash="dash"),
)
fig_regional_bridge.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1600,
    height=700,
    title=dict(
        text=(
            "Pooled six-municipality trajectory and NGCP regional load<br>"
            "<sup>Directional consistency check only; not local validation</sup>"
        ),
        x=0.5,
        font=dict(size=28),
    ),
    xaxis=dict(title="Date", tickformat="%b<br>%Y", showgrid=True, gridcolor="#E8EDF3"),
    yaxis=dict(title="Recovery (% baseline)", rangemode="tozero", showgrid=True, gridcolor="#E8EDF3"),
    legend=dict(orientation="h", x=0.5, y=1.02, xanchor="center", font=dict(size=17)),
    font=dict(family="Arial", size=17, color="#243B5A"),
    margin=dict(l=100, r=45, t=130, b=75),
    hovermode="x unified",
)
fig_regional_bridge.show()
display(Markdown(f"**Regional bridge decision:** {bridge_decision}"))

## 6. Recovery milestones instead of result tables

Impact-drop error bars reflect baseline sensitivity. T50, T80, and T90 points show the first persistent supporting composite; horizontal bars extend back to the last admissible below-threshold observation. Missing milestones are not plotted and must be read with the quality flag in the compact summary.

In [ ]:
# ============================================================
# FIGURE 11. OBSERVED IMPACT DROP
# ============================================================


def parse_numeric_range(value):
    if value is None or pd.isna(value):
        return (np.nan, np.nan)
    numbers = re.findall(r"[-+]?\d+(?:\.\d+)?", str(value))
    if len(numbers) < 2:
        return (np.nan, np.nan)
    return float(numbers[0]), float(numbers[1])


poi_parent_lookup = dict(zip(poi_table["poi_name"], poi_table["municipality_name"]))
metric_frames = [municipal_metrics.assign(support_type="Municipality")]
if not poi_metrics.empty:
    metric_frames.append(poi_metrics.assign(support_type="POI kernel"))
all_location_metrics = pd.concat(metric_frames, ignore_index=True)
all_location_metrics["parent_unit"] = np.where(
    all_location_metrics["support_type"].eq("Municipality"),
    all_location_metrics["unit_name"].astype(str),
    all_location_metrics["unit_name"].astype(str).map(poi_parent_lookup),
)
all_location_metrics["display_name"] = (
    all_location_metrics["unit_name"].astype(str)
    + np.where(all_location_metrics["support_type"].eq("POI kernel"), " · POI", "")
)

impact_plot = all_location_metrics.sort_values(
    ["parent_unit", "support_type"], ascending=[False, False]
).copy()
impact_ranges = impact_plot["impact_drop_range_pct"].map(parse_numeric_range)
impact_low = np.array([value[0] for value in impact_ranges], dtype=float)
impact_high = np.array([value[1] for value in impact_ranges], dtype=float)

fig_impact = go.Figure(
    go.Bar(
        x=impact_plot["impact_drop_pct"],
        y=impact_plot["display_name"],
        orientation="h",
        marker=dict(
            color=impact_plot["parent_unit"].map(LOCATION_COLORS),
            pattern_shape=np.where(impact_plot["support_type"].eq("POI kernel"), "/", ""),
        ),
        error_x=dict(
            type="data",
            symmetric=False,
            array=np.maximum(0, impact_high - impact_plot["impact_drop_pct"].to_numpy()),
            arrayminus=np.maximum(0, impact_plot["impact_drop_pct"].to_numpy() - impact_low),
            color="#3B3B3B",
            thickness=1.5,
            width=3,
        ),
        customdata=np.column_stack([
            impact_plot["quality_flag"],
            impact_plot["support_type"],
            impact_plot["parent_unit"],
        ]),
        hovertemplate=(
            "%{y}<br>Observed impact drop: %{x:.1f}%"
            "<br>%{customdata[0]} · %{customdata[1]}"
            "<br>Location colour: %{customdata[2]}<extra></extra>"
        ),
    )
)
fig_impact.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    width=1600,
    height=850,
    title="Deepest observed four-day median NTL drop within 60 days of Haiyan",
    xaxis_title="Drop from baseline (%)",
    yaxis_title=None,
    font=dict(family="Arial", size=16, color="#243B5A"),
    margin=dict(l=260, r=55, t=95, b=70),
)
fig_impact.show()

In [ ]:
# ============================================================
# FIGURE 12. T50, T80, AND T90 — MARKERS ONLY
# ============================================================

municipality_milestones = all_location_metrics.loc[
    all_location_metrics["support_type"] == "Municipality"
].copy()
poi_milestones = all_location_metrics.loc[
    all_location_metrics["support_type"] == "POI kernel"
].copy()

fig_milestones = make_subplots(
    rows=1, cols=2,
    horizontal_spacing=0.18,
    subplot_titles=("Six municipalities", "Six POI kernels"),
)

for panel_column, panel_metrics in enumerate(
    [municipality_milestones, poi_milestones], start=1
):
    if panel_column == 1:
        location_order = LOCATION_ORDER
    else:
        location_order = POI_ORDER
    category_order = location_order[::-1]
    fig_milestones.add_trace(
        go.Scatter(
            x=np.zeros(len(location_order)), y=location_order,
            mode="markers", marker=dict(size=1, color="rgba(0,0,0,0)"),
            showlegend=False, hoverinfo="skip",
        ),
        row=1, col=panel_column,
    )
    for threshold in (50, 80, 90):
        available = panel_metrics.loc[
            panel_metrics[f"T{threshold}_day"].notna()
        ].copy()
        fig_milestones.add_trace(
            go.Scatter(
                x=available[f"T{threshold}_day"],
                y=available["unit_name"].astype(str),
                mode="markers",
                name=f"T{threshold}",
                legendgroup=f"T{threshold}",
                showlegend=panel_column == 1,
                marker=dict(
                    size=18,
                    color=MILESTONE_COLORS[threshold],
                    symbol=MILESTONE_SYMBOLS[threshold],
                    line=dict(color="white", width=1.5),
                ),
                customdata=np.column_stack([
                    available[f"T{threshold}_observation_interval"].fillna("—"),
                    available[f"T{threshold}_status"].fillna("—"),
                ]),
                hovertemplate=(
                    f"%{{y}}<br>T{threshold}: %{{x:.0f}} days"
                    "<br>Observation interval: %{customdata[0]}"
                    "<br>%{customdata[1]}<extra></extra>"
                ),
            ),
            row=1, col=panel_column,
        )
    fig_milestones.update_xaxes(
        range=[0, int((PROFILE_END - EVENT_DATE).days)],
        title_text="Days after Haiyan",
        tickvals=[0, 60, 120, 180, 240, 300, 360],
        tickfont=dict(size=16),
        showgrid=True, gridcolor="#E8EDF3",
        row=1, col=panel_column,
    )
    fig_milestones.update_yaxes(
        categoryorder="array", categoryarray=category_order,
        tickmode="array", tickvals=location_order, ticktext=location_order,
        tickfont=dict(size=17), automargin=True,
        showgrid=True, gridcolor="#EEF2F6",
        row=1, col=panel_column,
    )

fig_milestones.update_layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="white",
    width=1800,
    height=820,
    title=dict(
        text="Persistent recovery milestones — markers only",
        x=0.5, font=dict(size=31),
    ),
    legend=dict(
        orientation="h", x=0.5, y=1.05,
        xanchor="center", font=dict(size=18),
    ),
    font=dict(family="Arial", size=17, color="#243B5A"),
    margin=dict(l=230, r=70, t=145, b=80),
)
fig_milestones.update_annotations(font=dict(size=21, color="#243B5A"))
fig_milestones.show()

### What T50, T80, and T90 mean here

- The denominator is the **matched 60-day baseline** immediately before Haiyan: 9 September–7 November 2013.
- The impact reference is the lowest admissible four-day composite during days 0–59.
- T50, T80, and T90 are searched **after that observed nadir** and require two consecutive admissible four-day composites at or above 50%, 80%, or 90% of baseline.
- If the nadir never fell below a threshold, the result is **“threshold not lost,”** not a recovery time.
- The plotted milestone symbol sits on the threshold itself. Hover text reports the supporting composite, which may be higher.

These are interval-censored observation-based milestones. They do not imply that recovery occurred continuously between observations.

### Compact exact-value summary

The charts remain the primary result. This table is retained only for exact milestone values and the distinction between supported, observation-limited, and not-observable cases.

In [ ]:
# ============================================================
# TABLE 1. COMPACT RECOVERY SUMMARY
# ============================================================

compact_summary = all_location_metrics[
    [
        "support_type",
        "unit_name",
        "quality_flag",
        "retained_pct",
        "max_gap_days",
        "median_event_sc_pct",
        "impact_drop_pct",
        "T50_day",
        "T50_status",
        "T80_day",
        "T80_status",
        "T90_day",
        "T90_status",
    ]
].rename(
    columns={
        "support_type": "Support",
        "unit_name": "Location",
        "quality_flag": "Quality",
        "retained_pct": "Retained (%)",
        "max_gap_days": "Max gap (d)",
        "median_event_sc_pct": "Median SC (%)",
        "impact_drop_pct": "Impact drop (%)",
        "T50_day": "T50 (d)",
        "T50_status": "T50 status",
        "T80_day": "T80 (d)",
        "T80_status": "T80 status",
        "T90_day": "T90 (d)",
        "T90_status": "T90 status",
    }
).sort_values(["Support", "Location"])

display(
    compact_summary.style.format(
        {
            "Retained (%)": "{:.1f}",
            "Median SC (%)": "{:.1f}",
            "Impact drop (%)": "{:.1f}",
            "T50 (d)": "{:.0f}",
            "T80 (d)": "{:.0f}",
            "T90 (d)": "{:.0f}",
        },
        na_rep="—",
    ).hide(axis="index")
)

## 7. Discussion against documented recovery

The spatial and temporal results are consistent with a severe, geographically uneven disruption followed by a non-monotonic recovery. The evidence supports three restrained interpretations.

1. **A regional electricity-related shock is visible, but recovery is local.** Regional agreement with NGCP demand supports the broad trajectory. Municipality and POI profiles then show substantial differences in timing, amplitude, and stability.
2. **Functional nighttime activity and physical reconstruction are related but not interchangeable.** Very-high-resolution studies of Tacloban document heterogeneous building and land-use recovery, while resilience assessments stress that remote sensing cannot by itself explain household-level causes or outcomes ([Sheykhmousa et al., 2019](https://doi.org/10.3390/rs11101174); [Ghaffarian et al., 2019](https://doi.org/10.3390/rs11212511)).
3. **Scale changes the result.** POI kernels isolate city or municipal centres; municipality aggregates describe broader settlement support. Divergence between them may indicate spatially uneven restoration, but it may also arise from small valid-pixel samples and changing observability.

The notebook therefore treats reliability-qualified NTL as a proxy for electricity-dependent nocturnal activity. It does not equate a return to baseline with complete social, economic, or physical recovery.

## 8. Diagnostic comparison with gap-filled nighttime lights

The primary analysis uses direct `DNB_BRDF_Corrected_NTL` observations with `Mandatory_Quality_Flag == 0`. The gap-filled band is retained only as a diagnostic comparison. NASA's Black Marble documentation identifies gap-filling and the age of the source observation as quality information; a filled value can therefore smooth, delay, or attenuate an abrupt disaster signal rather than represent a fresh observation ([NASA Black Marble User Guide](https://viirsland.gsfc.nasa.gov/PDF/BlackMarbleUserGuide_v1.2_20210421.pdf)).

The end-of-notebook comparison asks whether the gap-filled series changes the estimated shock timing or magnitude for the focus municipality. It is not used to replace missing direct observations or to calculate the principal recovery milestones. This follows the RQ1 result that tropical nighttime lights must be qualified for observability before interpretation ([Principe et al., 2026](https://doi.org/10.5194/isprs-archives-XLIX-B2-2026-1091-2026)).

In [ ]:
# ============================================================
# 16. GAP-FILLED DIAGNOSTIC FOR THE FOCUS LGU
# ============================================================

focus_key = canonical_unit_name(FOCUS_UNIT)
focus_match = selected_raster_crs.loc[selected_raster_crs["unit_key"] == focus_key]

if gap_filled is not None and len(focus_match) == 1:
    focus_row = focus_match.iloc[0]
    focus_support_full = (selected_zone_id == int(focus_row["profile_id"])) & ghsl_mask
    gap_cube_full = gap_filled.where(rq_base_mask)
    focus_gap_cube, focus_gap_support = crop_to_support(gap_cube_full, focus_support_full)

    try:
        (
            focus_gap_profile,
            _,
            _,
            _,
            focus_gap_report,
        ) = build_pixel_matched_profile(
            cube=focus_gap_cube,
            support_mask=focus_gap_support,
            aggregation_days=4,
            unit_name=focus_row["unit_name"],
            unit_type="City/municipality",
            method="Gap-filled diagnostic",
        )

        focus_rq = municipal_four_day.loc[
            municipal_four_day["unit_name"] == focus_row["unit_name"]
        ]

        fig_gap_diagnostic = go.Figure()
        fig_gap_diagnostic.add_trace(
            go.Scatter(
                x=focus_rq["date_start"],
                y=focus_rq["recovery_pct"],
                mode="lines+markers",
                name="RQ direct DNB-BRDF",
                connectgaps=False,
                line=dict(color="#009227", width=2.8, shape="hv"),
            )
        )
        fig_gap_diagnostic.add_trace(
            go.Scatter(
                x=focus_gap_profile["date_start"],
                y=focus_gap_profile["recovery_pct"],
                mode="lines",
                name="Gap-filled diagnostic",
                line=dict(color="#D62728", width=2.4, dash="dash"),
            )
        )
        fig_gap_diagnostic.add_hline(y=100, line_dash="dot", line_color="#7F8C8D")
        fig_gap_diagnostic.add_vline(
            x=EVENT_DATE.to_pydatetime(),
            line_dash="dash",
            line_color="#0057FF",
        )
        fig_gap_diagnostic.update_layout(
            template="plotly_white",
            paper_bgcolor="rgba(0,0,0,0)",
            title=(
                f"Observed versus gap-filled recovery: {focus_row['unit_name']}<br>"
                "<sup>Diagnostic only; milestones use direct DNB-BRDF observations</sup>"
            ),
            xaxis_title="Date",
            yaxis_title="Recovery relative to baseline (%)",
            width=1200,
            height=520,
            font=dict(family="Arial", size=13, color="#243B5A"),
            legend=dict(orientation="h", y=1.04),
        )
        fig_gap_diagnostic.show()
    except ValueError as error:
        print("Gap-filled diagnostic unavailable:", error)
else:
    print("Gap-filled band or unique focus municipality is unavailable; diagnostic skipped.")

## 9. Methods and provenance

### Signal and observability

- Product: VNP46A2 daily Black Marble.
- Main signal: `DNB_BRDF_Corrected_NTL`.
- Fresh observation: `Mandatory_Quality_Flag == 0`.
- Spatial support: the fixed six MuniCities intersected with GHSL G3 classes 22, 23, and 30.
- Spatial context: Roads and the supplied Haiyan/Yolanda path shapefile; the path is contextual, not a local intensity surface.
- Spatial completeness: share of fixed baseline-lit support pixels with a valid direct observation in each composite.
- Four-day value: median of admissible daily observations inside each non-overlapping event-anchored block.

### Baseline and recovery

- Baseline window: the 60 calendar days immediately before Haiyan, 9 September–7 November 2013.
- Per-pixel baseline: median of complete reliability-qualified composites in that window.
- Pixel matching: each date is compared only with the baseline values of the same currently valid pixels.
- Raw magnitude: spatial median NTL across currently valid fixed-support pixels.
- Recovery: 100 × current summed radiance / matched baseline summed radiance; no temporal mean is used.
- T50/T80/T90: first of two consecutive admissible four-day composites at or above the stated share of baseline, searched after the observed Stage-1 nadir.

### Interpretive boundary

Missing observations remain missing. “Not recovered,” “threshold not lost,” and “not observable” are distinct outcomes. NGCP is a regional consistency check; daytime remote sensing and documentary evidence provide external context rather than pixel-level validation.

In [ ]:
# ============================================================
# EXPORT CLEAN SIX-LOCATION DATA AND FIGURES
# ============================================================

municipal_daily.to_csv(OUTPUT_DIR / "six_municipalities_daily_rq_profiles.csv", index=False)
municipal_four_day.to_csv(OUTPUT_DIR / "six_municipalities_four_day_median_profiles.csv", index=False)
municipal_metrics.to_csv(OUTPUT_DIR / "six_municipalities_recovery_metrics.csv", index=False)
municipal_stage_summary.to_csv(OUTPUT_DIR / "six_municipalities_stage_summary.csv", index=False)
municipal_reports.to_csv(OUTPUT_DIR / "six_municipalities_profile_provenance.csv", index=False)
poi_daily.to_csv(OUTPUT_DIR / "six_poi_daily_rq_profiles.csv", index=False)
poi_four_day.to_csv(OUTPUT_DIR / "six_poi_four_day_median_profiles.csv", index=False)
poi_metrics.to_csv(OUTPUT_DIR / "six_poi_recovery_metrics.csv", index=False)
poi_stage_summary.to_csv(OUTPUT_DIR / "six_poi_stage_summary.csv", index=False)
poi_reports.to_csv(OUTPUT_DIR / "six_poi_profile_provenance.csv", index=False)
selection_summary.to_csv(OUTPUT_DIR / "six_location_selection_rationale.csv", index=False)
bridge_summary.to_csv(OUTPUT_DIR / "six_location_ngcp_bridge.csv", index=False)

numbered_figures = [
    ("01_six_location_rationale", fig_selection_rationale),
    ("02_square_six_municipality_map", fig_locations),
    ("03_spatial_map_and_raw_daily_ntl", fig_raw_daily_map),
    ("04_ghsl_g3_with_tacloban_inset", fig_locations_g3),
    ("05_median_baseline_ntl_horizontal_colorbar", fig_locations_g3_ntl),
    ("06_stage_maps_with_tacloban_insets", fig_regional_recovery),
    ("07_six_municipality_profiles", fig_municipality_profiles),
    ("08_six_poi_profiles", fig_poi_profiles),
    ("09_raw_vs_normalized_observed_summary", fig_raw_vs_observed_summary),
    ("10_pooled_six_location_ngcp_bridge", fig_regional_bridge),
    ("11_observed_impact_drop", fig_impact),
    ("12_t50_t80_t90_markers_only", fig_milestones),
]
for stem, figure in numbered_figures:
    save_plotly_figure(figure, stem)

if "fig_gap_diagnostic" in globals():
    save_plotly_figure(fig_gap_diagnostic, "13_observed_vs_gap_filled_diagnostic")

print("Data written to:", OUTPUT_DIR)
print("Figures written to:", FIGURE_DIR)

## Takeaway

This revision restores a simpler visual hierarchy: square spatial context first, raw daily observations second, and normalized recovery third. Municipalities and POIs are separated in the main profile grids. T50, T80, and T90 appear as markers only. The solid-black Haiyan path and repeated Tacloban insets provide spatial context without overwhelming the nighttime-light signal.